# Phase II — Interactive Detective Mystery (web server)

Hosts the detective game as a **live website** backed by vLLM + FastAPI.
Every player command goes through the real action-interpreter, drama-manager,
and LLM narration stack.

**Before running:** upload your Phase I output files to `/content/data/`:
- `plot_points.json` — story beats from `phase1_story_generator`
- `case_file.json`   — case metadata from `phase1_story_generator`

The notebook runs the full Phase II pipeline: `story_to_plan` (→ `plan.json`) → `world_builder` (→ `world.json`).

**GPU required.** Runtime → Change runtime type → T4 / L4 / A100.


## 1. Install dependencies

~3–5 min on a fresh runtime.

In [ ]:
!pip install --quiet 'vllm>=0.11,<0.13' 'openai>=1.52' 'fastapi>=0.110' 'uvicorn[standard]>=0.29'
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('GPU memory:', f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB" if torch.cuda.is_available() else 'n/a')


## 2. Pick model and set up directories

T4 16 GB → `Qwen2.5-3B-Instruct` · L4 24 GB → `Qwen2.5-7B-Instruct` · A100 40 GB → `Qwen3-8B`

In [ ]:
import os, torch, pathlib
os.chdir('/content')
for d in ('data', 'logs', 'logs/web', 'web'):
    pathlib.Path(d).mkdir(exist_ok=True)

if torch.cuda.is_available():
    props    = torch.cuda.get_device_properties(0)
    gpu_gb   = props.total_memory / 1e9
    cc_major = props.major
else:
    gpu_gb, cc_major = 0, 0

if gpu_gb >= 35:
    MODEL = 'Qwen/Qwen3-8B'
elif gpu_gb >= 22:
    MODEL = 'Qwen/Qwen2.5-7B-Instruct'
else:
    MODEL = 'Qwen/Qwen2.5-3B-Instruct'

DTYPE    = 'bfloat16' if cc_major >= 8 else 'float16'
PORT     = 8000
WEB_PORT = 7860

os.environ['LLM_MODEL']    = MODEL
os.environ['LLM_ENDPOINT'] = f'http://localhost:{PORT}/v1'
os.environ['LLM_API_KEY']  = 'EMPTY'
os.environ['LLM_DTYPE']    = DTYPE

print('Model    :', MODEL)
print('Dtype    :', DTYPE, f'(CC {cc_major}.x)')
print('Endpoint :', os.environ['LLM_ENDPOINT'])


## 3. Write Python modules

In [ ]:
%%writefile llm_client.py
"""OpenAI-compatible wrapper around a local vLLM server.

Single point of LLM access for the whole project. Every other module
must route calls through `chat()` or `chat_json()` so the backend can
be swapped without touching the rest of the code.

Endpoint and model come from env vars, never hard-coded:
    LLM_ENDPOINT   default http://localhost:8000/v1
    LLM_MODEL      default Qwen/Qwen3-8B
    LLM_API_KEY    default EMPTY (vLLM ignores it)
"""
from __future__ import annotations

import json
import os
import re
import time
from typing import Any

from openai import (
    APIConnectionError,
    APITimeoutError,
    InternalServerError,
    OpenAI,
    RateLimitError,
)

_client: OpenAI | None = None
_DEFAULT_MODEL = "Qwen/Qwen3-8B"


def _get_client() -> OpenAI:
    global _client
    if _client is None:
        base_url = os.environ.get("LLM_ENDPOINT", "http://localhost:8000/v1")
        api_key = os.environ.get("LLM_API_KEY", "EMPTY")
        _client = OpenAI(base_url=base_url, api_key=api_key, timeout=600.0)
    return _client


def _resolve_model(model: str | None) -> str:
    return model or os.environ.get("LLM_MODEL", _DEFAULT_MODEL)


_THINK_BLOCK = re.compile(r"<think>.*?</think>\s*", re.DOTALL | re.IGNORECASE)


def _strip_think(text: str) -> str:
    """Qwen3 and other reasoning models wrap inner monologue in <think>...</think>.
    We want only the final answer, so strip the tag and anything inside.
    Also tolerate an unclosed <think>... tail from a truncated response by cutting
    at the first </think>."""
    text = _THINK_BLOCK.sub("", text)
    if "</think>" in text:
        text = text.split("</think>", 1)[-1]
    if text.lstrip().startswith("<think>"):
        text = text.lstrip()[len("<think>"):]
    return text.strip()


def chat(
    messages: list[dict[str, str]],
    *,
    model: str | None = None,
    max_tokens: int = 1024,
    temperature: float = 0.7,
    retries: int = 8,
    enable_thinking: bool = False,
    **kwargs: Any,
) -> str:
    """Send a chat request and return the assistant message content.

    Retries transient connection / timeout / rate-limit errors with
    exponential backoff so the caller can fire requests while the vLLM
    server is still warming up.

    `enable_thinking` controls Qwen3's reasoning mode. Defaults to False —
    we want terse structured outputs, not monologue. vLLM accepts this via
    the `chat_template_kwargs` extra body field; we also strip any residual
    <think> blocks from the response.
    """
    client = _get_client()
    resolved_model = _resolve_model(model)

    # Merge any user-provided extra_body so we don't clobber it.
    extra_body = dict(kwargs.pop("extra_body", {}) or {})
    ctk = dict(extra_body.get("chat_template_kwargs", {}) or {})
    ctk.setdefault("enable_thinking", enable_thinking)
    extra_body["chat_template_kwargs"] = ctk

    last_err: Exception | None = None
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=resolved_model,
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature,
                extra_body=extra_body,
                **kwargs,
            )
            return _strip_think((resp.choices[0].message.content or "").strip())
        except (APIConnectionError, APITimeoutError, InternalServerError, RateLimitError) as err:
            last_err = err
            backoff = min(60.0, 2.0 ** attempt)
            time.sleep(backoff)
    raise RuntimeError(f"chat failed after {retries} retries: {last_err!r}")


def chat_simple(prompt: str, *, system: str | None = None, **kwargs: Any) -> str:
    """Convenience wrapper for a single-turn user prompt."""
    messages: list[dict[str, str]] = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    return chat(messages, **kwargs)


_JSON_FENCE = re.compile(r"```(?:json)?\s*(.*?)```", re.DOTALL)


def parse_json_safe(text: str) -> Any:
    """Strip markdown fences and parse JSON. Fallback: find first `{` / `[`."""
    text = text.strip()
    m = _JSON_FENCE.search(text)
    if m:
        text = m.group(1).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        for opener, closer in (("{", "}"), ("[", "]")):
            start = text.find(opener)
            end = text.rfind(closer)
            if start != -1 and end > start:
                try:
                    return json.loads(text[start : end + 1])
                except json.JSONDecodeError:
                    continue
        raise


def chat_json(
    prompt: str,
    *,
    system: str | None = None,
    max_tokens: int = 2048,
    temperature: float = 0.3,
    max_parse_retries: int = 3,
    **kwargs: Any,
) -> Any:
    """Ask for JSON, parse it, reprompt briefly if it fails to parse."""
    last_raw = ""
    for attempt in range(max_parse_retries):
        raw = chat_simple(
            prompt,
            system=system,
            max_tokens=max_tokens,
            temperature=temperature if attempt == 0 else max(0.1, temperature - 0.1 * attempt),
            **kwargs,
        )
        last_raw = raw
        try:
            return parse_json_safe(raw)
        except (json.JSONDecodeError, ValueError):
            continue
    raise ValueError(f"could not parse JSON after {max_parse_retries} attempts. Last raw:\n{last_raw[:400]}")


def health_check() -> dict[str, Any]:
    """Return `{ok, endpoint, model}` — fast smoke-test."""
    endpoint = os.environ.get("LLM_ENDPOINT", "http://localhost:8000/v1")
    model = _resolve_model(None)
    try:
        out = chat_simple("Reply with just: ok", max_tokens=16, temperature=0.0, retries=2)
        return {"ok": True, "endpoint": endpoint, "model": model, "reply": out}
    except Exception as err:  # noqa: BLE001 — surface any failure to caller
        return {"ok": False, "endpoint": endpoint, "model": model, "error": repr(err)}


if __name__ == "__main__":
    print(json.dumps(health_check(), indent=2))


In [ ]:
%%writefile plan_types.py
"""Shared dataclasses for the plan representation.

An `Event` is a single step the detective (or an NPC) takes. It has explicit
preconditions and effects expressed as predicates over world-state subjects.
A `CausalLink` (producer, condition, consumer) indicates that the producer
event establishes `condition`, which the consumer event requires — so the
condition must remain true across the span between them. The drama manager
watches those spans.

World state is a flat dict[str, dict[str, Any]]: state[subject_id][attr].
Subjects are identifiers like "character.victoria_harrington",
"location.gallery_main_hall", "object.fountain_pen", "detective.inventory".
"""
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from typing import Any


@dataclass
class Condition:
    """A predicate over world state: `state[subject][attr] <op> value`.

    `op` ∈ {"==", "!=", "contains", "not_contains", ">=", "<="}.
    """
    subject: str
    attr: str
    op: str
    value: Any

    def evaluate(self, state: dict[str, dict[str, Any]]) -> bool:
        slot = state.get(self.subject, {}).get(self.attr)
        if self.op == "==":
            return slot == self.value
        if self.op == "!=":
            return slot != self.value
        if self.op == "contains":
            return slot is not None and self.value in slot
        if self.op == "not_contains":
            return slot is None or self.value not in slot
        if self.op == ">=":
            return slot is not None and slot >= self.value
        if self.op == "<=":
            return slot is not None and slot <= self.value
        raise ValueError(f"unknown op: {self.op}")

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "Condition":
        return cls(subject=d["subject"], attr=d["attr"], op=d["op"], value=d["value"])


@dataclass
class Effect:
    """A world-state mutation.

    `op` ∈ {"set", "add", "remove"}. "add"/"remove" treat the slot as a
    set / list of items; "set" overwrites.
    """
    subject: str
    attr: str
    op: str
    value: Any

    def apply(self, state: dict[str, dict[str, Any]]) -> None:
        bucket = state.setdefault(self.subject, {})
        if self.op == "set":
            bucket[self.attr] = self.value
            return
        cur = bucket.get(self.attr)
        if self.op == "add":
            if isinstance(cur, list):
                if self.value not in cur:
                    cur.append(self.value)
            elif isinstance(cur, set):
                cur.add(self.value)
            else:
                bucket[self.attr] = [self.value]
        elif self.op == "remove":
            if isinstance(cur, list) and self.value in cur:
                cur.remove(self.value)
            elif isinstance(cur, set) and self.value in cur:
                cur.discard(self.value)
        else:
            raise ValueError(f"unknown op: {self.op}")

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "Effect":
        return cls(subject=d["subject"], attr=d["attr"], op=d["op"], value=d["value"])


@dataclass
class Event:
    id: str
    actor: str
    verb: str
    args: list[str] = field(default_factory=list)
    location: str = ""
    preconditions: list[Condition] = field(default_factory=list)
    effects: list[Effect] = field(default_factory=list)
    reveals: list[str] = field(default_factory=list)
    description: str = ""
    narrative: str = ""
    source_plot_idx: int | None = None

    def to_dict(self) -> dict[str, Any]:
        return {
            "id": self.id,
            "actor": self.actor,
            "verb": self.verb,
            "args": self.args,
            "location": self.location,
            "preconditions": [c.to_dict() for c in self.preconditions],
            "effects": [e.to_dict() for e in self.effects],
            "reveals": self.reveals,
            "description": self.description,
            "narrative": self.narrative,
            "source_plot_idx": self.source_plot_idx,
        }

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "Event":
        return cls(
            id=d["id"],
            actor=d.get("actor", "detective"),
            verb=d.get("verb", "act"),
            args=list(d.get("args", [])),
            location=d.get("location", ""),
            preconditions=[Condition.from_dict(c) for c in d.get("preconditions", [])],
            effects=[Effect.from_dict(e) for e in d.get("effects", [])],
            reveals=list(d.get("reveals", [])),
            description=d.get("description", ""),
            narrative=d.get("narrative", ""),
            source_plot_idx=d.get("source_plot_idx"),
        )


@dataclass
class CausalLink:
    """(producer_event, condition, consumer_event)

    Condition must remain true from the moment `producer` executes until
    `consumer` executes. If something in between breaks it, we have an
    exception.
    """
    producer: str
    condition: Condition
    consumer: str

    def to_dict(self) -> dict[str, Any]:
        return {
            "producer": self.producer,
            "consumer": self.consumer,
            "condition": self.condition.to_dict(),
        }

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "CausalLink":
        return cls(
            producer=d["producer"],
            consumer=d["consumer"],
            condition=Condition.from_dict(d["condition"]),
        )


@dataclass
class Plan:
    events: dict[str, Event] = field(default_factory=dict)
    order: list[tuple[str, str]] = field(default_factory=list)
    causal_links: list[CausalLink] = field(default_factory=list)
    initial_state: dict[str, dict[str, Any]] = field(default_factory=dict)
    goal: list[Condition] = field(default_factory=list)

    def to_dict(self) -> dict[str, Any]:
        return {
            "events": {eid: ev.to_dict() for eid, ev in self.events.items()},
            "order": [list(edge) for edge in self.order],
            "causal_links": [cl.to_dict() for cl in self.causal_links],
            "initial_state": self.initial_state,
            "goal": [c.to_dict() for c in self.goal],
        }

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "Plan":
        return cls(
            events={eid: Event.from_dict(e) for eid, e in d.get("events", {}).items()},
            order=[tuple(edge) for edge in d.get("order", [])],
            causal_links=[CausalLink.from_dict(cl) for cl in d.get("causal_links", [])],
            initial_state=d.get("initial_state", {}),
            goal=[Condition.from_dict(c) for c in d.get("goal", [])],
        )


In [ ]:
%%writefile phase1_story_generator.py
"""Phase I story generator — ported from Phase_I_Final_Story_Generator.ipynb.

All Claude API calls replaced with `llm_client.chat_simple` / `chat_json`.
Prompts preserved verbatim except where Claude-specific assumptions had to
be loosened for the Qwen chat template (see inline notes).

Usage:
    from phase1_story_generator import generate_full_story
    artifacts = generate_full_story("A poisoning murder at a 1920s London gallery")
    # artifacts["case_file"], ["complexities"], ["plot_points"], ["story_bible"], ["story_md"]
"""
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

from llm_client import chat_json, chat_simple


# ---------------------------------------------------------------------------
# Checkpoint helpers
# ---------------------------------------------------------------------------
def save_checkpoint(data: Any, filename: str | Path) -> None:
    path = Path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved: {path}")


def load_checkpoint(filename: str | Path) -> Any:
    return json.loads(Path(filename).read_text(encoding="utf-8"))


# ---------------------------------------------------------------------------
# Stage 1: case file
# ---------------------------------------------------------------------------
CASE_FILE_PROMPT = """You are a crime story architect. Generate a detailed murder mystery case file.
Output ONLY valid JSON, no markdown fences, no explanation.

{
  "criminal": {"name": str, "motive": str, "means": str, "opportunity": str},
  "victim": {"name": str, "background": str},
  "conspirators": [{"name": str, "role": str, "alibi": str}],
  "suspects": [{"name": str, "motive": str, "alibi": str}],
  "evidence": [
    {"id": str, "type": "physical|digital|testimonial",
     "description": str, "real_meaning": str, "steps_to_uncover": 2}
  ],
  "crime_timeline": [{"time": str, "event": str}],
  "solving_timeline": [
    {"step": int, "action": str, "target_evidence": [str], "max_actions": 3}
  ],
  "detective": {"name": str, "personal_stake": str, "deadline": str, "dire_consequence": str}
}

Generate at least 3 conspirators, 4 suspects, 8 evidence items, 6 solving steps."""


def generate_case_file(user_prompt: str) -> dict[str, Any]:
    full = f"{CASE_FILE_PROMPT}\n\nCrime context: {user_prompt}"
    return chat_json(full, max_tokens=3000, temperature=0.7)


# ---------------------------------------------------------------------------
# Stage 2: cover narrative / complexities
# ---------------------------------------------------------------------------
COMPLEXITIES_PROMPT = """Given real crime facts, generate a fabricated cover narrative.
Output ONLY valid JSON.

{
  "fake_suspect": {"name": str, "framing_reason": str},
  "planted_evidence": [{"description": str, "points_to": "fake_suspect"}],
  "false_testimonies": [{"witness": str, "claim": str}],
  "fake_timeline": [{"time": str, "event": str}],
  "evidence_fabrications": {"<evidence_id>": "fabricated explanation"},
  "conspirator_alibis": {"<name>": "alibi story"}
}

Rules:
- fake_suspect must NOT be the real criminal
- Every evidence id must appear in evidence_fabrications
- Alibis must be internally consistent"""


def generate_complexities(case_file: dict[str, Any]) -> dict[str, Any]:
    facts_summary = json.dumps(
        {
            "criminal": case_file["criminal"]["name"],
            "crime_timeline": case_file["crime_timeline"],
            "evidence_ids": [e["id"] for e in case_file["evidence"]],
            "conspirators": [c["name"] for c in case_file["conspirators"]],
        }
    )
    prompt = f"{COMPLEXITIES_PROMPT}\n\nReal facts: {facts_summary}"
    return chat_json(prompt, max_tokens=2000, temperature=0.7)


# ---------------------------------------------------------------------------
# Stage 3: meta-controller (plot points)
# ---------------------------------------------------------------------------
class StoryState:
    def __init__(self, case_file: dict[str, Any], complexities: dict[str, Any], max_points: int = 18) -> None:
        self.countdown = max_points + 3
        self.plot_points: list[dict[str, Any]] = []
        self.action_history: list[str] = []
        self.success_prob = 1.0
        self.evidence_progress = {e["id"]: 0 for e in case_file["evidence"]}
        self.alibi_status = {c["name"]: "unverified" for c in case_file["conspirators"]}
        self.closed_paths: set[str] = set()
        self.milestones_completed: set[int] = set()
        self.case_file = case_file
        self.complexities = complexities

    def tick(self) -> None:
        self.countdown -= 1
        self.success_prob = max(0.05, self.success_prob - 0.01)

    def is_done(self, min_points: int = 15) -> bool:
        all_milestones = len(self.milestones_completed) >= len(self.case_file["solving_timeline"])
        return all_milestones and len(self.plot_points) >= min_points


def _collision_detect(action: str, case_file: dict[str, Any]) -> dict[str, Any]:
    action_lower = action.lower()
    investigative_verbs = [
        "interview", "question", "investigate", "follow",
        "confront", "check", "ask", "visit", "examine", "search",
    ]
    for conspirator in case_file["conspirators"]:
        name_parts = conspirator["name"].lower().split()
        if any(part in action_lower for part in name_parts):
            if any(v in action_lower for v in investigative_verbs):
                return {"collision": True, "type": "conspirator", "target": conspirator["name"]}
    for evidence in case_file["evidence"]:
        keywords = set(evidence["description"].lower().split()) - {"the", "a", "an", "of", "in", "at", "to"}
        action_words = set(action_lower.split())
        if len(keywords & action_words) >= 2:
            return {"collision": True, "type": "evidence", "target": evidence["id"]}
    return {"collision": False, "type": None, "target": None}


def _check_extra_requirements(alibi_checks_done, multistep_clues_done, case_file):
    all_suspects = [s["name"] for s in case_file["suspects"]]
    alibis_covered = all(s in alibi_checks_done for s in all_suspects)
    multistep_done = sum(1 for v in multistep_clues_done.values() if v >= 2)
    return alibis_covered and multistep_done >= 3


def _decide_plot_type(state, alibi_checks_done, multistep_clues_done, case_file, iteration):
    suspects = [s["name"] for s in case_file["suspects"]]
    unchecked_suspects = [s for s in suspects if s not in alibi_checks_done]
    unfinished_clues = [eid for eid, steps in multistep_clues_done.items() if 0 < steps < 2]
    unstarted_clues = [eid for eid, steps in multistep_clues_done.items() if steps == 0]
    if unchecked_suspects and iteration % 3 == 1:
        return "alibi_check"
    if unfinished_clues:
        return "clue_followup"
    if unstarted_clues and iteration % 4 == 0:
        return "clue_start"
    if state.success_prob < 0.4:
        return "obstacle"
    return "progress"


def _generate_action(state, milestone, plot_type, case_file, complexities,
                     alibi_checks, multistep_clues, story_bible):
    suspects = case_file["suspects"]
    unchecked = [s for s in suspects if s["name"] not in alibi_checks]
    unfinished = [eid for eid, v in multistep_clues.items() if 0 < v < 2]
    unstarted = [eid for eid, v in multistep_clues.items() if v == 0]

    constraint = (
        f"CASE: Murder of {story_bible['victim_name']}.\n"
        f"Detective: {story_bible['detective_name']} and partner {story_bible['partner_name']}.\n"
        "All actions must relate to THIS murder case only. No hospitals, no unrelated crimes."
    )

    if plot_type == "alibi_check":
        target = unchecked[0] if unchecked else suspects[0]
        target_name = target["name"] if isinstance(target, dict) else target
        prompt = f"""{constraint}

Generate a detective action where the detective investigates the alibi of suspect: {target_name}
The action should involve contacting witnesses, checking records, or visiting locations related to the murder.
Recent actions (avoid repeating): {state.action_history[-3:]}
Output ONLY the action description (2-3 sentences)."""

    elif plot_type == "clue_followup":
        eid = unfinished[0]
        evidence = next((e for e in case_file["evidence"] if e["id"] == eid), None)
        prompt = f"""{constraint}

Generate a detective action that is a FOLLOW-UP step on this evidence from the murder scene:
Evidence: {evidence['description'] if evidence else eid}
This is step 2. The detective digs deeper (lab analysis, expert consultation, cross-referencing).
Output ONLY the action description (2-3 sentences)."""

    elif plot_type == "clue_start":
        eid = unstarted[0] if unstarted else list(multistep_clues.keys())[0]
        evidence = next((e for e in case_file["evidence"] if e["id"] == eid), None)
        prompt = f"""{constraint}

Generate a detective action discovering this murder evidence for the first time:
Evidence: {evidence['description'] if evidence else eid}
Step 1: detective notices something unusual but does NOT fully understand it yet.
Output ONLY the action description (2-3 sentences)."""
    else:
        prompt = f"""{constraint}

Generate a detective action for the murder investigation.
Current milestone: {milestone['action']}
Plot type: {plot_type}
Recent actions (avoid repeating): {state.action_history[-3:]}
Time pressure: {state.countdown} steps remaining
Output ONLY the action description (2-3 sentences)."""

    return chat_simple(prompt, max_tokens=150, temperature=0.8)


def _generate_narrative(action, collision, state, plot_type, case_file,
                        complexities, alibi_checks, multistep_clues, story_bible):
    fake_suspect = story_bible["fake_suspect"]
    constraint_header = f"""STRICT RULES - NEVER VIOLATE:
- This story is ONLY about the murder of: {story_bible['victim_name']}
- Detective: {story_bible['detective_name']} and partner {story_bible['partner_name']} (names never change)
- Real criminal (DO NOT reveal): {story_bible['real_criminal']}
- Fake suspect being framed: {fake_suspect} (names never change)
- Conspirators (exact names only): {story_bible['conspirator_names']}
- Key evidence: {story_bible['key_evidence']}
- Murder method: {story_bible['murder_method']}
- FORBIDDEN: Do NOT introduce hospitals, psychiatric wards, ambulances,
  unrelated victims, international crime networks, or any new murder cases.
- All red herrings must relate ONLY to the original murder of {story_bible['victim_name']}.
- Character names must NEVER change between chapters.

"""
    if plot_type == "alibi_check":
        instruction = (
            f"Write an alibi verification scene about the murder of {story_bible['victim_name']}. "
            "The detective checks a suspect's alibi for the night of the murder. "
            "A conspirator subtly provides false confirmation. "
            f"The detective ends up misled, suspicion points toward {fake_suspect}."
        )
    elif plot_type in ("clue_start", "clue_followup"):
        step = "initial discovery" if plot_type == "clue_start" else "deeper investigation"
        instruction = (
            f"Write a multi-step clue investigation ({step}) about the murder of {story_bible['victim_name']}. "
            "The detective examines physical evidence from the murder scene. "
            "Partial findings raise more questions. Do NOT reveal the full truth yet. "
            f"The clue should hint toward {fake_suspect} being guilty (misleadingly)."
        )
    elif collision["collision"]:
        instruction = (
            "Write a conspirator intervention scene. "
            f"A conspirator from the murder of {story_bible['victim_name']} smoothly misdirects the detective. "
            f"Detective almost gets close to real criminal {story_bible['real_criminal']}, then gets redirected toward {fake_suspect}."
        )
    elif plot_type == "obstacle":
        instruction = (
            f"Write an obstacle scene in the murder investigation of {story_bible['victim_name']}. "
            "A clue goes cold or a witness recants their statement about the murder. "
            "Detective frustrated, time running out to solve THIS case."
        )
    else:
        instruction = (
            f"Write a progress scene in the murder investigation of {story_bible['victim_name']}. "
            f"Small discovery points toward {fake_suspect} (the wrong person). "
            f"Dramatic irony: reader knows {story_bible['real_criminal']} is the real killer."
        )
    prompt = f"""{constraint_header}Write a suspenseful mystery plot point (2-3 paragraphs, 3rd person).
Detective's action: {action}
Writing instruction: {instruction}
Output ONLY the narrative paragraphs. Literary prose with dialogue."""
    return chat_simple(prompt, max_tokens=400, temperature=0.85)


def _update_tracking(plot_type, action, alibi_checks, multistep_clues, case_file):
    action_lower = action.lower()
    for suspect in case_file["suspects"]:
        name_parts = suspect["name"].lower().split()
        if any(p in action_lower for p in name_parts):
            if "alibi" in action_lower or plot_type == "alibi_check":
                if suspect["name"] not in alibi_checks:
                    alibi_checks[suspect["name"]] = "checked_false"
    for evidence in case_file["evidence"]:
        keywords = set(evidence["description"].lower().split()) - {"the", "a", "an", "of", "in"}
        if len(keywords & set(action_lower.split())) >= 2 or plot_type in ("clue_start", "clue_followup"):
            if plot_type == "clue_start" and multistep_clues.get(evidence["id"], 0) == 0:
                multistep_clues[evidence["id"]] = 1
                break
            if plot_type == "clue_followup" and multistep_clues.get(evidence["id"], 0) == 1:
                multistep_clues[evidence["id"]] = 2
                break


def _build_story_bible(case_file: dict[str, Any]) -> dict[str, Any]:
    return {
        "case_summary": f"Murder of {case_file['victim']['name']}",
        "victim_name": case_file["victim"]["name"],
        "detective_name": case_file["detective"]["name"],
        "partner_name": "Detective Martinez",
        "real_criminal": case_file["criminal"]["name"],
        "fake_suspect": case_file.get("fake_suspect", {}).get("name", "unknown"),
        "conspirator_names": [c["name"] for c in case_file["conspirators"]],
        "suspect_names": [s["name"] for s in case_file["suspects"]],
        "key_evidence": [e["description"] for e in case_file["evidence"][:3]],
        "murder_method": case_file["criminal"]["means"],
        "murder_location": case_file["victim"]["background"],
    }


def run_meta_controller(
    case_file: dict[str, Any],
    complexities: dict[str, Any],
    story_bible: dict[str, Any],
    min_points: int = 20,
    max_iter: int = 60,
) -> list[dict[str, Any]]:
    state = StoryState(case_file, complexities)
    milestone_idx = 0
    solving_tl = case_file["solving_timeline"]
    alibi_checks_done: dict[str, str] = {}
    multistep_clues_done = {e["id"]: 0 for e in case_file["evidence"]}

    for iteration in range(max_iter):
        if state.is_done(min_points) and _check_extra_requirements(
            alibi_checks_done, multistep_clues_done, case_file
        ):
            break
        state.tick()
        current_milestone = solving_tl[min(milestone_idx, len(solving_tl) - 1)]
        plot_type = _decide_plot_type(
            state, alibi_checks_done, multistep_clues_done, case_file, iteration
        )
        action = _generate_action(
            state, current_milestone, plot_type, case_file, complexities,
            alibi_checks_done, multistep_clues_done, story_bible,
        )
        state.action_history.append(action)
        collision = _collision_detect(action, case_file)
        narrative = _generate_narrative(
            action, collision, state, plot_type, case_file, complexities,
            alibi_checks_done, multistep_clues_done, story_bible,
        )
        state.plot_points.append(
            {
                "action": action, "narrative": narrative, "collision": collision,
                "prob": state.success_prob, "plot_type": plot_type,
            }
        )
        if collision["collision"]:
            state.success_prob = max(0.05, state.success_prob - 0.08)
        elif plot_type == "progress":
            state.success_prob = min(1.0, state.success_prob + 0.02)
        _update_tracking(plot_type, action, alibi_checks_done, multistep_clues_done, case_file)
        actions_on_ms = sum(
            1 for a in state.action_history
            if any(w in a.lower() for w in current_milestone["action"].lower().split()[:2])
        )
        if actions_on_ms >= current_milestone.get("max_actions", 3):
            state.milestones_completed.add(milestone_idx)
            milestone_idx = min(milestone_idx + 1, len(solving_tl) - 1)
        print(
            f"  Plot #{len(state.plot_points):2d} [{plot_type:15s}] "
            f"| prob={state.success_prob:.0%} "
            f"| collision={'YES' if collision['collision'] else 'no'}"
        )
    return state.plot_points


# ---------------------------------------------------------------------------
# Stage 4: assemble a complete novel-style markdown (Prologue + Chapters + Resolution + Epilogue)
# ---------------------------------------------------------------------------
_HANDCUFF_RE = re.compile(r"[^.]*handcuff[^.]*\.", re.IGNORECASE)


def _clean_plot_points(plot_points: list[dict[str, Any]], story_bible: dict[str, Any]) -> list[dict[str, Any]]:
    """Scrub premature arrests and leaked real-criminal names from individual
    plot-point narratives so the chapters can hide the truth until the end."""
    real = story_bible["real_criminal"]
    fake = story_bible["fake_suspect"]
    victim = story_bible["victim_name"]

    resolution_markers = (
        "you are under arrest", "i am arresting", "handcuffs clicked",
        "placed under arrest", "you're under arrest", "handcuffs snapped",
    )
    premature_rewrites = [
        (re.compile(rf"you['’]re under arrest(?: for)?[^.]*", re.IGNORECASE),
         f"you are a person of interest in the death of {victim}"),
        (re.compile(rf"you are under arrest(?: for)?[^.]*", re.IGNORECASE),
         f"you are a person of interest in the death of {victim}"),
        (re.compile(rf"i am arresting[^.]*", re.IGNORECASE),
         f"I need you to come in for further questioning regarding {victim}'s death"),
    ]

    cleaned: list[dict[str, Any]] = []
    for p in plot_points:
        narrative = p.get("narrative", "")
        lower = narrative.lower()

        for pat, repl in premature_rewrites:
            narrative = pat.sub(repl, narrative)
        if "handcuff" in lower:
            narrative = _HANDCUFF_RE.sub(
                "The detective made a mental note to arrange a formal interview.", narrative
            )
        # If the real criminal is named alongside any resolution marker, rename
        # them to the fake suspect — we must not reveal the killer in the body.
        if real and fake and real in narrative and any(mk in narrative.lower() for mk in resolution_markers):
            narrative = narrative.replace(real, fake)

        q = dict(p)
        q["narrative"] = narrative
        cleaned.append(q)
    return cleaned


def _stage_split(plot_points: list[dict[str, Any]]) -> list[tuple[int, str, str]]:
    """Return [(size, stage_title, stage_desc), ...] — the same five-stage
    breakdown used in the original notebook, scaled to the number of plot
    points produced this run."""
    total = max(len(plot_points), 5)
    s1 = max(2, int(total * 0.15))
    s2 = max(2, int(total * 0.20))
    s3 = max(4, int(total * 0.25))
    s4 = max(2, int(total * 0.20))
    s5 = max(1, total - s1 - s2 - s3 - s4)
    return [
        (s1, "Crime Scene Discovery",
         "Detective arrives at the scene, surveys the body, and notes the first physical clues."),
        (s2, "The Evidence Trail",
         "Multi-step investigation of key clues — examine, send to lab, cross-reference registries."),
        (s3, "Suspects and Misdirection",
         "Alibi checks for each suspect. Conspirators intervene and redirect suspicion toward the framed suspect."),
        (s4, "Closing In",
         "Detective narrows down suspects as evidence increasingly points (misleadingly) toward the framed suspect."),
        (s5, "Building the False Case",
         "Detective becomes convinced of the framed suspect's guilt; the final pieces are assembled."),
    ]


def assemble_story(
    case_file: dict[str, Any],
    plot_points: list[dict[str, Any]],
    story_bible: dict[str, Any],
    out_path: str | Path | None = None,
) -> str:
    """Generate a fluent novel-length markdown story from the plot points.

    Ported from the original Colab notebook (cell 8). The five-stage shape is
    preserved; prompts are rephrased so they work with Qwen's chat template
    and don't rely on Anthropic-specific formatting.
    """
    real = story_bible["real_criminal"]
    fake = story_bible["fake_suspect"]
    victim = story_bible["victim_name"]
    detective = story_bible["detective_name"]
    partner = story_bible.get("partner_name", "the detective's partner")
    conspirator_names = story_bible.get("conspirator_names", [])
    suspect_names = story_bible.get("suspect_names", [])
    murder_method = story_bible.get("murder_method", "unknown means")
    motive = case_file.get("criminal", {}).get("motive", "unknown motive")

    plot_points = _clean_plot_points(plot_points, story_bible)

    parts: list[str] = []

    prologue = chat_simple(
        f"""Write a mystery novel prologue revealing the TRUTH to the reader.

Constraints:
- Victim: {victim}
- Real killer: {real}
- Framed suspect (not the killer): {fake}
- Conspirators: {conspirator_names}
- Means: {murder_method}

Show what really happened on the night of the crime, atmospheric third-person prose,
220-260 words. Do not call it a prologue; open straight into the scene.""",
        max_tokens=500, temperature=0.75,
    )
    parts.append(f"# Prologue\n\n{prologue.strip()}")

    stage_defs = _stage_split(plot_points)
    idx = 0
    chapter_num = 1
    for size, stage_title, stage_desc in stage_defs:
        stage_points = plot_points[idx : idx + size]
        idx += size
        if not stage_points:
            continue
        # Two plot points per chapter.
        chunks = [stage_points[i : i + 2] for i in range(0, len(stage_points), 2)]
        for chunk in chunks:
            narratives = "\n\n".join(c["narrative"] for c in chunk)
            chapter = chat_simple(
                f"""Write Chapter {chapter_num} of a murder mystery novel.

STRICT RULES:
- The case is ONLY about the murder of: {victim}
- Detective: {detective} + partner {partner}
- Framed suspect being investigated: {fake}
- Real killer (do NOT reveal, do NOT arrest in this chapter): {real}
- Conspirator names (exact): {conspirator_names}
- Murder method: {murder_method}
- FORBIDDEN: hospitals, psychiatric wards, new crimes, new victims, unrelated plots.
- The detective must NOT solve the case or arrest anyone in this chapter.

Stage goal: {stage_desc}

Weave these two scenes together as one continuous chapter:

{narratives}

300-400 words. Literary prose with dialogue.""",
                max_tokens=650, temperature=0.8,
            )
            parts.append(f"# Chapter {chapter_num}: {stage_title}\n\n{chapter.strip()}")
            print(f"  Chapter {chapter_num} ({stage_title}) done")
            chapter_num += 1

    resolution = chat_simple(
        f"""Write a "Resolution" chapter for this murder mystery.

Setup:
- Victim: {victim}
- Detective: {detective}
- Detective WRONGLY arrests: {fake}
- Real killer is NOT caught: {real}
- Conspirator names: {conspirator_names}
- Suspect names: {suspect_names}
- Murder method: {murder_method}

Structure these four beats clearly:

1. EVIDENCE REVIEW — detective lays out the case against {fake} using concrete items.
2. ALIBI ELIMINATION — rule out each of these suspects by name and reason: {suspect_names}.
3. RECONSTRUCTION — "Here is what I believe happened that night..." walking step
   by step through a plausible (but wrong) sequence. Confident tone.
4. ARREST — detective confronts {fake}, who protests innocence. {real} watches
   from nearby, expression carefully composed.

450-520 words. Dramatic, confident tone.""",
        max_tokens=950, temperature=0.75,
    )
    parts.append(f"# The Resolution\n\n{resolution.strip()}")
    print("  Resolution chapter done")

    epilogue = chat_simple(
        f"""Write the epilogue for this murder mystery novel.

Constraints:
- Real killer: {real}
- Wrongly arrested: {fake}
- Victim: {victim}
- Conspirators: {conspirator_names}
- Motive: {motive}
- Means: {murder_method}

Four beats:
1. WRONG ARREST COMPLETE — {fake} is led away. Detective {detective} senses a
   small detail is off but dismisses it; the case is officially closed.
2. REAL KILLER'S HIDDEN CONFESSION — {real} mentally replays the truth, every
   gap filled: exact method, timing, disposal of evidence, role of each
   conspirator, and the motive ({motive}).
3. CONSPIRATORS' ROLES FULLY REVEALED (still in {real}'s thoughts).
4. THE LOOSE END — a minor character noticed something. They are too frightened
   to act tonight, but the truth has not been buried. End with a haunting
   one-line image.

320-360 words. Bittersweet, atmospheric tone.""",
        max_tokens=700, temperature=0.75,
    )
    parts.append(f"# Epilogue\n\n{epilogue.strip()}")
    print("  Epilogue done")

    story = "\n\n---\n\n".join(parts)
    if out_path:
        out = Path(out_path)
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(story, encoding="utf-8")
        print(f"\nStory saved -> {out}  ({chapter_num - 1} chapters + Resolution + Epilogue)")
    return story


# ---------------------------------------------------------------------------
# Top-level driver
# ---------------------------------------------------------------------------
def generate_full_story(
    user_prompt: str = "A poisoning murder at a prestigious 1920s London art gallery opening",
    out_dir: str | Path = "data",
    min_points: int = 20,
) -> dict[str, Any]:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print("Generating case file...")
    case_file = generate_case_file(user_prompt)
    save_checkpoint(case_file, out_dir / "case_file.json")

    print("Generating complexities...")
    complexities = generate_complexities(case_file)
    case_file["fake_suspect"] = complexities.get("fake_suspect", {})
    save_checkpoint(complexities, out_dir / "complexities.json")

    story_bible = _build_story_bible(case_file)
    save_checkpoint(story_bible, out_dir / "story_bible.json")

    print("Running meta-controller...")
    plot_points = run_meta_controller(case_file, complexities, story_bible, min_points=min_points)
    save_checkpoint(plot_points, out_dir / "plot_points.json")

    return {
        "case_file": case_file,
        "complexities": complexities,
        "story_bible": story_bible,
        "plot_points": plot_points,
    }


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    sub = parser.add_subparsers(dest="cmd")

    gen = sub.add_parser("generate", help="Run Phase I from scratch")
    gen.add_argument("--prompt", default="A poisoning murder at a prestigious 1920s London art gallery opening")
    gen.add_argument("--out-dir", default="data")
    gen.add_argument("--min-points", type=int, default=20)

    asm = sub.add_parser("assemble", help="Assemble a markdown novel from an existing plot_points.json")
    asm.add_argument("--data-dir", default="data")
    asm.add_argument("--out", default="data/final_story.md")

    # Backward-compat: if no subcommand, default to generate with the given flags.
    parser.add_argument("--prompt", default=None)
    parser.add_argument("--out-dir", default=None)
    parser.add_argument("--min-points", type=int, default=None)

    args = parser.parse_args()
    if args.cmd == "assemble":
        data_dir = Path(args.data_dir)
        case_file = load_checkpoint(data_dir / "case_file.json")
        plot_points = load_checkpoint(data_dir / "plot_points.json")
        story_bible = load_checkpoint(data_dir / "story_bible.json")
        assemble_story(case_file, plot_points, story_bible, out_path=args.out)
    else:
        generate_full_story(
            args.prompt or "A poisoning murder at a prestigious 1920s London art gallery opening",
            args.out_dir or "data",
            args.min_points or 20,
        )


In [ ]:
%%writefile story_to_plan.py
"""Convert Phase I output into a partially ordered plan with causal links.

Phase I gives us:
  - case_file.json: evidence[], conspirators[], suspects[], solving_timeline[]
  - plot_points.json: 20-ish {action, narrative, plot_type, collision}

Phase I does NOT give us preconditions / effects / locations. We reconstruct
them here, one event at a time, with a structured LLM call and then sanity
filter the output to our typed dataclasses. A second pass builds causal
links by pattern-matching producer effects to consumer preconditions.

Output: plan.json (dict matching Plan.to_dict()).
"""
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

from llm_client import chat_json
from plan_types import CausalLink, Condition, Effect, Event, Plan


# ---------------------------------------------------------------------------
# Prompts
# ---------------------------------------------------------------------------
EVENT_EXTRACTION_SYSTEM = """You are a plan engineer for an interactive mystery game. Your job is to convert a detective's free-text action into a structured plan event with explicit preconditions and effects over a shared world state.

Use these subject-id conventions:
  character.<snake_case_name>   e.g. character.victoria_harrington
  location.<snake_case>         e.g. location.gallery_main_hall
  object.<snake_case>           e.g. object.fountain_pen
  evidence.<evidence_id>        e.g. evidence.E03
  detective                     (the player; has attrs: location, knowledge, inventory)

Always output valid JSON. Never add explanation outside the JSON."""


EVENT_EXTRACTION_PROMPT = """Convert this detective action into a structured event.

Context:
  victim: {victim}
  detective: {detective}
  suspects: {suspects}
  conspirators: {conspirators}
  available evidence ids: {evidence_ids}

Plot point index: {idx}
Plot type: {plot_type}
Action (free text): {action}
Narrative excerpt (for context only, do not copy verbatim): {narrative_short}

Output JSON with exactly these keys:
{{
  "verb": "interview | examine | visit | search | analyze | consult | confront | observe | reconstruct",
  "args": [string, ...],
  "location": "location.<snake_case>",
  "preconditions": [
    {{"subject": "...", "attr": "...", "op": "==", "value": ...}}, ...
  ],
  "effects": [
    {{"subject": "...", "attr": "...", "op": "set|add|remove", "value": ...}}, ...
  ],
  "reveals": ["evidence.<id>", ...]
}}

Rules:
- At least ONE precondition should reference detective.location == "<location>".
- At least ONE effect should update detective.knowledge (op=add) with what the detective now knows.
- If the action examines or finds a physical object, include an effect on that object.
- If the action interviews a character, include a precondition that the character is available
  (character.<name>.available == true) and an effect updating that character's
  alibi_status or willingness_to_talk.
- Use lowercase snake_case for all subject ids. Keep effects minimal and specific.
- Do NOT invent evidence ids that are not in the provided list."""


CAUSAL_LINK_PROMPT = """You are validating causal links between plan events.

Producer event (E_i) — effects: {producer_effects}
Consumer event (E_j) — preconditions: {consumer_preconditions}

Question: for each consumer precondition, does a producer effect directly establish it?
Output JSON: {{"links": [{{"condition": <precondition object>, "established_by_producer": true|false}}]}}

Return ONLY the JSON."""


# ---------------------------------------------------------------------------
# Extraction pipeline
# ---------------------------------------------------------------------------
def _slug(s: str) -> str:
    import re
    out = re.sub(r"[^a-zA-Z0-9]+", "_", s).strip("_").lower()
    return out or "unknown"


def _short(text: str, max_chars: int = 320) -> str:
    text = text.strip().replace("\n", " ")
    return text[:max_chars]


def extract_event_from_plot_point(
    idx: int,
    plot: dict[str, Any],
    case_file: dict[str, Any],
) -> Event:
    """Call the LLM once to infer structured event fields for one plot point."""
    ctx = {
        "victim": case_file["victim"]["name"],
        "detective": case_file["detective"]["name"],
        "suspects": [s["name"] for s in case_file["suspects"]],
        "conspirators": [c["name"] for c in case_file["conspirators"]],
        "evidence_ids": [e["id"] for e in case_file["evidence"]],
        "idx": idx,
        "plot_type": plot.get("plot_type", "progress"),
        "action": plot.get("action", ""),
        "narrative_short": _short(plot.get("narrative", "")),
    }
    prompt = EVENT_EXTRACTION_PROMPT.format(**ctx)
    try:
        parsed = chat_json(
            prompt,
            system=EVENT_EXTRACTION_SYSTEM,
            max_tokens=900,
            temperature=0.3,
        )
    except (ValueError, json.JSONDecodeError) as err:
        print(f"  [warn] event {idx} extraction failed ({err!r}); using fallback")
        parsed = _fallback_event_dict(plot)

    event_id = f"E{idx:02d}"
    preconditions = [Condition.from_dict(c) for c in parsed.get("preconditions", [])]
    effects = [Effect.from_dict(e) for e in parsed.get("effects", [])]
    # Guarantee the minimum contract: detective must be at the action location,
    # and the detective learns at least one thing from the event.
    location = parsed.get("location") or "location.unknown"
    if not any(pc.subject == "detective" and pc.attr == "location" for pc in preconditions):
        preconditions.append(Condition("detective", "location", "==", location))
    if not any(ef.subject == "detective" and ef.attr == "knowledge" for ef in effects):
        effects.append(Effect("detective", "knowledge", "add", f"learned_from_{event_id}"))

    return Event(
        id=event_id,
        actor="detective",
        verb=parsed.get("verb", "act"),
        args=list(parsed.get("args", [])),
        location=location,
        preconditions=preconditions,
        effects=effects,
        reveals=list(parsed.get("reveals", [])),
        description=plot.get("action", ""),
        narrative=plot.get("narrative", ""),
        source_plot_idx=idx,
    )


def _fallback_event_dict(plot: dict[str, Any]) -> dict[str, Any]:
    """Deterministic fallback when the LLM returns unparseable JSON."""
    return {
        "verb": "act",
        "args": [],
        "location": "location.unknown",
        "preconditions": [],
        "effects": [],
        "reveals": [],
    }


# ---------------------------------------------------------------------------
# Causal link derivation
# ---------------------------------------------------------------------------
def _conditions_match(effect: Effect, precondition: Condition) -> bool:
    """Does this effect establish this precondition?"""
    if effect.subject != precondition.subject or effect.attr != precondition.attr:
        return False
    if precondition.op == "==" and effect.op == "set":
        return effect.value == precondition.value
    if precondition.op == "contains" and effect.op == "add":
        return effect.value == precondition.value
    if precondition.op == "!=" and effect.op == "set":
        return effect.value != precondition.value
    return False


def derive_causal_links(events: list[Event]) -> list[CausalLink]:
    """Walk events in order; for each consumer precondition, find the nearest
    prior event whose effect establishes it. That pair becomes a causal link.

    We augment the strict effect↔precondition match with two heuristics so the
    plan isn't left link-less when the LLM's surface form drifts:

    1. **Reveal → reference**: if event E_i reveals evidence.X and a later
       event E_j references evidence.X (in args or reveals), add a link
       `(E_i, evidence.X.discovered == True, E_j)`. The evidence must stay
       discovered across the span — it being destroyed breaks the link.
    2. **Evidence analyzed → referenced**: if event E_i has an effect setting
       evidence.X.analyzed=true and a later event references X, add a link
       `(E_i, evidence.X.analyzed == True, E_j)`.
    """
    links: list[CausalLink] = []
    seen: set[tuple[str, str, str]] = set()  # (producer, consumer, attr) dedupe key

    def _add(producer: str, consumer: str, condition: Condition) -> None:
        key = (producer, consumer, f"{condition.subject}:{condition.attr}:{condition.value}")
        if key in seen:
            return
        seen.add(key)
        links.append(CausalLink(producer=producer, condition=condition, consumer=consumer))

    # Pass 1: strict effect↔precondition match.
    for j, consumer in enumerate(events):
        for pc in consumer.preconditions:
            for i in range(j - 1, -1, -1):
                producer = events[i]
                if any(_conditions_match(ef, pc) for ef in producer.effects):
                    _add(producer.id, consumer.id, pc)
                    break

    # Pass 2: reveal → reference heuristic.
    def _refs_to_evidence(ev: Event) -> set[str]:
        refs: set[str] = set(ev.reveals)
        for a in ev.args:
            if isinstance(a, str) and a.startswith("evidence."):
                refs.add(a)
        return refs

    for i, producer in enumerate(events):
        for ev_id in producer.reveals:
            if not ev_id.startswith("evidence."):
                continue
            # Consumer = next event that also references this evidence id.
            for j in range(i + 1, len(events)):
                if ev_id in _refs_to_evidence(events[j]):
                    _add(producer.id, events[j].id,
                         Condition(ev_id, "discovered", "==", True))

    # Pass 3: analyzed → reference.
    for i, producer in enumerate(events):
        for ef in producer.effects:
            if ef.subject.startswith("evidence.") and ef.attr == "analyzed" and ef.op == "set" and ef.value is True:
                for j in range(i + 1, len(events)):
                    if ef.subject in _refs_to_evidence(events[j]):
                        _add(producer.id, events[j].id,
                             Condition(ef.subject, "analyzed", "==", True))

    return links


# ---------------------------------------------------------------------------
# Initial state construction
# ---------------------------------------------------------------------------
def build_initial_state(case_file: dict[str, Any]) -> dict[str, dict[str, Any]]:
    """Seed world state with characters, evidence, and the detective.

    We don't yet know locations — world_builder.py will pin each character
    and evidence item to a starting location once the location graph exists.
    """
    state: dict[str, dict[str, Any]] = {}

    state["detective"] = {
        "location": "location.unknown",
        "knowledge": [],
        "inventory": [],
        "alive": True,
    }

    for s in case_file["suspects"]:
        sid = f"character.{_slug(s['name'])}"
        state[sid] = {
            "name": s["name"],
            "role": "suspect",
            "available": True,
            "alibi_status": "unverified",
            "alive": True,
            "willingness_to_talk": "neutral",
        }
    for c in case_file["conspirators"]:
        cid = f"character.{_slug(c['name'])}"
        state[cid] = {
            "name": c["name"],
            "role": "conspirator",
            "available": True,
            "alibi_status": "unverified",
            "alive": True,
            "willingness_to_talk": "evasive",
        }
    state[f"character.{_slug(case_file['victim']['name'])}"] = {
        "name": case_file["victim"]["name"],
        "role": "victim",
        "available": False,
        "alive": False,
    }

    for ev in case_file["evidence"]:
        eid = f"evidence.{ev['id']}"
        state[eid] = {
            "type": ev.get("type", "physical"),
            "description": ev.get("description", ""),
            "discovered": False,
            "analyzed": False,
            "destroyed": False,
            "location": "location.unknown",
        }

    return state


def build_goal(case_file: dict[str, Any]) -> list[Condition]:
    """Detective wins when they have identified the real criminal and linked
    the key evidence to that person (by name)."""
    real = _slug(case_file["criminal"]["name"])
    goal = [
        Condition("detective", "knowledge", "contains", f"identified:character.{real}"),
        Condition("detective", "knowledge", "contains", f"linked_evidence:character.{real}"),
    ]
    return goal


# ---------------------------------------------------------------------------
# Top-level driver
# ---------------------------------------------------------------------------
def build_plan(
    case_file: dict[str, Any],
    plot_points: list[dict[str, Any]],
    out_path: str | Path | None = None,
) -> Plan:
    print(f"Extracting events from {len(plot_points)} plot points...")
    events: list[Event] = []
    for idx, plot in enumerate(plot_points):
        ev = extract_event_from_plot_point(idx, plot, case_file)
        events.append(ev)
        print(f"  {ev.id} verb={ev.verb:10s} loc={ev.location:32s} "
              f"pre={len(ev.preconditions)} eff={len(ev.effects)} reveals={ev.reveals}")

    print("Deriving causal links...")
    links = derive_causal_links(events)
    print(f"  {len(links)} causal links derived")

    order = [(events[i].id, events[i + 1].id) for i in range(len(events) - 1)]

    plan = Plan(
        events={ev.id: ev for ev in events},
        order=order,
        causal_links=links,
        initial_state=build_initial_state(case_file),
        goal=build_goal(case_file),
    )

    if out_path:
        out = Path(out_path)
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(json.dumps(plan.to_dict(), indent=2), encoding="utf-8")
        print(f"Saved: {out}")

    return plan


def load_plan(path: str | Path) -> Plan:
    return Plan.from_dict(json.loads(Path(path).read_text(encoding="utf-8")))


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", default="data")
    parser.add_argument("--out", default="data/plan.json")
    args = parser.parse_args()

    data_dir = Path(args.data_dir)
    case_file = json.loads((data_dir / "case_file.json").read_text())
    plot_points = json.loads((data_dir / "plot_points.json").read_text())
    build_plan(case_file, plot_points, out_path=args.out)


In [ ]:
%%writefile world_builder.py
"""Build a graph-based world from a plan.

For every distinct location referenced in plan events, we create a node.
Adjacency is decided by a commonsense LLM call: two locations are adjacent
if someone would reasonably walk between them without crossing a third
place. If two *consecutive* events occur in locations that wouldn't be
adjacent in the real world (bedroom → restaurant), we insert one or two
intermediate locations so the user can actually traverse.

We also populate each location with the characters and evidence the plan
expects to be there, and initialize the detective at the story's first
location.
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from llm_client import chat_json
from plan_types import Plan


@dataclass
class Location:
    id: str
    name: str
    description: str = ""
    adjacent: set[str] = field(default_factory=set)
    characters: set[str] = field(default_factory=set)
    evidence: set[str] = field(default_factory=set)

    def to_dict(self) -> dict[str, Any]:
        return {
            "id": self.id,
            "name": self.name,
            "description": self.description,
            "adjacent": sorted(self.adjacent),
            "characters": sorted(self.characters),
            "evidence": sorted(self.evidence),
        }

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "Location":
        return cls(
            id=d["id"],
            name=d["name"],
            description=d.get("description", ""),
            adjacent=set(d.get("adjacent", [])),
            characters=set(d.get("characters", [])),
            evidence=set(d.get("evidence", [])),
        )


@dataclass
class World:
    locations: dict[str, Location] = field(default_factory=dict)
    starting_location: str = ""

    def to_dict(self) -> dict[str, Any]:
        return {
            "locations": {lid: loc.to_dict() for lid, loc in self.locations.items()},
            "starting_location": self.starting_location,
        }

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "World":
        return cls(
            locations={lid: Location.from_dict(v) for lid, v in d["locations"].items()},
            starting_location=d.get("starting_location", ""),
        )


# ---------------------------------------------------------------------------
# Prompts
# ---------------------------------------------------------------------------
LOCATION_DESCRIBE_PROMPT = """Describe this mystery-story location in 1-2 sentences of atmospheric detail.
Location id: {loc_id}
Context (story era/setting): {era}
Output JSON: {{"name": "...", "description": "..."}}"""


ADJACENCY_PROMPT = """You are a commonsense spatial reasoner for a 1920s London murder mystery.

Below is a list of locations that appear in the story, in the order the
detective must visit them. For each pair of *consecutive* locations, answer
whether they would naturally be adjacent (someone could walk directly from
one to the other without traversing a third distinct place) and if not,
propose 1-2 intermediate locations that bridge them.

Locations: {locations}

Output JSON:
{{
  "pairs": [
    {{"a": "<id>", "b": "<id>",
      "adjacent": true|false,
      "intermediates": ["location.<snake_case>", ...]}},
    ...
  ],
  "extra_intermediate_descriptions": {{
    "location.<id>": "short description",
    ...
  }}
}}

Rules:
- Include one entry per consecutive pair.
- If adjacent is true, intermediates must be [].
- Only invent intermediate locations when the pair clearly wouldn't be adjacent."""


# ---------------------------------------------------------------------------
# Builder
# ---------------------------------------------------------------------------
def _describe_location(loc_id: str, era: str) -> dict[str, str]:
    try:
        return chat_json(
            LOCATION_DESCRIBE_PROMPT.format(loc_id=loc_id, era=era),
            max_tokens=200,
            temperature=0.6,
        )
    except Exception:
        pretty = loc_id.split(".", 1)[-1].replace("_", " ").title()
        return {"name": pretty, "description": f"A location in the case: {pretty}."}


def _analyze_adjacency(loc_ids: list[str]) -> dict[str, Any]:
    try:
        return chat_json(
            ADJACENCY_PROMPT.format(locations=loc_ids),
            max_tokens=1500,
            temperature=0.3,
        )
    except Exception as err:
        print(f"  [warn] adjacency prompt failed ({err!r}); falling back to linear chain")
        return {
            "pairs": [{"a": a, "b": b, "adjacent": True, "intermediates": []}
                      for a, b in zip(loc_ids[:-1], loc_ids[1:])],
            "extra_intermediate_descriptions": {},
        }


def build_world(plan: Plan, era: str = "1920s London") -> World:
    event_ids_in_order = sorted(plan.events.keys())
    loc_sequence: list[str] = []
    for eid in event_ids_in_order:
        loc = plan.events[eid].location or "location.unknown"
        loc_sequence.append(loc)

    # Dedupe while preserving order, but remember the sequential visits so
    # we can reason about consecutive-pair adjacency.
    unique_locs = list(dict.fromkeys(loc_sequence))
    print(f"Unique plan locations: {len(unique_locs)}")

    world = World()
    for lid in unique_locs:
        info = _describe_location(lid, era)
        world.locations[lid] = Location(id=lid, name=info.get("name", lid), description=info.get("description", ""))

    # Consecutive pairs along the event trajectory — these are the ones that
    # must be reachable by the player without teleporting.
    pairs: list[tuple[str, str]] = []
    for a, b in zip(loc_sequence[:-1], loc_sequence[1:]):
        if a != b and (a, b) not in pairs and (b, a) not in pairs:
            pairs.append((a, b))
    print(f"Consecutive location transitions to resolve: {len(pairs)}")

    ids_for_llm = list({lid for pair in pairs for lid in pair})
    adjacency = _analyze_adjacency(ids_for_llm) if ids_for_llm else {"pairs": [], "extra_intermediate_descriptions": {}}

    # Insert intermediates first so we can describe them before wiring edges.
    for mid_id, mid_desc in (adjacency.get("extra_intermediate_descriptions") or {}).items():
        if mid_id not in world.locations:
            pretty = mid_id.split(".", 1)[-1].replace("_", " ").title()
            world.locations[mid_id] = Location(id=mid_id, name=pretty, description=mid_desc)

    verdict_by_pair = {(p["a"], p["b"]): p for p in adjacency.get("pairs", []) if "a" in p and "b" in p}

    for a, b in pairs:
        verdict = verdict_by_pair.get((a, b)) or verdict_by_pair.get((b, a))
        if verdict and not verdict.get("adjacent", True):
            mids = [m for m in verdict.get("intermediates", []) if isinstance(m, str)]
            chain = [a, *mids, b]
            for mid in mids:
                if mid not in world.locations:
                    pretty = mid.split(".", 1)[-1].replace("_", " ").title()
                    world.locations[mid] = Location(id=mid, name=pretty, description=f"A passage between areas of the story.")
            for x, y in zip(chain[:-1], chain[1:]):
                world.locations[x].adjacent.add(y)
                world.locations[y].adjacent.add(x)
        else:
            world.locations[a].adjacent.add(b)
            world.locations[b].adjacent.add(a)

    # Seed contents: every event that reveals an evidence places that
    # evidence in the event's location; characters mentioned in args are
    # placed there too.
    for ev in plan.events.values():
        loc_id = ev.location
        if loc_id in world.locations:
            for rev in ev.reveals:
                world.locations[loc_id].evidence.add(rev)

    # Pin characters to the location of the first event that references them.
    for ev in plan.events.values():
        for arg in ev.args:
            if arg.startswith("character.") and ev.location in world.locations:
                # Only set if not already placed (first mention wins).
                already_placed = any(arg in loc.characters for loc in world.locations.values())
                if not already_placed:
                    world.locations[ev.location].characters.add(arg)

    world.starting_location = loc_sequence[0] if loc_sequence else next(iter(world.locations))
    if world.starting_location not in world.locations:
        # Unknown fallback — create a generic "case_briefing_room".
        world.locations["location.case_briefing_room"] = Location(
            id="location.case_briefing_room",
            name="Case Briefing Room",
            description="A quiet room where the detective sorts notes between outings.",
        )
        for lid in list(world.locations.keys()):
            if lid == "location.case_briefing_room":
                continue
            world.locations["location.case_briefing_room"].adjacent.add(lid)
            world.locations[lid].adjacent.add("location.case_briefing_room")
        world.starting_location = "location.case_briefing_room"

    # Also hook initial_state["detective"]["location"] to the start.
    plan.initial_state.setdefault("detective", {})["location"] = world.starting_location

    # Make sure every evidence entry has its "location" attr patched from the world.
    for loc in world.locations.values():
        for ev_id in loc.evidence:
            if ev_id in plan.initial_state:
                plan.initial_state[ev_id]["location"] = loc.id

    return world


def save_world(world: World, path: str | Path) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(world.to_dict(), indent=2), encoding="utf-8")
    print(f"Saved: {p}")


def load_world(path: str | Path) -> World:
    return World.from_dict(json.loads(Path(path).read_text(encoding="utf-8")))


if __name__ == "__main__":
    import argparse

    from story_to_plan import load_plan

    parser = argparse.ArgumentParser()
    parser.add_argument("--plan", default="data/plan.json")
    parser.add_argument("--out", default="data/world.json")
    args = parser.parse_args()

    plan = load_plan(args.plan)
    world = build_world(plan)
    save_world(world, args.out)
    # Persist the patched initial_state back into plan.json so game_engine
    # sees the resolved starting location.
    Path(args.plan).write_text(json.dumps(plan.to_dict(), indent=2), encoding="utf-8")
    print(f"Updated: {args.plan}")


In [ ]:
%%writefile action_interpreter.py
"""LLM-based parser that maps free-form user input into a structured action.

The parser returns a dict of the same shape as a plan `Event` (minus the id):
verb, args, location, preconditions, effects, reveals. The drama manager
then checks preconditions and either executes the action, flags it as an
exception, or triggers accommodation.

The parser *does not* know about causal links. Only about surface effects
it can infer from commonsense. The drama manager is the one that decides
whether those effects constitute a story exception.
"""
from __future__ import annotations

import json
from typing import Any

from llm_client import chat_json
from plan_types import Condition, Effect


PARSE_SYSTEM = """You are an action interpreter for a text adventure. Convert the player's short command into a structured action over the game world. Output only valid JSON. Never write explanation."""


PARSE_PROMPT = """World summary:
  Player location: {player_location}
  Adjacent locations: {adjacent}
  Objects here: {here_objects}
  Characters here: {here_characters}
  Known evidence ids: {evidence_ids}
  Player inventory: {inventory}
  Player knowledge snippets (partial): {knowledge_snippets}

Subject id conventions:
  detective                       the player
  character.<snake_name>
  location.<snake>
  evidence.<id>
  object.<snake>

Player's raw input (<=5 words expected, may exceed): "{raw}"

Produce this JSON:
{{
  "verb": "move | examine | interview | search | take | drop | use | analyze | confront | accuse | observe | wait | custom",
  "args": ["<subject_id or short phrase>", ...],
  "target_location": "<location id or empty string>",
  "preconditions": [{{"subject":"...", "attr":"...", "op":"==", "value":...}}, ...],
  "effects": [{{"subject":"...", "attr":"...", "op":"set|add|remove", "value":...}}, ...],
  "reveals": ["evidence.<id>", ...],
  "novel_state_vars": [
    {{"subject":"...", "attr":"...", "why":"short reason why this new state matters"}}, ...
  ],
  "plain_summary": "one-sentence description of what the player tries to do"
}}

Critical rules:
- For movement, set verb="move" and fill target_location with an adjacent id.
- For any physical manipulation that creates a NEW persistent state on an object
  (jamming, breaking, locking, hiding, destroying), list that state under
  "novel_state_vars" in addition to listing it in effects. This tells the
  drama manager the action introduced a state slot not previously modeled.
- If the player tries to "accuse" or "arrest" someone, add an effect updating
  detective.knowledge by adding "accused:<character.id>".
- Keep preconditions minimal — usually just where the player and target must be."""


def interpret_action(
    raw_input: str,
    world_summary: dict[str, Any],
) -> dict[str, Any]:
    """Parse one user command into a structured action dict.

    `world_summary` should include: player_location, adjacent (list of ids),
    here_objects, here_characters, evidence_ids, inventory, knowledge_snippets.
    """
    prompt = PARSE_PROMPT.format(
        raw=raw_input.strip(),
        player_location=world_summary.get("player_location", ""),
        adjacent=world_summary.get("adjacent", []),
        here_objects=world_summary.get("here_objects", []),
        here_characters=world_summary.get("here_characters", []),
        evidence_ids=world_summary.get("evidence_ids", []),
        inventory=world_summary.get("inventory", []),
        knowledge_snippets=world_summary.get("knowledge_snippets", [])[-6:],
    )
    try:
        parsed = chat_json(prompt, system=PARSE_SYSTEM, max_tokens=700, temperature=0.2)
    except (ValueError, json.JSONDecodeError):
        parsed = {
            "verb": "custom",
            "args": [raw_input.strip()],
            "target_location": "",
            "preconditions": [],
            "effects": [],
            "reveals": [],
            "novel_state_vars": [],
            "plain_summary": raw_input.strip(),
        }
    parsed.setdefault("novel_state_vars", [])
    parsed["_raw"] = raw_input
    return parsed


def structured_preconditions(parsed: dict[str, Any]) -> list[Condition]:
    out: list[Condition] = []
    for pc in parsed.get("preconditions", []):
        try:
            out.append(Condition.from_dict(pc))
        except Exception:  # noqa: BLE001 — malformed LLM output, skip this precondition only
            continue
    return out


def structured_effects(parsed: dict[str, Any]) -> list[Effect]:
    out: list[Effect] = []
    for ef in parsed.get("effects", []):
        try:
            out.append(Effect.from_dict(ef))
        except Exception:  # noqa: BLE001 — malformed LLM output, skip this effect only
            continue
    return out


In [ ]:
%%writefile drama_manager.py
"""Intervention and Accommodation drama manager (Template 2).

Classifies every user action as constituent / consistent / exceptional and
repairs the plan when exceptions occur.

Key design points:

1. **Open-action problem**. The interpreter may produce effects on state
   variables that were never in the original plan (e.g. jam door with
   chair). Commonsense says those still threaten future events — a blocked
   door blocks anyone who must pass through it, even if no pre-declared
   causal link mentions "jammed". We solve this by running a *commonsense
   threat query* for every user action: we ask the LLM, given the effects
   and the remaining plan events, whether any future event would become
   impossible under normal physical/social reasoning. If yes, the action
   is exceptional even when no hard causal link is broken.

2. **Accommodation**. When something is exceptional, we remove the events
   that are no longer reachable and the links that depended on them, then
   ask the LLM to generate replacement events that re-establish the goal
   preconditions from the current world state. Replacement events are
   constrained to the existing world (characters + locations); they may
   modify *unrevealed* crime details but not the original crime outcome.

3. **Inspectable logs**. Every decision — classification, threat query,
   removed events, replacement events — is appended to an in-memory log
   and a JSONL file, so the final video can scroll the log for evidence of
   behind-the-scenes behavior.
"""
from __future__ import annotations

import json
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from llm_client import chat_json
from plan_types import CausalLink, Condition, Effect, Event, Plan


# ---------------------------------------------------------------------------
# Verb synonym families & token helpers for constituent matching
# ---------------------------------------------------------------------------

# Each frozenset is a family of interchangeable verbs. If the plan event verb
# and the parsed verb land in the same family, they are treated as equivalent.
_VERB_FAMILIES: list[frozenset[str]] = [
    # physical inspection — examine / check / inspect / search / investigate all count
    frozenset({"examine", "check", "inspect", "observe", "look", "study",
               "review", "search", "investigate", "explore", "scour"}),
    # scientific analysis
    frozenset({"analyze", "analyse", "test", "process"}),
    # social interaction — question / interview / consult / confront / accuse / visit all count
    frozenset({"interview", "question", "talk", "speak", "ask",
               "consult", "confront", "accuse", "arrest", "charge", "visit", "meet"}),
    # movement
    frozenset({"move", "go", "walk", "travel", "head"}),
]

_STOP = frozenset({
    "the", "and", "for", "with", "from", "that", "this", "are", "was",
    "has", "have", "been", "his", "her", "its", "into", "upon", "over",
    "under", "around", "within", "when", "then", "also", "only", "just",
})


def _verb_in_same_family(ev_verb: str, parsed_verb: str) -> bool:
    """Return True if two verbs belong to the same synonym family."""
    if ev_verb == parsed_verb:
        return True
    for family in _VERB_FAMILIES:
        if ev_verb in family and parsed_verb in family:
            return True
    return False


def _tok(text: str) -> set[str]:
    """Lowercase content tokens: ≥4 chars, not a stop word."""
    return {t for t in re.split(r"\W+", text.lower())
            if len(t) >= 4 and t not in _STOP}


# Verbs/keywords that are always exceptional for a detective — no LLM call
# needed to decide; the keyword match is sufficient.
_DESTRUCTIVE_VERBS: frozenset[str] = frozenset({
    "burn", "destroy", "smash", "break", "shatter", "tamper",
    "contaminate", "conceal", "steal", "forge", "alter", "corrupt",
    "shoot", "kill", "attack", "assault", "threaten", "bribe",
    "dispose", "hide", "cover",
})
_DESTRUCTIVE_RE = re.compile(
    r"\b(" + "|".join(re.escape(v) for v in sorted(_DESTRUCTIVE_VERBS)) + r")\b",
    re.IGNORECASE,
)


# ---------------------------------------------------------------------------
# Log entry
# ---------------------------------------------------------------------------
@dataclass
class LogEntry:
    kind: str
    payload: dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> dict[str, Any]:
        return {"kind": self.kind, **self.payload}


# ---------------------------------------------------------------------------
# Commonsense threat / accommodation prompts
# ---------------------------------------------------------------------------
THREAT_SYSTEM = """You are a commonsense reasoner for an interactive detective story.
You assess whether a just-performed action makes future plan events impossible.
Output only valid JSON."""


THREAT_PROMPT = """User action (structured):
{parsed_action}

Remaining plan events (summarized; id, verb, args, location, key preconditions):
{remaining_events}

Active causal link conditions (must hold until the listed consumer event):
{active_links}

Question: does the user's action render any remaining event impossible or
meaningfully harder for a realistic actor (detective or NPC) to perform,
even if no hard causal link is formally broken?

Consider physical blocks ("jammed", "locked permanently", "broken"),
social blocks ("witness dead", "suspect fled the country"), and epistemic
blocks ("evidence destroyed").

Output JSON:
{{
  "threatened_events": [
    {{"event_id": "<id>",
      "reason": "short commonsense reason",
      "repairable": true|false}},
    ...
  ],
  "overall_classification_hint": "constituent | consistent | exceptional"
}}"""


ACCOMMODATION_SYSTEM = """You are a partial-order-plan repair agent. You output only valid JSON.
Your job: given a broken detective plan and the current world state, propose
the MINIMUM number of replacement events that restore the detective's path to
the goal. You may modify unrevealed crime details but never the core outcome.
CRITICAL RULES — violating any of these makes your output unusable:
1. ONLY use characters from the provided characters list. Never invent new people.
2. ONLY use locations from the provided locations list. Never invent new places.
3. Prefer 1 replacement event. Use 2 only if truly necessary. Never use 3+.
4. If the goal is already reachable through surviving events, output 0 replacements.
5. Each replacement must be immediately achievable by the detective with current resources."""


ACCOMMODATION_PROMPT = """Current world snapshot: {world_snapshot}

Goal conditions that must still be reachable: {goal}

Events just removed (unreachable due to exceptional action): {removed_events}
Surviving causal links: {surviving_links}

ALLOWED characters (use ONLY these exact ids): {characters}
ALLOWED locations (use ONLY these exact ids): {locations}
Undestroyed evidence: {live_evidence}

If the goal conditions are still satisfiable by the surviving events alone,
output zero replacement events (empty list). Otherwise produce the minimum
replacements needed — usually just 1.

Output JSON:
{{
  "replacement_events": [
    {{"id": "R01",
      "verb": "interview|examine|search|analyze|consult|confront",
      "args": ["<one of the ALLOWED character or evidence ids above>"],
      "location": "<one of the ALLOWED location ids above>",
      "preconditions": [],
      "effects": [{{"subject":"detective", "attr":"knowledge", "op":"add", "value":"<slug>"}}],
      "reveals": [],
      "description": "one sentence the detective would do",
      "narrative": "one short paragraph of 1920s-noir prose, staying in the established story context"
    }}
  ],
  "rationale": "one sentence explaining the repair"
}}

Hard constraints:
- Do NOT invent characters, locations, or evidence IDs not in the ALLOWED lists above.
- Do NOT reveal the real criminal. Preserve mystery.
- Keep effects minimal — one or two knowledge additions at most."""


# ---------------------------------------------------------------------------
# Drama manager
# ---------------------------------------------------------------------------
class DramaManager:
    def __init__(self, plan: Plan, world: Any = None, log_path: str | Path = "logs/drama.jsonl") -> None:
        self.plan = plan
        self.world = world
        self.executed: list[str] = []
        self.remaining: list[str] = sorted(plan.events.keys())
        self.log_path = Path(log_path)
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        self.log: list[LogEntry] = []
        self._next_repair_idx = 1
        # Build predecessor index from plan.order for ordering enforcement.
        # predecessors[eid] = {set of event ids that must execute before eid}
        self._predecessors: dict[str, set[str]] = {eid: set() for eid in plan.events}
        for producer, consumer in plan.order:
            if consumer in self._predecessors:
                self._predecessors[consumer].add(producer)

    # ---- logging ---------------------------------------------------------
    def _log(self, kind: str, **payload: Any) -> None:
        entry = LogEntry(kind=kind, payload=payload)
        self.log.append(entry)
        with self.log_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(entry.to_dict(), default=str) + "\n")

    # ---- active causal links ---------------------------------------------
    def active_links(self) -> list[CausalLink]:
        """Links whose producer has fired but whose consumer has not."""
        out: list[CausalLink] = []
        for cl in self.plan.causal_links:
            if cl.producer in self.executed and cl.consumer not in self.executed and cl.consumer in self.remaining:
                out.append(cl)
        return out

    # ---- classification --------------------------------------------------
    def classify(
        self,
        parsed_action: dict[str, Any],
        state: dict[str, dict[str, Any]],
    ) -> dict[str, Any]:
        """Return {classification, matched_event_id|None, threats, details}."""
        constituent_match = self._find_constituent_match(parsed_action, state)
        proposed_effects = []
        for _e in parsed_action.get("effects", []):
            if not isinstance(_e, dict):
                continue
            try:
                proposed_effects.append(Effect.from_dict(_e))
            except (KeyError, TypeError):
                pass  # malformed LLM effect — skip
        hard_violations = self._hard_violations(proposed_effects)

        # Detect destructive inputs by keyword — these are always exceptional
        # regardless of whether the LLM action interpreter modelled any effects.
        raw_input = (parsed_action.get("_raw") or "").lower()
        is_destructive = (
            parsed_action.get("verb", "").lower() in _DESTRUCTIVE_VERBS
            or bool(_DESTRUCTIVE_RE.search(raw_input))
        )

        # Commonsense threat check — skip when a hard violation is already
        # confirmed; always run for destructive inputs (effects may be empty).
        soft_threats: list[dict[str, Any]] = []
        cs_hint = "consistent"
        needs_cs = not hard_violations and (
            is_destructive or proposed_effects or parsed_action.get("novel_state_vars")
        )
        if needs_cs:
            soft_threats, cs_hint = self._commonsense_threats(parsed_action)

        # Destructive inputs with no constituent match are always exceptional,
        # overriding a "consistent" commonsense hint.
        if is_destructive and not constituent_match:
            cs_hint = "exceptional"

        classification = self._decide_classification(constituent_match, hard_violations, soft_threats, cs_hint)

        result = {
            "classification": classification,
            "matched_event_id": constituent_match,
            "hard_violations": [cl.to_dict() for cl in hard_violations],
            "soft_threats": soft_threats,
        }
        self._log(
            "classification",
            action=parsed_action,
            classification=classification,
            matched_event_id=constituent_match,
            hard_violations=[cl.to_dict() for cl in hard_violations],
            soft_threats=soft_threats,
            cs_hint=cs_hint,
            active_links=[cl.to_dict() for cl in self.active_links()],
            remaining_count=len(self.remaining),
            executed_count=len(self.executed),
        )
        return result

    def _find_constituent_match(
        self,
        parsed_action: dict[str, Any],
        state: dict[str, dict[str, Any]],
    ) -> str | None:
        """Return the first remaining event whose verb + args substantially
        overlap with the parsed action, respecting the detective's current
        location and character availability."""
        raw = (parsed_action.get("_raw") or "").lower()
        parsed_verb = parsed_action.get("verb", "").lower()
        cur_loc = state.get("detective", {}).get("location", "")

        for eid in self.remaining:
            ev = self.plan.events[eid]

            # Skip if the event is tied to a different location.
            if ev.location and cur_loc and ev.location != cur_loc:
                continue

            # Respect partial-order constraints: all declared predecessors must
            # already be executed (or absent from the plan) before this event
            # can match.
            required = self._predecessors.get(eid, set())
            if required and not required.issubset(set(self.executed) | (set(self.plan.events) - set(self.remaining))):
                continue

            # Skip if any required character is not at the detective's location
            # or is no longer available.
            if self.world and cur_loc:
                loc_obj = self.world.locations.get(cur_loc)
                chars_here = set(loc_obj.characters) if loc_obj else set()
                ev_chars = {str(a) for a in ev.args if str(a).startswith("character.")}
                if ev_chars:
                    available_here = {
                        cid for cid in chars_here
                        if state.get(cid, {}).get("alive", True)
                        and state.get(cid, {}).get("available", True)
                    }
                    if not ev_chars & available_here:
                        continue

            ev_verb = ev.verb.lower()
            verb_matches = (not parsed_verb) or _verb_in_same_family(ev_verb, parsed_verb)

            # Token-level content match: tolerates LLM paraphrasing and the
            # mismatch between short user words ("ring", "body") and full plan
            # arg phrases ("ladies' ring with flower design").
            ev_tokens = _tok(" ".join(str(a) for a in ev.args) + " " + ev.description)
            input_tokens = _tok(raw) | _tok(" ".join(str(a) for a in parsed_action.get("args", [])))
            content_match = bool(input_tokens & ev_tokens)

            if verb_matches and content_match:
                return eid
            # Softer fallback: strong content match even if verb is off
            # (guards against LLM verb drift, e.g. "observe" for "search").
            if content_match and len(input_tokens & ev_tokens) >= 2:
                return eid
        return None

    def _hard_violations(self, proposed_effects: list[Effect]) -> list[CausalLink]:
        """Which active causal links are negated by these effects?"""
        violated: list[CausalLink] = []
        for cl in self.active_links():
            for ef in proposed_effects:
                if self._effect_negates_condition(ef, cl.condition):
                    violated.append(cl)
                    break
        return violated

    @staticmethod
    def _effect_negates_condition(effect: Effect, condition: Condition) -> bool:
        if effect.subject != condition.subject or effect.attr != condition.attr:
            return False
        if condition.op == "==" and effect.op == "set":
            return effect.value != condition.value
        if condition.op == "!=" and effect.op == "set":
            return effect.value == condition.value
        if condition.op == "contains" and effect.op == "remove":
            return effect.value == condition.value
        if condition.op == "not_contains" and effect.op == "add":
            return effect.value == condition.value
        return False

    def _commonsense_threats(
        self, parsed_action: dict[str, Any]
    ) -> tuple[list[dict[str, Any]], str]:
        """Return (threatened_events, overall_classification_hint)."""
        remaining_summary = [
            {
                "id": eid,
                "verb": self.plan.events[eid].verb,
                "args": self.plan.events[eid].args,
                "location": self.plan.events[eid].location,
                "preconditions": [pc.to_dict() for pc in self.plan.events[eid].preconditions],
            }
            for eid in self.remaining[:10]
        ]
        active = [cl.to_dict() for cl in self.active_links()]
        prompt = THREAT_PROMPT.format(
            parsed_action=json.dumps(parsed_action)[:1200],
            remaining_events=json.dumps(remaining_summary)[:2500],
            active_links=json.dumps(active)[:1500],
        )
        try:
            parsed = chat_json(prompt, system=THREAT_SYSTEM, max_tokens=600, temperature=0.2)
        except Exception as err:  # noqa: BLE001 — falling back is safer than crashing
            self._log("threat_query_failed", error=repr(err))
            return [], "consistent"
        if not isinstance(parsed, dict):
            return [], "consistent"
        threats = parsed.get("threatened_events", [])
        hint = parsed.get("overall_classification_hint", "consistent")
        return threats, hint

    @staticmethod
    def _decide_classification(
        constituent_match: str | None,
        hard_violations: list[CausalLink],
        soft_threats: list[dict[str, Any]],
        cs_hint: str = "consistent",
    ) -> str:
        # Hard causal-link violations always win — the plan structure is broken.
        if hard_violations:
            return "exceptional"
        # A constituent match means the player is performing exactly the planned
        # action. Soft threats from the commonsense reasoner should yield to an
        # explicit plan match; they only matter for free (non-plan) actions.
        if constituent_match:
            return "constituent"
        # For free actions, escalate on commonsense threats.
        if soft_threats or cs_hint == "exceptional":
            return "exceptional"
        return "consistent"

    # ---- execution -------------------------------------------------------
    def execute_constituent(self, event_id: str, state: dict[str, dict[str, Any]]) -> None:
        event = self.plan.events[event_id]
        for ef in event.effects:
            ef.apply(state)
        self.executed.append(event_id)
        if event_id in self.remaining:
            self.remaining.remove(event_id)
        self._log(
            "executed_constituent",
            event_id=event_id,
            event_verb=event.verb,
            event_description=event.description,
            effects_applied=[ef.to_dict() for ef in event.effects],
            reveals=list(event.reveals),
            remaining_after=len(self.remaining),
            executed_total=len(self.executed),
        )

    def apply_free_effects(
        self,
        parsed_action: dict[str, Any],
        state: dict[str, dict[str, Any]],
    ) -> None:
        for ef_dict in parsed_action.get("effects", []):
            try:
                Effect.from_dict(ef_dict).apply(state)
            except Exception:  # noqa: BLE001 — malformed LLM output, skip this effect only
                continue
        self._log("applied_free_effects", effects=parsed_action.get("effects", []))

    # ---- accommodation ---------------------------------------------------
    def accommodate(
        self,
        parsed_action: dict[str, Any],
        classification: dict[str, Any],
        state: dict[str, dict[str, Any]],
        world_locations: list[str],
        characters: list[str],
    ) -> dict[str, Any]:
        """Remove unreachable events + generate replacements."""
        threatened_ids = {t["event_id"] for t in classification.get("soft_threats", []) if "event_id" in t}
        for cl_dict in classification.get("hard_violations", []):
            threatened_ids.add(cl_dict.get("consumer"))
        threatened_ids.discard(None)

        # Snapshot plan state before removal for logging.
        plan_before = [
            {"id": eid, "verb": self.plan.events[eid].verb,
             "desc": self.plan.events[eid].description[:70]}
            for eid in self.remaining if eid in self.plan.events
        ]
        active_links_before = [cl.to_dict() for cl in self.active_links()]

        removed_events: list[str] = []
        for eid in list(self.remaining):
            if eid in threatened_ids:
                self.remaining.remove(eid)
                removed_events.append(eid)
        self.plan.causal_links = [
            cl for cl in self.plan.causal_links
            if cl.producer in self.remaining or cl.producer in self.executed
            if cl.consumer in self.remaining
        ]

        # Ask the LLM for replacement events. Keep world snapshot lean.
        live_evidence = [k for k, v in state.items() if k.startswith("evidence.") and not v.get("destroyed", False)]
        prompt = ACCOMMODATION_PROMPT.format(
            world_snapshot=json.dumps(_compact_state(state))[:2500],
            goal=json.dumps([c.to_dict() for c in self.plan.goal]),
            removed_events=json.dumps([self.plan.events[eid].to_dict() for eid in removed_events if eid in self.plan.events])[:2000],
            surviving_links=json.dumps([cl.to_dict() for cl in self.plan.causal_links])[:1500],
            characters=json.dumps(characters),
            locations=json.dumps(world_locations),
            live_evidence=json.dumps(live_evidence),
        )
        try:
            parsed = chat_json(prompt, system=ACCOMMODATION_SYSTEM, max_tokens=1500, temperature=0.4)
        except Exception as err:  # noqa: BLE001 — worst case: skip repair and let goal remain reachable by other paths
            self._log("accommodation_failed", error=repr(err), removed_events=removed_events)
            return {"removed_events": removed_events, "replacement_events": [], "rationale": "repair failed"}

        replacement_dicts = parsed.get("replacement_events", []) if isinstance(parsed, dict) else []
        replacements: list[Event] = []
        for rd in replacement_dicts:
            rid = f"R{self._next_repair_idx:02d}"
            self._next_repair_idx += 1
            try:
                ev = Event(
                    id=rid,
                    actor="detective",
                    verb=rd.get("verb", "act"),
                    args=list(rd.get("args", [])),
                    location=rd.get("location", "location.unknown"),
                    preconditions=[Condition.from_dict(c) for c in rd.get("preconditions", [])],
                    effects=[Effect.from_dict(e) for e in rd.get("effects", [])],
                    reveals=list(rd.get("reveals", [])),
                    description=rd.get("description", ""),
                    narrative=rd.get("narrative", ""),
                )
            except Exception:  # noqa: BLE001 — skip malformed replacement, continue with others
                continue
            self.plan.events[rid] = ev
            self.remaining.append(rid)
            replacements.append(ev)

        plan_after = [
            {"id": eid, "verb": self.plan.events[eid].verb,
             "desc": self.plan.events[eid].description[:70]}
            for eid in self.remaining if eid in self.plan.events
        ]
        self._log(
            "accommodation",
            removed_events=removed_events,
            removed_descriptions=[
                self.plan.events[eid].description for eid in removed_events if eid in self.plan.events
            ],
            replacement_event_ids=[e.id for e in replacements],
            replacement_descriptions=[e.description for e in replacements],
            rationale=parsed.get("rationale", "") if isinstance(parsed, dict) else "",
            plan_before=plan_before,
            plan_after=plan_after,
            active_links_before=active_links_before,
            goal_reachability=self.goal_reachability(),
        )
        return {
            "removed_events": removed_events,
            "replacement_events": [e.to_dict() for e in replacements],
            "rationale": parsed.get("rationale", "") if isinstance(parsed, dict) else "",
        }

    # ---- goal check ------------------------------------------------------
    def goal_satisfied(self, state: dict[str, dict[str, Any]]) -> bool:
        return all(c.evaluate(state) for c in self.plan.goal)

    def goal_reachability(self) -> dict[str, Any]:
        """Heuristic: which goal conditions can still be satisfied by remaining events?"""
        reachable, blocked = [], []
        for cond in self.plan.goal:
            can_satisfy = any(
                any(self._effect_satisfies_condition(ef, cond)
                    for ef in self.plan.events[eid].effects)
                for eid in self.remaining if eid in self.plan.events
            )
            (reachable if can_satisfy else blocked).append(
                f"{cond.subject}.{cond.attr} {cond.op} {cond.value}"
            )
        verdict = "reachable" if not blocked else ("no_path" if not reachable else "partially_blocked")
        return {"reachable": reachable, "blocked": blocked, "verdict": verdict}

    @staticmethod
    def _effect_satisfies_condition(effect: Effect, condition: Condition) -> bool:
        if effect.subject != condition.subject or effect.attr != condition.attr:
            return False
        if condition.op == "==" and effect.op == "set":
            return effect.value == condition.value
        if condition.op == "contains" and effect.op == "add":
            return effect.value == condition.value
        if condition.op == "not_contains" and effect.op == "remove":
            return effect.value == condition.value
        return False

    # ---- turn-start marker (called by GameEngine) ------------------------
    def log_turn_start(self, turn: int, raw: str, location: str) -> None:
        self._log(
            "turn_start",
            turn=turn,
            command=raw,
            detective_location=location,
            remaining_count=len(self.remaining),
            executed_count=len(self.executed),
            remaining_events=[
                {"id": eid, "verb": self.plan.events[eid].verb,
                 "desc": self.plan.events[eid].description[:70]}
                for eid in self.remaining if eid in self.plan.events
            ],
        )


def _compact_state(state: dict[str, dict[str, Any]], max_chars: int = 2000) -> dict[str, Any]:
    """Skip bulky fields so the LLM prompt doesn't blow past context."""
    compact: dict[str, Any] = {}
    for sid, slots in state.items():
        if sid.startswith("evidence.") or sid == "detective" or sid.startswith("character."):
            compact[sid] = slots
        else:
            compact[sid] = {k: v for k, v in slots.items() if k not in {"description"}}
    text = json.dumps(compact)
    if len(text) > max_chars:
        compact = {sid: slots for sid, slots in compact.items() if sid == "detective" or sid.startswith("evidence.")}
    return compact


In [ ]:
%%writefile game_engine.py
"""The text game engine: world state, I/O loop, action execution.

Responsibilities:
  - Own the mutable world state (initialized from plan.initial_state +
    world.locations).
  - Render a brief text description of the player's current location.
  - Call action_interpreter for every line of input.
  - Hand off to drama_manager for classification + accommodation.
  - Apply the resulting effects and narrate them back to the player.
  - Log everything.

The engine tries to stay dumb: it does not invent effects, it only applies
what the interpreter and drama manager produce. That keeps the decision
pipeline inspectable.
"""
from __future__ import annotations

import copy
import json
import re
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

from action_interpreter import interpret_action
from drama_manager import DramaManager
from llm_client import chat_simple
from plan_types import Effect, Plan
from world_builder import World


MAX_WORDS_PER_COMMAND = 8  # loose; spec suggests ~5


@dataclass
class EngineConfig:
    max_turns: int = 80
    prompt_label: str = "> "
    narrate_with_llm: bool = True
    log_dir: Path = Path("logs")
    detective_name: str = "Inspector Rothwell"


@dataclass
class TurnLog:
    turn: int
    raw: str
    parsed: dict[str, Any]
    classification: dict[str, Any]
    effects_applied: list[dict[str, Any]] = field(default_factory=list)
    accommodation: dict[str, Any] | None = None
    narration: str = ""


class GameEngine:
    def __init__(
        self,
        plan: Plan,
        world: World,
        config: EngineConfig | None = None,
    ) -> None:
        self.plan = plan
        self.world = world
        self.config = config or EngineConfig()
        self.config.log_dir.mkdir(parents=True, exist_ok=True)

        self.state: dict[str, dict[str, Any]] = copy.deepcopy(plan.initial_state)
        self.state.setdefault("detective", {})
        self.state["detective"]["location"] = world.starting_location

        # Normalize evidence IDs in every location: some world.json entries omit
        # the "evidence." prefix, which causes lookup failures in the scene sketch.
        for loc in self.world.locations.values():
            loc.evidence = [
                e if e.startswith("evidence.") else "evidence." + e
                for e in loc.evidence
            ]

        # Ensure the criminal character exists in state so name lookups work in
        # scene sketches (the criminal is not added to plan.initial_state by default).
        for loc in self.world.locations.values():
            for cid in loc.characters:
                if cid not in self.state:
                    name = cid.split(".", 1)[-1].replace("_", " ").title()
                    self.state[cid] = {"name": name, "role": "associate", "alive": True, "available": True}

        self.drama = DramaManager(plan, world=world, log_path=self.config.log_dir / "drama.jsonl")
        self.turn_logs: list[TurnLog] = []
        self.turn_log_path = self.config.log_dir / "turns.jsonl"
        # Truncate the file at game start so replays are clean.
        self.turn_log_path.write_text("", encoding="utf-8")

    # ---------------- rendering ---------------------------------------
    def render_location(self) -> str:
        loc_id = self.state["detective"]["location"]
        loc = self.world.locations.get(loc_id)
        if loc is None:
            return f"[The detective stands in an unmapped place: {loc_id}.]"
        parts = [f"-- {loc.name} --", loc.description]
        if loc.adjacent:
            adj_names = ", ".join(self.world.locations[a].name for a in sorted(loc.adjacent) if a in self.world.locations)
            parts.append(f"Exits: {adj_names}")
        chars_here = [self._char_name(cid) for cid in loc.characters if self._is_available(cid)]
        if chars_here:
            parts.append("You see: " + ", ".join(chars_here))
        evidence_here = [
            self.state.get(eid, {}).get("description", eid)
            for eid in loc.evidence
            if not self.state.get(eid, {}).get("destroyed", False)
            and not self.state.get(eid, {}).get("analyzed", False)
        ]
        if evidence_here:
            parts.append("Nearby: " + ", ".join(evidence_here[:3]))
        return "\n".join(parts)

    def render_map(self) -> str:
        """One-time overview of all locations and their connections."""
        lines = ["=== Location Map ==="]
        for lid in sorted(self.world.locations):
            loc = self.world.locations[lid]
            adj = [self.world.locations[a].name for a in sorted(loc.adjacent) if a in self.world.locations]
            lines.append(f"  {loc.name}")
            if adj:
                lines.append("    → " + ", ".join(adj))
        lines.append("===================")
        return "\n".join(lines)

    def _pending_event_hint(self, raw: str, loc_id: str) -> str | None:
        """Return a hint when the player mentioned a character with pending plan events.
        Returns None if the hint would just repeat what the player already typed."""
        raw_lower = raw.lower()
        for eid in self.drama.remaining:
            ev = self.plan.events.get(eid)
            if not ev:
                continue
            for arg in ev.args:
                if not str(arg).startswith("character."):
                    continue
                char_name = self.state.get(str(arg), {}).get("name", "")
                if not char_name:
                    continue
                if not any(tok in raw_lower for tok in char_name.lower().split() if len(tok) > 2):
                    continue
                verb = "question" if ev.verb not in {"examine", "search", "analyze"} else ev.verb
                surname = char_name.split()[-1]
                cmd = f"{verb} {surname}".lower()
                # Suppress if this is exactly what the player just typed
                if raw_lower.strip().startswith(cmd) or cmd in raw_lower:
                    return None
                if ev.location == loc_id:
                    # Use a special prefix so the JS can render this as a button
                    return f"HINT_CMD:{cmd}"
                loc = self.world.locations.get(ev.location or "")
                loc_name = loc.name if loc else ev.location
                return f"Head to {loc_name} to follow up on {char_name}."
        return None

    def _char_name(self, cid: str) -> str:
        return self.state.get(cid, {}).get("name", cid)

    def _evidence_desc(self, eid: str) -> str:
        """Return a human-readable description for an evidence id.
        Falls back to plan.initial_state so undiscovered items still show
        a proper description rather than the raw id like 'evidence.E003'."""
        desc = self.state.get(eid, {}).get("description", "")
        if not desc:
            desc = self.plan.initial_state.get(eid, {}).get("description", "")
        if not desc:
            desc = eid.replace("evidence.", "").replace("_", " ")
        return desc

    def _is_available(self, cid: str) -> bool:
        return self.state.get(cid, {}).get("alive", True) and self.state.get(cid, {}).get("available", True)

    def _world_summary_for_interpreter(self) -> dict[str, Any]:
        loc_id = self.state["detective"]["location"]
        loc = self.world.locations.get(loc_id)
        adj = sorted(loc.adjacent) if loc else []
        here_chars = [f"{cid} ({self._char_name(cid)})" for cid in (loc.characters if loc else [])]
        here_evidence = [
            eid for eid in (loc.evidence if loc else [])
            if not self.state.get(eid, {}).get("destroyed", False)
        ]
        return {
            "player_location": loc_id,
            "adjacent": adj,
            "here_objects": here_evidence,
            "here_characters": here_chars,
            "evidence_ids": [k for k in self.state if k.startswith("evidence.")],
            "inventory": self.state["detective"].get("inventory", []),
            "knowledge_snippets": self.state["detective"].get("knowledge", []),
        }

    # ---------------- input loop --------------------------------------
    def run(self, get_input=input, echo=print) -> str:
        echo(self.render_location())
        echo("")
        for turn in range(1, self.config.max_turns + 1):
            try:
                raw = get_input(self.config.prompt_label)
            except EOFError:
                echo("\n[input closed]")
                break
            if not raw.strip():
                continue
            if raw.strip().lower() in {"quit", "exit"}:
                echo("(detective signs off)")
                break
            raw = _truncate_input(raw)

            parsed = interpret_action(raw, self._world_summary_for_interpreter())
            classification = self.drama.classify(parsed, self.state)
            effects_applied: list[dict[str, Any]] = []
            accommodation_result: dict[str, Any] | None = None

            if classification["classification"] == "constituent":
                eid = classification["matched_event_id"]
                self.drama.execute_constituent(eid, self.state)
                effects_applied = [ef.to_dict() for ef in self.plan.events[eid].effects]
            elif classification["classification"] == "consistent":
                self.drama.apply_free_effects(parsed, self.state)
                self._apply_movement_if_any(parsed)
                effects_applied = parsed.get("effects", [])
            else:  # exceptional
                # Apply the free effects first so the world reflects what the
                # player actually did, then repair the plan around it.
                self.drama.apply_free_effects(parsed, self.state)
                self._apply_movement_if_any(parsed)
                effects_applied = parsed.get("effects", [])
                accommodation_result = self.drama.accommodate(
                    parsed,
                    classification,
                    self.state,
                    world_locations=list(self.world.locations.keys()),
                    characters=[s for s in self.state if s.startswith("character.")],
                )

            narration = self._narrate(parsed, classification, effects_applied, accommodation_result)

            self._log_turn(turn, raw, parsed, classification, effects_applied, accommodation_result, narration)
            echo(narration)
            echo("")

            if self.drama.goal_satisfied(self.state):
                echo(">>> The case is solved. <<<")
                return "solved"
        return "ended"

    # ---------------- web API (single-turn) --------------------------
    def step(self, raw: str) -> dict[str, Any]:
        """Process one player command; return a structured dict for the web frontend.

        Returns:
            log_entries: list of {text, cls, title, as_html} to append to the log div
            triggered_event_id: plan event id if constituent, else None
            moved_to: new location id if detective moved, else None
            new_knowledge: list of strings to add to the notebook
            new_evidence_flags: {evidence_id: {discovered, analyzed}} patches
            characters_encountered: character ids newly seen
            characters_interviewed: character ids newly spoken to
            classification: "constituent" | "consistent" | "exceptional"
            dm_entry: {kind, summary, detail} from latest drama-manager decision
            game_over: True when goal conditions are satisfied
        """
        raw = raw.strip()
        if not raw:
            return _empty_step()

        raw = _truncate_input(raw)
        prev_loc = self.state["detective"]["location"]
        turn = len(self.turn_logs) + 1

        self.drama.log_turn_start(turn, raw, prev_loc)
        parsed = interpret_action(raw, self._world_summary_for_interpreter())
        classification = self.drama.classify(parsed, self.state)
        effects_applied: list[dict[str, Any]] = []
        accommodation_result: dict[str, Any] | None = None
        triggered_event_id: str | None = None
        tag = classification["classification"]

        if tag == "constituent":
            eid = classification["matched_event_id"]
            triggered_event_id = eid
            self.drama.execute_constituent(eid, self.state)
            effects_applied = [ef.to_dict() for ef in self.plan.events[eid].effects]
        elif tag == "consistent":
            self.drama.apply_free_effects(parsed, self.state)
            self._apply_movement_if_any(parsed)
            effects_applied = parsed.get("effects", [])
        else:  # exceptional
            self.drama.apply_free_effects(parsed, self.state)
            self._apply_movement_if_any(parsed)
            effects_applied = parsed.get("effects", [])
            accommodation_result = self.drama.accommodate(
                parsed, classification, self.state,
                world_locations=list(self.world.locations.keys()),
                characters=[s for s in self.state if s.startswith("character.")],
            )

        narration = self._narrate(parsed, classification, effects_applied, accommodation_result)
        self._log_turn(turn, raw, parsed, classification, effects_applied, accommodation_result, narration)

        new_loc = self.state["detective"]["location"]
        moved_to: str | None = new_loc if new_loc != prev_loc else None

        log_entries: list[dict[str, Any]] = []
        new_knowledge: list[str] = []
        new_evidence_flags: dict[str, dict[str, Any]] = {}
        characters_encountered: list[str] = []
        characters_interviewed: list[str] = []
        _social = {"interview", "consult", "confront", "question", "visit"}

        if tag == "constituent" and triggered_event_id:
            ev = self.plan.events[triggered_event_id]
            log_entries.append({"text": narration, "cls": "narration",
                                 "title": f"— {ev.description} —", "as_html": False})
            log_entries.append({
                "text": f"Your case grows clearer. ({len(self.drama.executed)} plot events explored.)",
                "cls": "system", "title": None, "as_html": False,
            })
            for arg in ev.args:
                arg_s = str(arg)
                if arg_s.startswith("character."):
                    name = self.state.get(arg_s, {}).get(
                        "name", arg_s.split(".", 1)[-1].replace("_", " ").title()
                    )
                    characters_encountered.append(arg_s)
                    if ev.verb in _social:
                        characters_interviewed.append(arg_s)
                        new_knowledge.append(f"Spoke with — {name}")
            for raw_reveal in ev.reveals:
                eid2 = str(raw_reveal)
                if not eid2.startswith("evidence."):
                    eid2 = "evidence." + eid2
                desc = self._evidence_desc(eid2)
                new_knowledge.append(f"Evidence: {desc[:60]}")
                new_evidence_flags[eid2] = {"discovered": True}
            if ev.verb == "analyze":
                for arg in ev.args:
                    arg_s = str(arg)
                    if arg_s.startswith("evidence."):
                        new_evidence_flags.setdefault(arg_s, {})["analyzed"] = True
            loc_id = self.state["detective"]["location"]
            remaining_here = sum(
                1 for eid2 in self.drama.remaining
                if self.plan.events.get(eid2) and self.plan.events[eid2].location == loc_id
            )
            if remaining_here == 0:
                loc = self.world.locations.get(loc_id)
                log_entries.append({
                    "text": f"Nothing more to investigate at {loc.name if loc else loc_id}. "
                            "Try an exit from the left panel.",
                    "cls": "system", "title": None, "as_html": False,
                })
            if not self.drama.remaining:
                log_entries.append({
                    "text": "Every trail you could follow has been followed. "
                            "Your notebook is full. Type 'accuse <suspect>'.",
                    "cls": "narration", "title": "— the case, fully explored —", "as_html": False,
                })

        elif tag == "consistent":
            if moved_to:
                loc = self.world.locations.get(moved_to)
                loc_name = loc.name if loc else moved_to
                log_entries.append({"text": f"You make your way to {loc_name}.",
                                     "cls": "system", "title": None, "as_html": False})
                if loc:
                    log_entries.append({"text": loc.description,
                                         "cls": "outcome", "title": None, "as_html": False})
                    chars = [
                        self.state.get(cid, {}).get("name", cid)
                        for cid in loc.characters
                        if self.state.get(cid, {}).get("alive", True)
                    ]
                    items = [
                        self._evidence_desc(eid)
                        for eid in loc.evidence
                        if not self.state.get(eid, {}).get("destroyed", False)
                    ]
                    sketch = []
                    if chars:
                        sketch.append("Here with you: " + ", ".join(chars) + ".")
                    if items:
                        sketch.append("You notice: " + "; ".join(items) + ".")
                    if sketch:
                        log_entries.append({"text": " ".join(sketch),
                                             "cls": "outcome", "title": None, "as_html": False})
                    characters_encountered = list(loc.characters)
            else:
                log_entries.append({"text": narration, "cls": "outcome",
                                     "title": None, "as_html": False})
                hint = self._pending_event_hint(raw, self.state["detective"]["location"])
                if hint:
                    if hint.startswith("HINT_CMD:"):
                        cmd = hint[len("HINT_CMD:"):]
                        hint_html = (
                            f'<span style="opacity:.75">You might try: </span>'
                            f'<button class="chip inline-hint" data-cmd="{cmd}" '
                            f'style="margin-left:4px"><span class="arrow">&gt;</span>{cmd}</button>'
                        )
                        log_entries.append({"text": hint_html, "cls": "system",
                                             "title": None, "as_html": True})
                    else:
                        log_entries.append({"text": hint, "cls": "system",
                                             "title": None, "as_html": False})

        else:  # exceptional
            log_entries.append({"text": narration, "cls": "exception",
                                 "title": None, "as_html": False})

        # Latest drama-manager log entry for the DM panel.
        # Search backward so the most meaningful entry wins regardless of order.
        dm_entry: dict[str, Any] | None = None
        for entry in reversed(self.drama.log):
            p = entry.payload
            if entry.kind == "accommodation":
                removed = p.get("removed_events", [])
                added   = p.get("replacement_event_ids", [])
                verdict = p.get("goal_reachability", {}).get("verdict", "unknown")
                dm_entry = {
                    "kind": "exceptional",
                    "summary": f"plan modified — removed {len(removed)}, added {len(added)} event(s)",
                    "detail": f"goal: {verdict} | removed: {removed} | added: {added}",
                    "plan_change": f"removed {len(removed)}, added {len(added)} event(s)",
                    "goal_verdict": verdict,
                }
                break
            if entry.kind == "executed_constituent":
                eid  = p.get("event_id", "")
                desc = p.get("event_description", "")
                rem  = p.get("remaining_after", "?")
                dm_entry = {
                    "kind": "constituent",
                    "summary": f"plan event {eid} fired — {desc[:60]}",
                    "detail": f"reveals: {p.get('reveals', [])} | remaining: {rem}",
                    "event_desc": desc,
                    "reveals": p.get("reveals", []),
                    "remaining_after": rem,
                }
                break
            if entry.kind == "classification":
                cls_val = p.get("classification", "consistent")
                matched = p.get("matched_event_id") or "none"
                rem     = p.get("remaining_count", "?")
                active  = len(p.get("active_links", []))
                dm_entry = {
                    "kind": cls_val,
                    "summary": f"{cls_val} — matched={matched} | {rem} events remaining",
                    "detail": f"active causal links: {active} | cs_hint: {p.get('cs_hint', '')}",
                }
                break
        if dm_entry is None and self.drama.log:
            last = self.drama.log[-1]
            dm_entry = {"kind": last.kind, "summary": last.kind, "detail": ""}

        return {
            "log_entries": log_entries,
            "triggered_event_id": triggered_event_id,
            "moved_to": moved_to,
            "new_knowledge": new_knowledge,
            "new_evidence_flags": new_evidence_flags,
            "characters_encountered": characters_encountered,
            "characters_interviewed": characters_interviewed,
            "classification": tag,
            "dm_entry": dm_entry,
            "game_over": self.drama.goal_satisfied(self.state),
        }

    # ---------------- hint-forced execution (Bug-1 fix) ---------------
    def step_force_event(self, event_id: str) -> dict[str, Any]:
        """Execute a plan event directly as constituent, bypassing the action interpreter.
        Called when the player clicks a hint chip — guarantees constituent classification."""
        if event_id not in self.plan.events:
            return {**_empty_step(), "error": f"unknown event {event_id}"}
        if event_id not in self.drama.remaining:
            return {**_empty_step(), "error": f"event {event_id} already executed or removed"}

        ev = self.plan.events[event_id]
        cur_loc = self.state["detective"]["location"]
        if ev.location and ev.location != cur_loc:
            return {**_empty_step(), "error": f"not at required location {ev.location}"}

        turn = len(self.turn_logs) + 1
        short_args = " ".join(
            str(a) for a in ev.args[:2] if not str(a).startswith("location.")
        )
        raw = f"[hint] {ev.verb} {short_args}".strip()
        parsed = {
            "_raw": raw,
            "verb": ev.verb,
            "args": list(ev.args),
            "location": ev.location or cur_loc,
            "effects": [ef.to_dict() for ef in ev.effects],
            "plain_summary": f"{ev.verb} {short_args}".strip(),
            "preconditions": [],
            "reveals": list(ev.reveals),
        }
        classification = {
            "classification": "constituent",
            "matched_event_id": event_id,
            "hard_violations": [],
            "soft_threats": [],
        }

        self.drama.log_turn_start(turn, raw, cur_loc)
        self.drama.execute_constituent(event_id, self.state)
        effects_applied = [ef.to_dict() for ef in ev.effects]

        narration = self._narrate(parsed, classification, effects_applied, None)
        self._log_turn(turn, raw, parsed, classification, effects_applied, None, narration)

        new_knowledge: list[str] = []
        new_evidence_flags: dict[str, dict[str, Any]] = {}
        characters_encountered: list[str] = []
        characters_interviewed: list[str] = []
        log_entries: list[dict[str, Any]] = []
        _social = {"interview", "consult", "confront", "question", "visit"}

        log_entries.append({"text": narration, "cls": "narration",
                             "title": f"— {ev.description} —", "as_html": False})
        log_entries.append({
            "text": f"Your case grows clearer. ({len(self.drama.executed)} plot events explored.)",
            "cls": "system", "title": None, "as_html": False,
        })
        for arg in ev.args:
            arg_s = str(arg)
            if arg_s.startswith("character."):
                name = self.state.get(arg_s, {}).get(
                    "name", arg_s.split(".", 1)[-1].replace("_", " ").title()
                )
                characters_encountered.append(arg_s)
                if ev.verb in _social:
                    characters_interviewed.append(arg_s)
                    new_knowledge.append(f"Spoke with — {name}")
        for raw_reveal in ev.reveals:
            eid2 = str(raw_reveal)
            if not eid2.startswith("evidence."):
                eid2 = "evidence." + eid2
            desc = self._evidence_desc(eid2)
            new_knowledge.append(f"Evidence: {desc[:60]}")
            new_evidence_flags[eid2] = {"discovered": True}
        if ev.verb == "analyze":
            for arg in ev.args:
                arg_s = str(arg)
                if arg_s.startswith("evidence."):
                    new_evidence_flags.setdefault(arg_s, {})["analyzed"] = True

        loc_id = self.state["detective"]["location"]
        remaining_here = sum(
            1 for eid2 in self.drama.remaining
            if self.plan.events.get(eid2) and self.plan.events[eid2].location == loc_id
        )
        if remaining_here == 0:
            loc = self.world.locations.get(loc_id)
            log_entries.append({
                "text": f"Nothing more to investigate at {loc.name if loc else loc_id}. "
                        "Try an exit from the left panel.",
                "cls": "system", "title": None, "as_html": False,
            })
        if not self.drama.remaining:
            log_entries.append({
                "text": "Every trail you could follow has been followed. "
                        "Your notebook is full. Type 'accuse <suspect>'.",
                "cls": "narration", "title": "— the case, fully explored —", "as_html": False,
            })

        dm_entry = {
            "kind": "constituent",
            "summary": f"plan event {event_id} fired — {ev.description[:60]}",
            "detail": f"reveals: {ev.reveals} | remaining: {len(self.drama.remaining)}",
            "event_desc": ev.description,
            "reveals": ev.reveals,
            "remaining_after": len(self.drama.remaining),
        }
        return {
            "log_entries": log_entries,
            "triggered_event_id": event_id,
            "moved_to": None,
            "new_knowledge": new_knowledge,
            "new_evidence_flags": new_evidence_flags,
            "characters_encountered": characters_encountered,
            "characters_interviewed": characters_interviewed,
            "classification": "constituent",
            "dm_entry": dm_entry,
            "game_over": self.drama.goal_satisfied(self.state),
        }

    # ---------------- helpers -----------------------------------------
    _MOVEMENT_VERBS = frozenset({"move", "go", "walk", "travel", "head", "visit", "approach", "leave"})

    def _apply_movement_if_any(self, parsed: dict[str, Any]) -> None:
        # Only move the detective when the player actually used a movement verb.
        # Without this guard, "check X" or "inspect X" could trigger location
        # changes if the action interpreter extracted a target_location.
        verb = (parsed.get("verb") or "").lower()
        if verb not in self._MOVEMENT_VERBS:
            return
        target = parsed.get("target_location") or ""
        if not target:
            return
        cur_loc = self.state["detective"]["location"]
        cur = self.world.locations.get(cur_loc)
        if cur and target in cur.adjacent and target in self.world.locations:
            self.state["detective"]["location"] = target
            Effect("detective", "location", "set", target).apply(self.state)

    def _narration_system(self) -> str:
        """Build a system message that anchors the LLM to the real characters/locations."""
        char_names = []
        for k, v in self.state.items():
            if k.startswith("character."):
                name = v.get("name", k.split(".", 1)[-1].replace("_", " ").title())
                char_names.append(name)
        loc_names = [loc.name for loc in self.world.locations.values()]
        det = self.config.detective_name
        return (
            f"You are the narrator for a 1920s London detective noir mystery. "
            f"The player IS {det}. Always address them as 'you' — NEVER say "
            f"'the detective' or invent any other detective name. "
            f"ONLY use these characters: {', '.join(char_names) or 'none listed'}. "
            f"ONLY use these locations: {', '.join(loc_names[:12]) or 'none listed'}. "
            f"Never invent new people or places. Never reveal the real killer's identity."
        )

    def _next_move_hint(self) -> str | None:
        """One-sentence suggestion about what to try next, for embedding in narration."""
        loc_id = self.state["detective"]["location"]
        for eid in self.drama.remaining:
            ev = self.plan.events.get(eid)
            if not ev:
                continue
            if ev.location == loc_id:
                for arg in ev.args:
                    arg_s = str(arg)
                    if arg_s.startswith("character."):
                        cname = self.state.get(arg_s, {}).get("name", "")
                        if cname:
                            surname = cname.split()[-1]
                            return f"Perhaps questioning {cname} could reveal more."
                    elif not arg_s.startswith("location."):
                        # Strip object./evidence. prefixes so the hint reads naturally
                        pretty = re.sub(r"^(?:object|evidence)\.", "", arg_s).replace("_", " ")
                        verb = ev.verb if ev.verb not in {"interview", "consult"} else "question"
                        return f"You might try to {verb} the {pretty}."
        for eid in self.drama.remaining:
            ev = self.plan.events.get(eid)
            if not ev or not ev.location or ev.location == loc_id:
                continue
            loc = self.world.locations.get(ev.location)
            if loc:
                return f"There are leads worth pursuing at {loc.name}."
        return None

    def _narrate(
        self,
        parsed: dict[str, Any],
        classification: dict[str, Any],
        effects_applied: list[dict[str, Any]],
        accommodation_result: dict[str, Any] | None,
    ) -> str:
        if not self.config.narrate_with_llm:
            return self._stub_narration(parsed, classification)
        tag = classification["classification"]
        raw_cmd = parsed.get("_raw", "")
        summary = parsed.get("plain_summary", raw_cmd)
        loc_id = self.state["detective"]["location"]
        loc = self.world.locations.get(loc_id)
        loc_name = loc.name if loc else loc_id
        det = self.config.detective_name
        system = self._narration_system()

        if tag == "exceptional":
            next_hint = self._next_move_hint()
            hint_context = (
                f" The most useful next step would be: {next_hint}"
                if next_hint else ""
            )
            prompt = (
                f"You are {det} at {loc_name}. You tried: \"{raw_cmd}\" but something stopped you.\n\n"
                "Write 2 sentences of immersive 1920s-noir prose in second person ('you'). "
                "First, describe in one sentence why the action can't happen right now — "
                f"a physical obstacle, your professional judgment, or a practical constraint at {loc_name}. "
                f"Second, in one sentence, hint at one concrete thing you could do instead to move forward.{hint_context} "
                "No invented characters. No new locations. No mechanical labels."
            )
            max_t, temp = 120, 0.55
        elif tag == "constituent":
            eid = classification.get("matched_event_id")
            event_hint = ""
            if eid and eid in self.plan.events:
                event_hint = self.plan.events[eid].narrative[:500]
            next_hint = self._next_move_hint()
            hint_line = (
                f"\nEnd with a subtle one-sentence lead-in to the next step: {next_hint}"
                if next_hint else ""
            )
            prompt = (
                f"You are {det} at {loc_name}. You just: {summary}.\n"
                f"Plot reference (what was found): {event_hint}\n"
                f"World-state effects applied: {json.dumps(effects_applied)[:400]}\n\n"
                "Write 3-4 sentences of 1920s-noir prose describing what you discovered. "
                "Use 'you' to address the player. Stay faithful to the plot reference above. "
                "Do NOT invent new characters or locations."
                + hint_line
            )
            max_t, temp = 250, 0.65
        else:  # consistent — no plan progress
            next_hint = self._next_move_hint()
            hint_line = f" Then: {next_hint}" if next_hint else ""
            prompt = (
                f"You are {det} at {loc_name}. You tried: {summary}.\n"
                "Write ONE sentence in 1920s-noir style acknowledging this led nowhere new. "
                "Use 'you'. No new plot facts. No invented characters."
                + hint_line
            )
            max_t, temp = 100, 0.55

        try:
            return chat_simple(prompt, system=system, max_tokens=max_t, temperature=temp).strip()
        except Exception:
            return self._stub_narration(parsed, classification)

    @staticmethod
    def _stub_narration(parsed: dict[str, Any], classification: dict[str, Any]) -> str:
        return f"[{classification['classification']}] {parsed.get('plain_summary') or parsed.get('_raw', '')}"

    def _log_turn(
        self,
        turn: int,
        raw: str,
        parsed: dict[str, Any],
        classification: dict[str, Any],
        effects_applied: list[dict[str, Any]],
        accommodation_result: dict[str, Any] | None,
        narration: str,
    ) -> None:
        log = TurnLog(
            turn=turn, raw=raw, parsed=parsed, classification=classification,
            effects_applied=effects_applied, accommodation=accommodation_result, narration=narration,
        )
        self.turn_logs.append(log)
        with self.turn_log_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(asdict(log), default=str) + "\n")


_WORD_RE = re.compile(r"\s+")


def _empty_step() -> dict:
    return {
        "log_entries": [], "triggered_event_id": None, "moved_to": None,
        "new_knowledge": [], "new_evidence_flags": {}, "characters_encountered": [],
        "characters_interviewed": [], "classification": "noop", "dm_entry": None,
        "game_over": False,
    }


def _truncate_input(raw: str) -> str:
    tokens = _WORD_RE.split(raw.strip())
    if len(tokens) > MAX_WORDS_PER_COMMAND:
        tokens = tokens[:MAX_WORDS_PER_COMMAND]
    return " ".join(tokens)


In [ ]:
%%writefile main.py
"""Entry point.

Modes:
  build    Run Phase I + story_to_plan + world_builder once; save everything.
  play     Load saved plan+world and launch the interactive game loop.
  replay   Read a scripted list of commands from a file and run them
           against the saved plan+world (for transcripts / regression tests).
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

# Reconfigure stdio to UTF-8 so LLM-generated unicode (em dashes, smart quotes,
# accented characters) never crashes the batch runner on locale-less SLURM shells.
for _stream in (sys.stdout, sys.stderr):
    try:
        _stream.reconfigure(encoding="utf-8", errors="replace")
    except AttributeError:
        pass

from game_engine import EngineConfig, GameEngine
from phase1_story_generator import assemble_story, generate_full_story, load_checkpoint
from story_to_plan import build_plan, load_plan
from world_builder import build_world, load_world, save_world


def cmd_build(args: argparse.Namespace) -> int:
    data_dir = Path(args.data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    if not args.skip_story or not (data_dir / "plot_points.json").exists():
        print("==> Phase I: generating story")
        generate_full_story(user_prompt=args.prompt, out_dir=data_dir, min_points=args.min_points)
    else:
        print("==> Reusing existing Phase I artifacts")

    case_file = load_checkpoint(data_dir / "case_file.json")
    plot_points = load_checkpoint(data_dir / "plot_points.json")

    print("==> Building plan")
    plan = build_plan(case_file, plot_points, out_path=data_dir / "plan.json")

    print("==> Building world")
    world = build_world(plan)
    save_world(world, data_dir / "world.json")
    # Persist plan again so detective.location is set correctly.
    (data_dir / "plan.json").write_text(json.dumps(plan.to_dict(), indent=2), encoding="utf-8")
    print(f"All artifacts saved to {data_dir}/")
    return 0


def cmd_play(args: argparse.Namespace) -> int:
    data_dir = Path(args.data_dir)
    plan = load_plan(data_dir / "plan.json")
    world = load_world(data_dir / "world.json")
    engine = GameEngine(plan, world, EngineConfig(log_dir=Path(args.log_dir)))
    status = engine.run()
    print(f"[game ended: {status}]")
    return 0


def cmd_assemble(args: argparse.Namespace) -> int:
    data_dir = Path(args.data_dir)
    case_file = load_checkpoint(data_dir / "case_file.json")
    plot_points = load_checkpoint(data_dir / "plot_points.json")
    story_bible = load_checkpoint(data_dir / "story_bible.json")
    assemble_story(case_file, plot_points, story_bible, out_path=args.out)
    return 0


def cmd_replay(args: argparse.Namespace) -> int:
    data_dir = Path(args.data_dir)
    plan = load_plan(data_dir / "plan.json")
    world = load_world(data_dir / "world.json")
    cmds = Path(args.script).read_text(encoding="utf-8").splitlines()
    cmds_iter = iter(cmds)

    def scripted_input(prompt: str) -> str:
        try:
            line = next(cmds_iter)
        except StopIteration:
            raise EOFError
        print(prompt + line)
        return line

    engine = GameEngine(plan, world, EngineConfig(log_dir=Path(args.log_dir)))
    status = engine.run(get_input=scripted_input)
    print(f"[replay ended: {status}]")
    return 0


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser()
    subs = parser.add_subparsers(dest="cmd", required=True)

    sp_build = subs.add_parser("build", help="Generate story + plan + world")
    sp_build.add_argument("--prompt", default="A poisoning murder at a prestigious 1920s London art gallery opening")
    sp_build.add_argument("--data-dir", default="data")
    sp_build.add_argument("--min-points", type=int, default=20)
    sp_build.add_argument("--skip-story", action="store_true",
                          help="Skip Phase I if data/plot_points.json already exists.")
    sp_build.set_defaults(func=cmd_build)

    sp_asm = subs.add_parser("assemble", help="Assemble a novel-style markdown from an existing plot_points.json")
    sp_asm.add_argument("--data-dir", default="data")
    sp_asm.add_argument("--out", default="data/final_story.md")
    sp_asm.set_defaults(func=cmd_assemble)

    sp_play = subs.add_parser("play", help="Launch interactive game")
    sp_play.add_argument("--data-dir", default="data")
    sp_play.add_argument("--log-dir", default="logs")
    sp_play.set_defaults(func=cmd_play)

    sp_replay = subs.add_parser("replay", help="Run a scripted transcript")
    sp_replay.add_argument("script")
    sp_replay.add_argument("--data-dir", default="data")
    sp_replay.add_argument("--log-dir", default="logs")
    sp_replay.set_defaults(func=cmd_replay)

    args = parser.parse_args(argv)
    return args.func(args)


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile web/build_game.py
"""Build a free-text interactive detective game as a single HTML file.

This is NOT a story reader. It is a playable text adventure. The user types
commands like `examine body`, `go to forensic lab`, `question gregory`,
`accuse vivienne`. A minimal in-browser game engine (pure JavaScript)
interprets the input against the same plan + world we generated for the
Phase II backend.

The engine intentionally implements a *simplified* version of
drama_manager's three classifications so the game is playable without an
LLM:
  - constituent -> match against a plan event at the current location
  - consistent  -> acknowledged but no plan progress
  - exceptional -> hand-picked destructive verbs are refused with noir
                   flavour text, simulating the real DM's repair step

Usage:
    python web/build_game.py
    # -> writes web/game.html
"""
from __future__ import annotations

import json
import re
from pathlib import Path

HERE = Path(__file__).resolve().parent
ROOT = HERE.parent
OUT = HERE / "game.html"


# ---------------------------------------------------------------------------
# Aliases
# ---------------------------------------------------------------------------
STOPWORDS = {"the", "a", "an", "of", "in", "at", "to", "on", "and", "or", "for",
             "with", "by", "from", "into", "near", "behind"}


def _tokens_from(name: str) -> list[str]:
    return [t.lower() for t in re.split(r"\W+", name) if t and t.lower() not in STOPWORDS and len(t) > 1]


def _id_aliases(entity_id: str, pretty_name: str = "") -> list[str]:
    seen: set[str] = set()
    for part in entity_id.split(".", 1)[-1].split("_"):
        if part and part not in STOPWORDS:
            seen.add(part.lower())
    for tok in _tokens_from(pretty_name):
        seen.add(tok)
    if pretty_name:
        seen.add(pretty_name.lower())
    return sorted(seen)


# ---------------------------------------------------------------------------
# Narrative shortener for the in-game event text
# ---------------------------------------------------------------------------
def _short_narrative(text: str, max_chars: int = 900) -> str:
    text = (text or "").strip()
    if len(text) <= max_chars:
        return text
    # truncate at a sentence boundary
    cut = text[:max_chars]
    last = max(cut.rfind(". "), cut.rfind("! "), cut.rfind("? "))
    if last > 400:
        return cut[: last + 1].strip()
    return cut.rstrip() + "…"


# ---------------------------------------------------------------------------
# Data packing
# ---------------------------------------------------------------------------
def build_game_data() -> dict:
    plan = json.loads((ROOT / "data/plan.json").read_text(encoding="utf-8"))
    world = json.loads((ROOT / "data/world.json").read_text(encoding="utf-8"))
    case = json.loads((ROOT / "data/case_file.json").read_text(encoding="utf-8"))
    story_path = ROOT / "data/final_story.md"
    story_text = story_path.read_text(encoding="utf-8") if story_path.exists() else ""

    events: dict[str, dict] = {}
    for eid, ev in plan["events"].items():
        events[eid] = {
            "id": eid,
            "verb": ev.get("verb", "act"),
            "args": ev.get("args", []),
            "location": ev.get("location", ""),
            "reveals": ev.get("reveals", []),
            "description": ev.get("description", ""),
            "narrative": _short_narrative(ev.get("narrative", "")),
        }

    characters: dict[str, dict] = {}
    for subj, fields in plan["initial_state"].items():
        if subj.startswith("character."):
            name = fields.get("name", subj.split(".", 1)[-1].replace("_", " ").title())
            raw_role = fields.get("role", "")
            # Mask conspirator status — revealing it in the browser spoils the mystery.
            display_role = "associate" if raw_role == "conspirator" else raw_role
            characters[subj] = {
                "id": subj,
                "name": name,
                "role": display_role,
                "alive": fields.get("alive", True),
                "aliases": _id_aliases(subj, name),
            }

    # Ensure the real criminal is present as a selectable suspect even if
    # plan.initial_state didn't surface them as a distinct character.
    crim_name = case.get("criminal", {}).get("name", "")
    crim_id = "character." + re.sub(r"\W+", "_", crim_name.lower()).strip("_") if crim_name else ""
    if crim_id and crim_id not in characters:
        characters[crim_id] = {
            "id": crim_id,
            "name": crim_name,
            "role": "associate",   # neutral, not 'criminal' -- don't spoil
            "alive": True,
            "aliases": _id_aliases(crim_id, crim_name),
        }

    # Phase I sometimes drifts on a character's surname mid-run (e.g.
    # "Vivienne Ashford" in the case file becomes "Vivienne Hartley" in an
    # event's args). Build a first-name -> canonical-id lookup and remap
    # stray references so they land on the right character.
    by_firstname: dict[str, list[str]] = {}
    for cid, ch in characters.items():
        if not ch.get("alive", True):
            continue
        first = re.split(r"\W+", (ch.get("name") or "").lower())[0]
        if first:
            by_firstname.setdefault(first, []).append(cid)
    alias_remap: dict[str, str] = {}
    all_arg_char_ids: set[str] = set()
    for ev in plan["events"].values():
        for a in ev.get("args", []):
            if isinstance(a, str) and a.startswith("character."):
                all_arg_char_ids.add(a)
    for lid, loc in world["locations"].items():
        for c in loc.get("characters", []):
            if isinstance(c, str) and c.startswith("character."):
                all_arg_char_ids.add(c)
    for bad_id in all_arg_char_ids:
        if bad_id in characters:
            continue
        body = bad_id.split(".", 1)[-1]
        first = body.split("_", 1)[0]
        candidates = by_firstname.get(first, [])
        if len(candidates) == 1:
            alias_remap[bad_id] = candidates[0]

    # Apply the remap in place so downstream packing sees only canonical ids.
    if alias_remap:
        for ev in plan["events"].values():
            ev["args"] = [alias_remap.get(a, a) if isinstance(a, str) else a
                          for a in ev.get("args", [])]
        for loc in world["locations"].values():
            loc["characters"] = [alias_remap.get(c, c) for c in loc.get("characters", [])]

    # Build per-character blurbs from case_file. Suspects are red herrings,
    # so their motives + alibis are safe to share. Conspirators' "role" is a
    # plot spoiler (it describes their complicity), so we expose only the
    # claimed alibi. The real criminal gets a neutral 'access/opportunity'
    # line that doesn't reveal their guilt.
    blurbs: dict[str, str] = {}
    for s in case.get("suspects", []):
        cid = "character." + re.sub(r"\W+", "_", s["name"].lower()).strip("_")
        parts = []
        if s.get("motive"):
            parts.append("Apparent motive — " + s["motive"])
        if s.get("alibi"):
            parts.append("Claimed alibi — " + s["alibi"])
        if parts:
            blurbs[cid] = "  ".join(parts)
    for s in case.get("conspirators", []):
        cid = "character." + re.sub(r"\W+", "_", s["name"].lower()).strip("_")
        if s.get("alibi"):
            blurbs[cid] = "Claimed alibi — " + s["alibi"]
    if crim_id:
        crim = case.get("criminal", {})
        opportunity = crim.get("opportunity", "")
        blurbs[crim_id] = ("Access — " + opportunity) if opportunity else "A close associate of the victim."

    for cid, ch in characters.items():
        ch["blurb"] = blurbs.get(cid, "")

    evidence_entities: dict[str, dict] = {}
    for subj, fields in plan["initial_state"].items():
        if subj.startswith("evidence."):
            desc = fields.get("description", subj)
            evidence_entities[subj] = {
                "id": subj,
                "description": desc,
                "aliases": _id_aliases(subj, desc),
            }

    locations: dict[str, dict] = {}
    for lid, loc in world["locations"].items():
        locations[lid] = {
            "id": lid,
            "name": loc["name"],
            "description": loc["description"],
            "adjacent": list(loc["adjacent"]),
            "characters": list(loc["characters"]),
            "evidence": list(loc["evidence"]),
            "aliases": _id_aliases(lid, loc["name"]),
        }

    # Ensure dead characters (victims) appear at the starting location so
    # the "Currently in room" panel shows the body from turn 1.
    start_loc = world.get("starting_location", "")
    if start_loc and start_loc in locations:
        for subj, fields in plan["initial_state"].items():
            if subj.startswith("character.") and not fields.get("alive", True):
                if subj not in locations[start_loc]["characters"]:
                    locations[start_loc]["characters"].insert(0, subj)

    # De-duplicate evidence across locations: each evidence item belongs to the
    # first location whose plan event reveals it. Items with no reveal event stay
    # in whichever location world_builder placed them.
    ev_primary_loc: dict[str, str] = {}
    for ev in plan["events"].values():
        ev_loc = ev.get("location", "")
        for rev in ev.get("reveals", []):
            rev_id = str(rev)
            if not rev_id.startswith("evidence."):
                rev_id = "evidence." + rev_id
            if rev_id not in ev_primary_loc:
                ev_primary_loc[rev_id] = ev_loc
    for lid, loc_data in locations.items():
        loc_data["evidence"] = [
            eid for eid in loc_data["evidence"]
            if ev_primary_loc.get(eid, lid) == lid
        ]

    # The real-criminal id used for the accusation win condition
    crim_name = case.get("criminal", {}).get("name", "")
    crim_id = "character." + re.sub(r"\W+", "_", crim_name.lower()).strip("_")

    # Build a predecessor map: {event_id: [required_event_ids]} from plan order
    predecessors: dict[str, list[str]] = {eid: [] for eid in plan["events"]}
    for producer, consumer in plan.get("order", []):
        if consumer in predecessors:
            predecessors[consumer].append(producer)

    return {
        "starting_location": world["starting_location"],
        "real_criminal_id": crim_id,
        "real_criminal_name": crim_name,
        "victim_name": case.get("victim", {}).get("name", ""),
        "detective_name": case.get("detective", {}).get("name", ""),
        "events": events,
        "characters": characters,
        "evidence": evidence_entities,
        "locations": locations,
        "goal_events_needed": 10,
        "story_text": story_text,
        "predecessors": predecessors,
    }


# ---------------------------------------------------------------------------
# HTML + JS
# ---------------------------------------------------------------------------
HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Play: The Hartley Affair</title>
<style>
:root {
  --bg: #14100d;
  --panel: #1d1812;
  --panel2: #251f17;
  --paper: #f4ecdc;
  --ink: #1c1a17;
  --accent: #b38a4a;
  --accent-soft: #c9a878;
  --good: #4a9b5d;
  --warn: #c7883c;
  --danger: #c26060;
  --muted: #8a7b66;
  --rule: #2c251e;
}
* { box-sizing: border-box; }
html, body {
  margin: 0; background: var(--bg); color: var(--paper);
  font-family: "Iowan Old Style", "Palatino Linotype", Palatino, Georgia, serif;
  font-size: 16px; line-height: 1.55;
  height: 100vh;
  overflow: hidden;       /* lock the page; inner regions scroll themselves */
}
body { display: flex; flex-direction: column; }
header.masthead {
  padding: 18px 28px 14px;
  border-bottom: 1px solid var(--rule);
  background: linear-gradient(180deg, #1a140f 0%, var(--bg) 100%);
  display: flex; justify-content: space-between; align-items: baseline;
  flex-wrap: wrap; gap: 10px;
}
header.masthead h1 {
  margin: 0;
  font-family: "Playfair Display", "Didot", Georgia, serif;
  font-size: 26px;
  color: var(--accent);
  letter-spacing: 0.02em;
}
header.masthead .subtitle {
  margin: 0;
  color: var(--muted);
  font-style: italic;
  font-size: 13px;
  letter-spacing: 0.06em;
  text-transform: uppercase;
}
header.masthead a { color: var(--accent-soft); }

.layout {
  display: grid;
  grid-template-columns: 280px 1fr 280px;
  flex: 1 1 auto; min-height: 0;   /* let the grid fill remaining viewport */
  overflow: hidden;                 /* children handle their own scroll */
}
aside {
  background: var(--panel);
  padding: 18px 18px 24px;
  overflow-y: auto;
  min-height: 0;
  border-color: var(--rule);
}
aside.left  { border-right: 1px solid var(--rule); }
aside.right {
  border-left: 1px solid var(--rule);
  overflow: hidden;   /* right column is locked; each section scrolls internally */
}
aside h3 {
  font-family: "Playfair Display", Georgia, serif;
  font-size: 13px;
  letter-spacing: 0.12em;
  text-transform: uppercase;
  color: var(--accent-soft);
  margin: 0 0 8px;
}
aside h3:not(:first-child) { margin-top: 18px; }
aside ul { list-style: none; margin: 0; padding: 0; font-size: 13.5px; }
aside li { padding: 3px 0; color: var(--paper); white-space: normal; word-break: break-word; line-height: 1.4; }
aside .muted { color: var(--muted); font-style: italic; font-size: 13px; }
aside .exit-btn {
  background: transparent;
  color: var(--accent-soft);
  border: 1px dashed var(--rule);
  padding: 4px 8px;
  margin: 2px 0;
  cursor: pointer;
  font-family: inherit;
  font-size: 13px;
  display: block;
  width: 100%;
  text-align: left;
  border-radius: 2px;
}
aside .exit-btn:hover { background: rgba(179,138,74,0.12); border-style: solid; }

.center {
  display: flex; flex-direction: column;
  min-height: 0;
  background: var(--bg);
  overflow: hidden;
}
aside.left .scene-name {
  font-family: "Playfair Display", "Didot", Georgia, serif;
  font-size: 17px;
  color: var(--accent);
  letter-spacing: 0.02em;
  margin: 2px 0 4px;
  line-height: 1.3;
}
aside.left .scene-desc {
  font-size: 12.5px;
  color: var(--accent-soft);
  font-style: italic;
  line-height: 1.45;
  margin-bottom: 2px;
}
.log {
  flex: 1 1 auto;
  min-height: 0;
  overflow-y: auto;
  padding: 22px 34px 20px;
  scroll-behavior: smooth;
}
.log .entry {
  margin-bottom: 16px;
  max-width: 780px;
}
.log .entry.system {
  color: var(--muted);
  font-style: italic;
  font-size: 13px;
  border-left: 2px solid var(--rule);
  padding-left: 10px;
}
.log .entry.user {
  color: var(--accent-soft);
  font-family: "Courier Prime", "Courier New", monospace;
  font-size: 14px;
  font-style: italic;
}
.log .entry.user::before {
  content: "> ";
  color: var(--accent);
  font-style: normal;
}
.log .entry.narration {
  background: #1a150f;
  padding: 14px 18px;
  border-left: 3px solid var(--accent);
  border-radius: 0 3px 3px 0;
  white-space: pre-wrap;
}
.log .entry.narration h4 {
  margin: 0 0 8px;
  font-family: "Playfair Display", Georgia, serif;
  font-size: 15px;
  color: var(--accent);
  letter-spacing: 0.04em;
}
.log .entry strong {
  color: var(--accent-soft);
  font-weight: 600;
}
.log .entry.victory strong { color: #cfe8c8; }
.log .entry.outcome {
  color: var(--warn);
  font-size: 14px;
}
.log .entry.exception {
  color: var(--danger);
  font-size: 14px;
  border-left: 3px solid var(--danger);
  padding-left: 10px;
  background: rgba(194,96,96,0.06);
}
.log .entry.victory {
  color: var(--good);
  background: rgba(74,155,93,0.1);
  border: 1px solid var(--good);
  padding: 14px 18px;
  border-radius: 3px;
  font-weight: 600;
}

.hidden { display: none !important; }

.hints-panel {
  border-top: 1px solid var(--rule);
  background: var(--panel2);
  padding: 10px 20px 0;
}
.hints-toggle {
  background: transparent;
  color: var(--muted);
  border: 1px dashed var(--rule);
  padding: 6px 12px;
  font-family: inherit;
  font-size: 12px;
  letter-spacing: 0.04em;
  cursor: pointer;
  border-radius: 2px;
}
.hints-toggle:hover {
  color: var(--accent-soft);
  border-color: var(--accent-soft);
}
.hints-toggle.open {
  color: var(--accent);
  border-color: var(--accent);
  border-style: solid;
}
.hints-list {
  display: flex; flex-wrap: wrap; gap: 6px;
  margin: 8px 0 2px;
}
.hints-list .chip {
  background: var(--panel);
  color: var(--paper);
  border: 1px solid var(--rule);
  padding: 6px 12px;
  font-family: "Courier Prime", "Courier New", monospace;
  font-size: 13px;
  cursor: pointer;
  border-radius: 2px;
  transition: border-color .12s, background .12s;
}
.hints-list .chip:hover {
  border-color: var(--accent);
  background: #241d16;
}
.hints-list .chip:active { background: #2c2219; }
.hints-list .chip-label {
  font-family: "Courier Prime", "Courier New", monospace;
  font-size: 13px;
  color: var(--accent-soft);
}
.hints-list .chip .arrow { color: var(--muted); margin-right: 6px; }
.hints-note {
  font-size: 11.5px;
  color: var(--muted);
  margin: 6px 0 2px;
  font-style: italic;
}

.dm-log {
  max-height: 26vh;
  overflow-y: auto;
  padding: 4px 6px 6px;
  margin: 8px -4px 0;
  background: #161210;
  border: 1px solid var(--rule);
  border-radius: 2px;
  font-family: "Courier Prime", "Courier New", monospace;
  font-size: 12px;
  scrollbar-width: thin;
  scrollbar-color: var(--rule) transparent;
}
.dm-log::-webkit-scrollbar { width: 6px; }
.dm-log::-webkit-scrollbar-thumb { background: var(--rule); border-radius: 3px; }
.dm-log::-webkit-scrollbar-thumb:hover { background: var(--accent); }
.dm-log .entry {
  padding: 3px 6px;
  margin: 1px 0;
  border-left: 2px solid var(--rule);
  color: var(--paper);
  white-space: normal;
  word-break: break-word;
}
.dm-log .entry.constituent { border-left-color: var(--good);  color: #b8e0b8; }
.dm-log .entry.consistent  { border-left-color: var(--muted); color: var(--muted); }
.dm-log .entry.exceptional { border-left-color: var(--danger); color: #e4a8a8; }
.dm-log .entry.move        { border-left-color: var(--accent-soft); color: var(--muted); }
.dm-log .entry .tag {
  display: inline-block;
  min-width: 84px;
  color: var(--muted);
  font-size: 11px;
  letter-spacing: 0.04em;
}

.input-bar {
  display: flex; gap: 8px;
  padding: 12px 20px 16px;
  border-top: 1px solid var(--rule);
  background: var(--panel2);
}
.input-bar input {
  flex: 1;
  padding: 10px 14px;
  border: 1px solid var(--rule);
  background: var(--bg);
  color: var(--paper);
  font-family: "Courier Prime", "Courier New", monospace;
  font-size: 15px;
  border-radius: 3px;
}
.input-bar input:focus { outline: 1px solid var(--accent); }
.input-bar button {
  padding: 10px 18px;
  background: var(--accent);
  color: var(--ink);
  border: none;
  font-family: inherit;
  font-size: 14px;
  font-weight: 600;
  cursor: pointer;
  letter-spacing: 0.05em;
  border-radius: 3px;
}
.input-bar button:disabled { opacity: 0.4; cursor: not-allowed; }
.input-bar .hint {
  padding: 0 20px 12px;
  font-size: 11.5px;
  color: var(--muted);
  letter-spacing: 0.04em;
}

.suspect-locked {
  color: var(--muted); font-style: italic; font-size: 13px; padding: 3px 0;
}
.suspect-row { padding: 2px 0; }
.suspect-row .suspect-name {
  background: transparent;
  color: var(--paper);
  border: none;
  padding: 2px 0;
  cursor: pointer;
  font-family: inherit;
  font-size: 13.5px;
  text-align: left;
  display: block;
  width: 100%;
}
.suspect-row .suspect-name:hover { color: var(--accent-soft); }
.suspect-row .suspect-name .chev {
  display: inline-block; width: 14px; color: var(--accent); font-size: 11px;
}
.suspect-row .blurb {
  margin: 4px 0 8px 14px;
  padding: 8px 10px;
  font-size: 12.5px;
  font-style: italic;
  color: var(--paper);
  border-left: 2px solid var(--accent);
  background: rgba(179,138,74,0.06);
  line-height: 1.5;
}

.scroll-list {
  overflow-y: auto;
  padding-right: 4px;
  margin-right: -4px;
  scrollbar-width: thin;
  scrollbar-color: var(--rule) transparent;
}
.scroll-list::-webkit-scrollbar { width: 6px; }
.scroll-list::-webkit-scrollbar-track { background: transparent; }
.scroll-list::-webkit-scrollbar-thumb { background: var(--rule); border-radius: 3px; }
.scroll-list::-webkit-scrollbar-thumb:hover { background: var(--accent); }

.knowledge { max-height: 26vh; }
#suspects  { max-height: 30vh; }

.knowledge li { position: relative; padding-left: 14px; }
.knowledge li::before { content: "●"; position: absolute; left: 0; color: var(--accent); font-size: 12px; top: 4px; }
.progress-bar {
  height: 4px; background: var(--rule); border-radius: 2px; overflow: hidden;
  margin-top: 8px;
}
.progress-bar > div { height: 100%; background: var(--accent); transition: width .3s; }

.footer-links {
  padding: 14px 28px; border-top: 1px solid var(--rule);
  background: var(--panel);
  text-align: center; font-size: 12px; color: var(--muted);
}
.footer-links a { color: var(--accent-soft); text-decoration: none; margin: 0 8px; }

@media (max-width: 980px) {
  .layout { grid-template-columns: 1fr; }
  aside.left, aside.right { border: none; border-bottom: 1px solid var(--rule); max-height: 200px; }
  .log { padding: 16px 14px; }
}
</style>
</head>
<body>
<header class="masthead">
  <div>
    <h1>The Hartley Affair &mdash; Play</h1>
    <p class="subtitle">You are Inspector Rothwell. Find the killer.</p>
  </div>
  <div style="font-size:12px; color: var(--muted);">Template 2 &mdash; CS 7634</div>
</header>

<div class="layout">
  <aside class="left">
    <h3>Current scene</h3>
    <div id="location-name" class="scene-name"></div>
    <div id="location-desc" class="scene-desc"></div>
    <h3>Exits</h3>
    <div id="exits"></div>
    <h3>Currently in room</h3>
    <ul id="characters-here"></ul>
    <h3>Notable here</h3>
    <ul id="evidence-here"></ul>
  </aside>

  <div class="center">
    <div id="log" class="log"></div>
    <div class="hints-panel">
      <button id="hints-toggle" class="hints-toggle" type="button">💡 Stuck? Show hints</button>
      <button id="guide-toggle" class="hints-toggle" type="button" style="margin-left:8px;">🗺️ Where next?</button>
      <button id="dmlog-toggle" class="hints-toggle" type="button" style="margin-left:8px;">🎬 Drama Manager log</button>
      <label id="dm-verbose-label" style="display:none; margin-left:10px; font-size:12px; color:var(--muted); cursor:pointer; user-select:none;">
        <input type="checkbox" id="dm-verbose-chk"> verbose
      </label>
      <div id="hints-content" class="hidden">
        <div id="hints-list" class="hints-list"></div>
        <div class="hints-note">Click a hint to pre-fill the input — you still press Enter (and can edit) to submit.</div>
      </div>
      <div id="guide-content" class="hidden">
        <div id="guide-body" class="hints-note" style="color: var(--paper); padding: 6px 0 2px;"></div>
        <div id="guide-list" class="hints-list"></div>
      </div>
      <div id="dmlog-content" class="hidden">
        <div id="dmlog-list" class="dm-log"></div>
        <div class="hints-note">Per-turn drama-manager decisions. Green = constituent (plan event fired). Grey = consistent (no plan effect). Red = exceptional (intervention).</div>
      </div>
    </div>
    <form class="input-bar" id="input-form">
      <input id="cmd" autocomplete="off" spellcheck="false"
             placeholder="Type a command (≤ 8 words). Try: examine body, go to london streets, question sutherland, accuse vivienne"
             maxlength="100" autofocus>
      <button type="submit">Enter</button>
      <button type="button" id="reset-btn" title="Start over" style="background: var(--panel); color: var(--paper);">Reset</button>
    </form>
  </div>

  <aside class="right">
    <h3>Detective's notebook</h3>
    <ul id="knowledge" class="knowledge scroll-list"></ul>
    <h3>Suspects</h3>
    <ul id="suspects" class="scroll-list"></ul>
    <h3>Progress</h3>
    <div style="font-size:12px; display:flex; justify-content:space-between; color:var(--accent-soft); margin-bottom:4px;">
      <span><span id="events-triggered">0</span> / <span id="events-needed">0</span> leads pursued</span>
      <span id="pbar-pct">0%</span>
    </div>
    <div class="progress-bar"><div id="pbar" style="width:0%"></div></div>
  </aside>
</div>

<footer class="footer-links">
  <button id="novel-btn" type="button" style="background:none;border:none;padding:0;cursor:pointer;color:var(--accent-soft);font:inherit;">📖 Read story</button>
  &middot; <a href="https://github.com/ivy3h/7634-s26">Source</a>
  &middot; Template 2: Intervention &amp; Accommodation &middot; CS 7634
</footer>

<div id="novel-overlay" style="display:none;position:fixed;inset:0;background:rgba(0,0,0,.88);z-index:1000;overflow-y:auto;padding:48px 24px;">
  <button id="novel-close" style="position:fixed;top:16px;right:22px;background:none;border:none;color:var(--paper);font-size:28px;cursor:pointer;line-height:1;">✕</button>
  <div id="novel-content" style="max-width:800px;margin:0 auto;color:var(--paper);font-family:Georgia,serif;line-height:1.85;font-size:17px;"></div>
</div>

<script>
const DATA = __GAME_DATA__;
const STORAGE_KEY = "hartley_affair_game_v1";
// null = offline/JS mode. Set to "" or a full URL to route through the LLM server.
let API_URL = null;
// When false, the DM log shows only constituent/exceptional one-liners.
// When true, every turn is shown with full classification detail.
let dmVerbose = false;
let _hintEventId = null;  // force-event id stored when a hint chip is clicked
let _hintCmd = null;      // cmd text stored alongside, cleared if user edits

// -------------------- state --------------------
let state;
function freshState() {
  return {
    location: DATA.starting_location,
    lastLocation: null,          // so hints don't suggest retreating
    knowledge: [],
    inventory: [],
    executedEvents: [],
    evidenceFlags: {}, // id -> {discovered, analyzed, destroyed}
    charactersInterviewed: [],
    encounteredCharacters: [],   // seen in a visited location or referenced by an executed event
    dmLog: [],                    // behind-the-scenes drama-manager decision trail
    turns: 0,
    gameOver: false,
  };
}

function dmLog(kind, summary, detail) {
  state.dmLog.push({
    turn: state.turns,
    kind: kind,
    summary: summary,
    detail: detail || "",
    t: Date.now(),
  });
  if (state.dmLog.length > 200) state.dmLog.shift();  // cap memory
}

function renderDmLog() {
  const list = document.getElementById("dmlog-list");
  if (!list) return;
  list.innerHTML = "";

  const entries = dmVerbose
    ? [...state.dmLog].reverse()
    : [...state.dmLog].reverse().filter(e => e.kind === "constituent" || e.kind === "exceptional");

  if (!entries.length) {
    const empty = document.createElement("div");
    empty.className = "hints-note";
    empty.style.padding = "12px 10px";
    empty.textContent = dmVerbose
      ? "No turns logged yet. Start playing to see drama-manager decisions."
      : "No significant events yet. Constituent (plan events fired) and exceptional (plan modified) entries will appear here.";
    list.appendChild(empty);
    return;
  }

  entries.forEach(e => {
    const div = document.createElement("div");
    div.className = "entry " + e.kind;
    const tLabel = `T${String(e.turn).padStart(2,"0")}`;

    if (dmVerbose) {
      // Full detail: tag + summary + detail
      const tag = `[${tLabel} · ${e.kind}]`;
      div.title = e.summary + (e.detail ? " — " + e.detail : "");
      div.innerHTML = '<span class="tag">' + tag + '</span> ' + e.summary
        + (e.detail ? ' — <em>' + e.detail + '</em>' : '');
    } else {
      // Minimal: only constituent and exceptional, condensed one-liner
      if (e.kind === "constituent") {
        const desc = e.event_desc ? e.event_desc.slice(0, 60) : e.summary;
        div.innerHTML = '<span class="tag">[' + tLabel + ' ✓]</span> ' + desc;
        div.title = e.detail || e.summary;
      } else {
        // exceptional
        let line = e.plan_change || e.summary;
        if (e.goal_verdict) line += ' — goal: ' + e.goal_verdict;
        div.innerHTML = '<span class="tag">[' + tLabel + ' ⚠]</span> ' + line;
        div.title = e.detail || "";
      }
    }
    list.appendChild(div);
  });
}
function encounter(cid) {
  if (!cid) return;
  if (!state.encounteredCharacters.includes(cid)) {
    state.encounteredCharacters.push(cid);
  }
}
function encounterAll(ids) { (ids || []).forEach(encounter); }
function loadState() {
  if (API_URL !== null) return freshState();
  try {
    const s = JSON.parse(localStorage.getItem(STORAGE_KEY));
    if (s && s.location) return s;
  } catch (e) {}
  return freshState();
}
function saveState() { localStorage.setItem(STORAGE_KEY, JSON.stringify(state)); }

state = loadState();

// -------------------- rendering --------------------
const logEl = document.getElementById("log");
// Delegated handler for inline hint chips rendered inside log entries
logEl.addEventListener("click", (e) => {
  const btn = e.target.closest(".inline-hint");
  if (!btn) return;
  const cmd = btn.dataset.cmd;
  if (cmd) { prefillInput(cmd); _hintEventId = null; _hintCmd = null; }
});
function addLog(text, cls = "outcome", title = null, asHtml = false) {
  const div = document.createElement("div");
  div.className = "entry " + cls;
  if (title) {
    const h = document.createElement("h4");
    h.textContent = title;
    div.appendChild(h);
  }
  const body = document.createElement("div");
  if (asHtml) body.innerHTML = text;
  else        body.textContent = text;
  div.appendChild(body);
  logEl.appendChild(div);
  logEl.scrollTop = logEl.scrollHeight;
}

// Highlight character names + location names inside a block of narrative
// prose. Returns HTML-safe text with <strong> around each match.
function _escapeHtml(s) {
  return String(s)
    .replace(/&/g, '&amp;')
    .replace(/</g, '&lt;')
    .replace(/>/g, '&gt;')
    .replace(/"/g, '&quot;');
}
let _HIGHLIGHT_TERMS_CACHE = null;
function _buildHighlightTerms() {
  const terms = new Set();
  const STOP = new Set(["name","the","your","this","that","lady","lord",
                        "inspector","detective","sir","madam","mrs","mr","ms","dr"]);
  Object.values(DATA.characters).forEach(c => {
    if (!c.name) return;
    terms.add(c.name);                          // full name
    const parts = c.name.split(/\s+/);
    parts.forEach(p => {
      const clean = p.replace(/[^\w']+$/, "").replace(/^[^\w]+/, "");
      if (clean.length >= 4 && !STOP.has(clean.toLowerCase())) terms.add(clean);
    });
  });
  Object.values(DATA.locations || {}).forEach(l => {
    if (l.name && l.name.length > 3) terms.add(l.name);
  });
  return [...terms].sort((a, b) => b.length - a.length);
}
function highlightText(text) {
  if (!_HIGHLIGHT_TERMS_CACHE) _HIGHLIGHT_TERMS_CACHE = _buildHighlightTerms();
  let out = _escapeHtml(text);
  const buckets = [];   // collected matches; placeholders inserted inline
  _HIGHLIGHT_TERMS_CACHE.forEach(term => {
    const pat = term.replace(/[.*+?^${}()|[\]\\]/g, '\\$&');
    const re = new RegExp('\\b(' + pat + ')\\b', 'g');
    out = out.replace(re, (m) => {
      buckets.push(m);
      return '\u0000' + (buckets.length - 1) + '\u0000';
    });
  });
  return out.replace(/\u0000(\d+)\u0000/g,
    (_, i) => '<strong>' + buckets[+i] + '</strong>');
}

// -------------------- hint engine (non-prescriptive) --------------------
// The player always types commands freely in the input box. The hint
// panel is opt-in: it lists 3-5 commands the Inspector *could* try next,
// given the current state. Clicking a hint only pre-fills the input --
// it never auto-submits. This preserves open-ended input per the spec
// ("interactions should not be from a menu of options").
function firstAlias(entity) {
  if (!entity || !entity.aliases || !entity.aliases.length) return "";
  const clean = entity.aliases.filter(a => !a.includes(".") && a.length > 2);
  clean.sort((a, b) => b.length - a.length);
  return clean[0] || entity.aliases[0];
}
function locAlias(loc) { return loc ? loc.name.toLowerCase() : ""; }
function truncLabel(s, n = 34) { return s.length > n ? s.slice(0, n - 1) + "…" : s; }

// Returns true if a plan-event arg string looks like a location name
// (so we don't include it in hint chip text).
function _argIsLocationLike(arg) {
  const s = String(arg).toLowerCase().trim();
  if (s.startsWith("location.")) return true;
  return Object.values(DATA.locations).some(loc => {
    const n = (loc.name || "").toLowerCase();
    if (n.length > 3 && (s === n || s.includes(n) || n.includes(s))) return true;
    return (loc.aliases || []).some(a => a.length > 3 && (s === a || s.includes(a)));
  });
}

function buildHintCommands() {
  const hints = [];
  const loc = currentLoc();

  // A. Remaining plan events at this location whose predecessors are all done.
  if (loc) {
    const localEvents = Object.values(DATA.events).filter(ev => {
      if (ev.location !== state.location || state.executedEvents.includes(ev.id)) return false;
      const preds = DATA.predecessors[ev.id] || [];
      return preds.every(p => state.executedEvents.includes(p));
    });
    // Count how many local events share the same first non-location arg —
    // if > 1, we need to include a distinguishing second arg in the chip
    // so the command scores onto a single event instead of ping-ponging
    // between them.
    const firstArgOf = ev => (ev.args || []).find(a => !String(a).startsWith("location."));
    const firstArgCount = {};
    localEvents.forEach(ev => {
      const fa = firstArgOf(ev);
      if (fa) firstArgCount[fa] = (firstArgCount[fa] || 0) + 1;
    });

    const prettyArg = raw => {
      const s = String(raw);
      if (s.startsWith("character.")) {
        const c = DATA.characters[s];
        return c ? firstAlias(c) : s.split(".", 2)[1].replace(/_/g, " ");
      }
      if (s.startsWith("evidence.")) {
        const e = DATA.evidence[s];
        return e ? firstAlias(e) : s.split(".", 2)[1];
      }
      if (s.startsWith("object.")) {
        return s.split(".", 2)[1].replace(/_/g, " ");
      }
      return s.replace(/_/g, " ").trim().toLowerCase()
              .replace(/[^\w\s]/g, "").split(/\s+/).slice(0, 4).join(" ");
    };

    for (const ev of localEvents) {
      const firstArg = firstArgOf(ev);
      let argIsCharacter = String(firstArg || "").startsWith("character.");
      let tgt = prettyArg(firstArg);

      // Always try to append one more distinctive arg so each hint chip
      // stays stable even after a sibling event finishes (prevents the
      // "…records" chip from visibly shortening mid-playthrough).
      if (firstArg) {
        const othersArgs = new Set();
        localEvents.forEach(ev2 => {
          if (ev2.id === ev.id) return;
          (ev2.args || []).forEach(a => othersArgs.add(String(a)));
        });
        const unique = (ev.args || []).find(a =>
          !String(a).startsWith("location.") && a !== firstArg && !othersArgs.has(a)
        );
        const anySecond = (ev.args || []).find(a =>
          !String(a).startsWith("location.") && a !== firstArg
        );
        const pick = unique || anySecond;
        if (pick && !_argIsLocationLike(String(pick))) tgt = tgt + " " + prettyArg(pick);
      }

      // Normalize a few plan verbs so the chip routes to a working handler.
      let verbNorm = ev.verb;
      if (verbNorm === "investigate") verbNorm = "search";
      if (verbNorm === "visit" && argIsCharacter) verbNorm = "question";
      // Hard cap: keep chips at most 6 words so they fit cleanly in the
      // panel and still fit under the parser's 8-word command ceiling.
      const finalHint = `${verbNorm} ${tgt}`.trim().split(/\s+/).slice(0, 6).join(" ");
      hints.push({cmd: finalHint, eventId: ev.id});
      if (hints.length >= 3) break;
    }
  }

  // B. Characters still here that haven't been interviewed AND still have
  //    a remaining social event tied to them at this location. Avoid
  //    suggesting "dead" hints that would just print the generic polite
  //    fall-through.
  if (loc) {
    const socialFamily = new Set(["interview","question","consult","confront","visit"]);
    for (const cid of loc.characters) {
      const c = DATA.characters[cid];
      if (!c || !c.alive) continue;
      if (state.charactersInterviewed.includes(cid)) continue;
      const alias = firstAlias(c);
      const already = hints.some(h => h.cmd.toLowerCase().includes(alias));
      if (already || hints.length >= 4) continue;
      const pendingSocialEv = Object.values(DATA.events).find(ev => {
        if (ev.location !== state.location || state.executedEvents.includes(ev.id)) return false;
        if (!socialFamily.has(ev.verb)) return false;
        if (!(ev.args || []).some(a => String(a).toLowerCase().includes(alias))) return false;
        const preds = DATA.predecessors[ev.id] || [];
        return preds.every(p => state.executedEvents.includes(p));
      });
      if (pendingSocialEv) {
        hints.push({cmd: "question " + alias, eventId: pendingSocialEv.id});
      }
    }
  }

  // C. Exit suggestions, depth-aware.
  //    - 'immediate' = pending events at that neighbor
  //    - 'within'    = pending events reachable within 3 more hops
  //    Rank by immediate * 10 + within. Exclude the location we came from.
  function eventsWaitingAt(locId) {
    return Object.values(DATA.events).filter(
      ev => ev.location === locId && !state.executedEvents.includes(ev.id)
    ).length;
  }
  function pendingReachable(fromId, maxHops) {
    maxHops = maxHops == null ? 4 : maxHops;
    const seen = new Set([fromId]);
    const queue = [[fromId, 0]];
    let count = 0;
    let firstHit = null;
    while (queue.length) {
      const [cur, d] = queue.shift();
      if (d > 0) {
        const hasPending = Object.values(DATA.events).some(
          ev => ev.location === cur && !state.executedEvents.includes(ev.id)
        );
        if (hasPending) {
          count++;
          if (firstHit === null) firstHit = cur;
        }
      }
      if (d >= maxHops) continue;
      const loc2 = DATA.locations[cur];
      if (!loc2) continue;
      for (const adj of loc2.adjacent) {
        if (!seen.has(adj)) {
          seen.add(adj);
          queue.push([adj, d + 1]);
        }
      }
    }
    return { count: count, firstHit: firstHit };
  }
  if (loc && loc.adjacent.length && hints.length < 4) {
    const ranked = loc.adjacent
      .filter(id => id !== state.lastLocation)
      .map(id => ({
        id: id,
        immediate: eventsWaitingAt(id),
        within: pendingReachable(id, 3).count,
        loc: DATA.locations[id],
      }))
      .filter(x => x.loc && (x.immediate > 0 || x.within > 0))
      .sort((a, b) => (b.immediate * 10 + b.within) - (a.immediate * 10 + a.within));
    for (const x of ranked.slice(0, 2)) {
      if (hints.length >= 4) break;
      hints.push({cmd: "go to " + locAlias(x.loc), eventId: null});
    }
  }

  // D. Long-range nudge. If the current location has nothing left and the
  //    immediate exits are dry too, BFS out to the nearest pending event
  //    and surface a direct "go to <that place>" hint so the player knows
  //    where the remaining work actually lives.
  if (hints.length < 2) {
    const reach = pendingReachable(state.location, 8);
    if (reach.firstHit && reach.firstHit !== state.location) {
      const targetLoc = DATA.locations[reach.firstHit];
      if (targetLoc) {
        const moveCmd = "go to " + locAlias(targetLoc);
        if (!hints.some(h => h.cmd === moveCmd)) hints.push({cmd: moveCmd, eventId: null});
      }
    }
  }

  // E. Fallback baseline.
  if (hints.length === 0) {
    hints.push({cmd: "look", eventId: null});
  }

  // Deliberately no 'accuse X' hint — the detective must choose their own
  // suspect. Accusation is always available via free text, no hint needed.

  // Deduplicate by cmd text and cap at 5.
  const _seen = new Set();
  return hints.filter(h => { if (_seen.has(h.cmd)) return false; _seen.add(h.cmd); return true; }).slice(0, 5);
}

function renderHints() {
  const list = document.getElementById("hints-list");
  list.innerHTML = "";
  if (state.gameOver) {
    const m = document.createElement("div");
    m.className = "hints-note";
    m.textContent = "Case closed. Press Reset below to start a new investigation.";
    list.appendChild(m);
    return;
  }
  const pendingCount = Object.values(DATA.events).filter(
    ev => !state.executedEvents.includes(ev.id)
  ).length;
  // Player has hit the evidence threshold — show a prominent accuse chip.
  const canAccuse = state.executedEvents.length >= DATA.goal_events_needed;
  if (canAccuse) {
    const note = document.createElement("div");
    note.className = "hints-note";
    note.style.color = "var(--accent-soft)";
    note.innerHTML = "You have enough evidence. Review your notebook, pick your suspect, and accuse.";
    list.appendChild(note);
    // Show the suspect names as accusation chips
    const suspects = Object.values(DATA.characters)
      .filter(c => c.alive && (c.role === "suspect" || c.role === "associate"))
      .filter(c => state.encounteredCharacters.includes(c.id));
    suspects.forEach(c => {
      const surname = (c.name || "").split(" ").slice(-1)[0].toLowerCase();
      const chip = document.createElement("button");
      chip.type = "button";
      chip.className = "chip";
      chip.innerHTML = `<span class="arrow">&gt;</span>accuse ${surname}`;
      chip.title = "Click to fill input with accusation";
      chip.addEventListener("click", () => {
        prefillInput("accuse " + surname);
        _hintEventId = null; _hintCmd = null;
      });
      list.appendChild(chip);
    });
    if (pendingCount > 0) {
      const more = document.createElement("div");
      more.className = "hints-note";
      more.style.marginTop = "6px";
      more.style.fontSize = "11px";
      more.textContent = `(${pendingCount} more lead${pendingCount > 1 ? "s" : ""} still available if you want to keep digging)`;
      list.appendChild(more);
    }
    return;
  }
  const hints = buildHintCommands();
  hints.forEach(h => {
    const chip = document.createElement("button");
    chip.type = "button";
    chip.className = "chip";
    chip.innerHTML = `<span class="arrow">&gt;</span>${h.cmd}`;
    chip.title = "Click to fill input, then press Enter to submit";
    chip.addEventListener("click", () => {
      prefillInput(h.cmd);
      _hintEventId = h.eventId || null;
      _hintCmd = h.cmd;
    });
    list.appendChild(chip);
  });
}

function renderGuide() {
  const body = document.getElementById("guide-body");
  const list = document.getElementById("guide-list");
  body.innerHTML = "";
  list.innerHTML = "";

  const totalEvents = Object.keys(DATA.events).length;
  const done = state.executedEvents.length;
  const pendingEvents = Object.values(DATA.events).filter(
    ev => !state.executedEvents.includes(ev.id)
  );

  if (state.gameOver) {
    body.textContent = "Case closed. Press Reset to start a fresh investigation.";
    return;
  }
  if (pendingEvents.length === 0) {
    body.innerHTML =
      "You've exhausted every lead — <strong>" + done + " / " + totalEvents +
      " plot events explored</strong>. No more investigation branches remain. " +
      "Review your Detective's notebook, pick a suspect, and type " +
      "<code>accuse &lt;name&gt;</code> to close the case.";
    return;
  }

  // Find the nearest pending-event location via BFS from here.
  function nearest(fromId) {
    const seen = new Set([fromId]);
    const q = [[fromId, 0, [fromId]]];
    while (q.length) {
      const [cur, d, path] = q.shift();
      if (d > 0) {
        const hasPending = pendingEvents.some(ev => ev.location === cur);
        if (hasPending) return { id: cur, hops: d, path: path };
      }
      const loc2 = DATA.locations[cur];
      if (!loc2) continue;
      for (const adj of loc2.adjacent) {
        if (!seen.has(adj)) {
          seen.add(adj);
          q.push([adj, d + 1, path.concat([adj])]);
        }
      }
    }
    return null;
  }

  const hit = nearest(state.location);
  const pendingLocs = [...new Set(pendingEvents.map(ev => ev.location))];
  const locNames = pendingLocs
    .map(id => DATA.locations[id] ? DATA.locations[id].name : id)
    .filter(Boolean);

  if (hit) {
    const target = DATA.locations[hit.id];
    const nextStep = hit.path[1];              // first hop of the path
    const nextName = DATA.locations[nextStep] ? DATA.locations[nextStep].name : nextStep;
    body.innerHTML =
      "<strong>" + done + " / " + totalEvents + " plot events explored.</strong> " +
      pendingEvents.length + " lead" + (pendingEvents.length === 1 ? "" : "s") +
      " still open across " + pendingLocs.length + " location" +
      (pendingLocs.length === 1 ? "" : "s") + ". " +
      "The nearest unfinished lead is <strong>" + target.name + "</strong> — " +
      (hit.hops === 1 ? "directly next door." : hit.hops + " rooms away.") +
      (hit.hops > 1 ? " Start by heading to <strong>" + nextName + "</strong>." : "");

  } else {
    body.innerHTML =
      "<strong>" + done + " / " + totalEvents + " plot events explored.</strong> " +
      "No remaining lead is reachable from here — try another exit and come back to this guide.";
  }

  // Short overview: all locations with pending work (capped at 5)
  if (locNames.length > 0) {
    const overview = document.createElement("div");
    overview.className = "hints-note";
    overview.style.marginTop = "8px";
    overview.innerHTML = "<strong>Still open:</strong> " + locNames.slice(0, 5).join(" · ")
      + (locNames.length > 5 ? " · …" : "");
    list.appendChild(overview);
  }
}

function prefillInput(cmd) {
  const el = document.getElementById("cmd");
  el.value = cmd;
  el.focus();
  // Place cursor at end.
  el.setSelectionRange(cmd.length, cmd.length);
}

function renderSidebar() {
  const loc = DATA.locations[state.location];
  document.getElementById("location-name").textContent = loc ? loc.name : state.location;
  document.getElementById("location-desc").textContent = loc ? loc.description : "";

  const exitsEl = document.getElementById("exits");
  exitsEl.innerHTML = "";
  if (loc && loc.adjacent.length) {
    loc.adjacent.forEach(adj => {
      const btn = document.createElement("button");
      btn.className = "exit-btn";
      btn.textContent = "→ " + (DATA.locations[adj] ? DATA.locations[adj].name : adj);
      btn.addEventListener("click", () => runCommand("go to " + (DATA.locations[adj] ? DATA.locations[adj].name : adj)));
      exitsEl.appendChild(btn);
    });
  } else {
    exitsEl.innerHTML = '<li class="muted">(no obvious exits)</li>';
  }

  const charsEl = document.getElementById("characters-here");
  charsEl.innerHTML = "";
  const here = loc ? loc.characters : [];
  const aliveHere = here.filter(cid => { const c = DATA.characters[cid]; return c && c.alive; });
  const deadHere  = here.filter(cid => { const c = DATA.characters[cid]; return c && !c.alive; });
  if (!aliveHere.length && !deadHere.length) {
    charsEl.innerHTML = '<li class="muted">(no-one of interest)</li>';
  } else {
    deadHere.forEach(cid => {
      const c = DATA.characters[cid];
      const li = document.createElement("li");
      li.innerHTML = '<em style="color:var(--muted)">†&nbsp;' + c.name + ' — victim\'s body</em>';
      charsEl.appendChild(li);
    });
    aliveHere.forEach(cid => {
      const c = DATA.characters[cid];
      const li = document.createElement("li");
      li.textContent = c.name;
      charsEl.appendChild(li);
    });
  }

  const evEl = document.getElementById("evidence-here");
  evEl.innerHTML = "";
  const evHere = loc ? loc.evidence : [];
  if (!evHere.length) {
    evEl.innerHTML = '<li class="muted">(nothing catches the eye)</li>';
  } else {
    evHere.forEach(rawEid => {
      const eid = _normEvidenceId(rawEid);
      const e = DATA.evidence[eid];
      if (!e) return;
      const li = document.createElement("li");
      const flag = state.evidenceFlags[eid] || {};
      const short = truncate(e.description, 80);
      if (flag.discovered) {
        li.textContent = "☑ " + short;
        if (flag.destroyed) li.style.textDecoration = "line-through";
      } else {
        li.textContent = "☐ " + short;
        li.className = "muted";
      }
      evEl.appendChild(li);
    });
  }

  const kEl = document.getElementById("knowledge");
  kEl.innerHTML = "";
  if (!state.knowledge.length) {
    kEl.innerHTML = '<li class="muted">(nothing yet; investigate)</li>';
  } else {
    state.knowledge.slice().reverse().forEach(k => {
      const li = document.createElement("li");
      li.textContent = k;
      kEl.appendChild(li);
    });
  }

  renderSuspectList();

  document.getElementById("events-triggered").textContent = state.executedEvents.length;
  document.getElementById("events-needed").textContent = DATA.goal_events_needed;
  const pct = Math.min(100, Math.round(100 * state.executedEvents.length / DATA.goal_events_needed));
  document.getElementById("pbar").style.width = pct + "%";
  const pctEl = document.getElementById("pbar-pct");
  if (pctEl) pctEl.textContent = pct + "%";
}

function truncate(s, n) { return s.length > n ? s.slice(0, n - 1) + "…" : s; }

function _normEvidenceId(eid) {
  // Normalize bare IDs like "E007" to "evidence.E007"
  return (typeof eid === "string" && !eid.startsWith("evidence.")) ? "evidence." + eid : eid;
}
function sceneSketch(loc) {
  if (!loc) return null;
  const chars = (loc.characters || [])
    .map(cid => {
      const c = DATA.characters[cid];
      if (c && c.alive) return c.name;
      // Fallback: pretty-print the ID if character data is missing
      if (!c) return cid.replace(/^character\./, "").replace(/_/g, " ").replace(/\b\w/g, l => l.toUpperCase());
      return null;
    })
    .filter(Boolean);
  const items = (loc.evidence || [])
    .map(eid => {
      const e = DATA.evidence[_normEvidenceId(eid)];
      return e ? e.description : null;
    })
    .filter(Boolean);
  const out = [];
  if (chars.length) out.push("Here with you: " + chars.join(", ") + ".");
  if (items.length) out.push("You notice: " + items.join("; ") + ".");
  return out.length ? out.join(" ") : null;
}

function renderSuspectList() {
  const susEl = document.getElementById("suspects");
  susEl.innerHTML = "";
  const candidates = Object.values(DATA.characters)
    .filter(c => c.alive && (c.role === "suspect" || c.role === "conspirator" || c.role === "associate"));
  const anyEncountered = candidates.some(c => state.encounteredCharacters.includes(c.id));
  if (!anyEncountered) {
    const li = document.createElement("li");
    li.className = "muted";
    li.style.fontStyle = "italic";
    li.textContent = "(no suspects known yet — investigate locations to uncover them)";
    susEl.appendChild(li);
    return;
  }
  candidates.forEach(c => {
    const li = document.createElement("li");
    if (!state.encounteredCharacters.includes(c.id)) {
      li.className = "suspect-locked";
      li.textContent = "? unidentified associate";
      susEl.appendChild(li);
      return;
    }
    li.className = "suspect-row";
    const btn = document.createElement("button");
    btn.className = "suspect-name";
    btn.type = "button";
    btn.innerHTML = '<span class="chev">▸</span>' + c.name;
    btn.addEventListener("click", () => toggleSuspectBlurb(li, btn, c));
    li.appendChild(btn);
    susEl.appendChild(li);
  });
}

function toggleSuspectBlurb(container, btn, c) {
  const existing = container.querySelector(".blurb");
  if (existing) {
    existing.remove();
    btn.querySelector(".chev").textContent = "▸";
    return;
  }
  const d = document.createElement("div");
  d.className = "blurb";
  d.textContent = c.blurb || "No notes yet.";
  container.appendChild(d);
  btn.querySelector(".chev").textContent = "▾";
}

// -------------------- command parsing --------------------
function interpret(raw) {
  const clean = raw.trim().toLowerCase().replace(/[^\w\s]/g, " ").replace(/\s+/g, " ").trim();
  if (!clean) return {verb: "noop"};
  const tokens = clean.split(" ");

  if (tokens[0] === "quit" || tokens[0] === "exit") return {verb: "quit"};
  if (clean === "look" || clean === "look around" || clean === "survey") return {verb: "look"};
  if (clean === "notebook" || clean === "notes") return {verb: "notebook"};
  if (clean === "help" || clean === "?") return {verb: "help"};

  // Go / move
  let m = clean.match(/^(?:go|move|walk|head|travel)\s+(?:to\s+|the\s+|into\s+)*(.+)$/);
  if (m) return {verb: "move", target: m[1].trim()};
  // Accuse
  m = clean.match(/^(?:accuse|arrest|charge|name)\s+(.+)$/);
  if (m) return {verb: "accuse", target: m[1].trim()};
  // Destroy-family (drama-manager exception)
  m = clean.match(/^(?:smash|destroy|break|shatter|burn|hide|steal|pocket|throw away)\s+(.+)$/);
  if (m) return {verb: "destroy", target: m[1].trim()};
  // Interview / question / talk to / confront / consult (expert or witness)
  m = clean.match(/^(?:interview|question|talk\s+to|ask|confront|speak\s+to|consult)\s+(.+)$/);
  if (m) return {verb: "interview", target: m[1].trim()};
  // Examine / inspect
  m = clean.match(/^(?:examine|inspect|look\s+at|study|observe|check)\s+(.+)$/);
  if (m) return {verb: "examine", target: m[1].trim()};
  // Analyze / test
  m = clean.match(/^(?:analyze|analyse|test|run)\s+(.+)$/);
  if (m) return {verb: "analyze", target: m[1].trim()};
  // Search / investigate
  m = clean.match(/^(?:search|explore|look\s+around|scour|investigate)\s+(.+)$/);
  if (m) return {verb: "search", target: m[1].trim()};
  // Visit
  m = clean.match(/^visit\s+(.+)$/);
  if (m) return {verb: "visit", target: m[1].trim()};
  // Take / pick up
  m = clean.match(/^(?:take|grab|pick\s+up|pocket)\s+(.+)$/);
  if (m) return {verb: "take", target: m[1].trim()};

  return {verb: "custom", target: clean};
}

// -------------------- entity matching --------------------
function matchEntity(target, pool) {
  if (!target) return null;
  const tgt = target.toLowerCase();
  const tgtTokens = new Set(tgt.split(" "));
  let best = null;
  let bestScore = 0;
  for (const e of pool) {
    let score = 0;
    for (const a of (e.aliases || [])) {
      if (tgt.includes(a)) score += a.length;
      else if (tgtTokens.has(a)) score += 2;
    }
    if (score > bestScore) { bestScore = score; best = e; }
  }
  return bestScore >= 3 ? best : null;
}

function matchLocation(target) {
  const loc = DATA.locations[state.location];
  const candidates = loc ? loc.adjacent.map(id => DATA.locations[id]).filter(Boolean) : [];
  // also allow matching all locations if user typed an obvious name
  const hit = matchEntity(target, candidates) || matchEntity(target, Object.values(DATA.locations));
  return hit;
}

function matchCharacter(target) {
  return matchEntity(target, Object.values(DATA.characters));
}

function matchEvidence(target) {
  return matchEntity(target, Object.values(DATA.evidence));
}

// -------------------- action handlers --------------------
function currentLoc() { return DATA.locations[state.location]; }

function eventAtHereMatching(verb, target) {
  // Find a plan event whose location == current AND verb matches AND args include an entity matching target
  const loc = state.location;
  const pool = Object.values(DATA.events).filter(ev => ev.location === loc);
  if (!pool.length) return null;

  // Verb family alignment
  const verbFamily = {
    "examine": ["examine", "search", "observe", "check", "analyze", "investigate"],
    "search":  ["search", "examine", "observe", "investigate"],
    "analyze": ["analyze", "examine", "investigate"],
    "interview": ["interview", "question", "consult", "confront", "visit"],
    "visit": ["visit", "interview", "investigate"],
  };
  const allowedVerbs = new Set(verbFamily[verb] || [verb]);

  const tgtLower = (target || "").toLowerCase();

  let best = null, bestScore = 0;
  for (const ev of pool) {
    if (state.executedEvents.includes(ev.id)) continue;
    if (!allowedVerbs.has(ev.verb)) continue;
    // Enforce partial order: all predecessors must have been executed first.
    const preds = DATA.predecessors[ev.id] || [];
    if (preds.some(p => !state.executedEvents.includes(p))) continue;

    // Match arg tokens against target
    let score = 0;
    for (const arg of ev.args) {
      const argStr = String(arg).toLowerCase();
      const argTokens = argStr.split(/[_\s.]+/).filter(t => t.length > 2);
      for (const t of argTokens) {
        if (tgtLower.includes(t)) score += t.length;
      }
    }
    if (score > bestScore) { bestScore = score; best = ev; }
  }
  return bestScore >= 3 ? best : null;
}

function _extractPersonLike(s) {
  // Heuristic name-spotter for free-string event args. Picks up
  // "Dr. Helena Frost" / "Professor Vane" / "Lord Ashworth" style names.
  const m = String(s).match(
    /\b(?:Dr|Mr|Mrs|Ms|Lady|Lord|Inspector|Sir|Madam|Professor|Prof)\.?\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b/
  );
  if (m) return m[0];
  // Fallback: two+ capitalised words in a row.
  const n = String(s).match(/\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,2}\b/);
  return n ? n[0] : null;
}

function note(entry) {
  // Deduplicate: same line never appears twice in the notebook.
  if (!state.knowledge.includes(entry)) state.knowledge.push(entry);
}

function executeEvent(ev) {
  state.executedEvents.push(ev.id);
  dmLog("constituent",
        "executed plan event " + ev.id + " (" + ev.verb + ")",
        "location=" + (ev.location || "?") + " reveals=[" + (ev.reveals || []).join(",") + "]");

  // Encounter any character whose name appears in this event's prose —
  // some conspirators only surface in narrative text (not in structured
  // args), so we scan description + narrative too.
  const eventText = ((ev.narrative || "") + " " + (ev.description || "")).toLowerCase();
  for (const cid of Object.keys(DATA.characters)) {
    if (state.encounteredCharacters.includes(cid)) continue;
    const c = DATA.characters[cid];
    if (!c || !c.alive) continue;
    const needles = [c.name.toLowerCase()].concat(
      (c.aliases || []).filter(a => a.length > 3 && !/^\d+$/.test(a))
    );
    if (needles.some(k => k && eventText.includes(k))) encounter(cid);
  }

  const socialVerbs = new Set(["interview","consult","confront","question","visit"]);
  (ev.args || []).forEach(a => {
    const asStr = String(a);
    if (asStr.startsWith("character.")) {
      encounter(asStr);
      if (socialVerbs.has(ev.verb)) {
        const c = DATA.characters[asStr];
        if (c) note("Spoke with — " + c.name);
      }
    } else if (socialVerbs.has(ev.verb)) {
      // Free-string arg referencing a person off the main character list
      // (e.g. an off-screen consultant like Dr. Helena Frost).
      const name = _extractPersonLike(asStr);
      if (name) note("Consulted — " + name);
    }
  });
  // Apply reveals: mark evidence discovered (dedup via note()).
  // Some events in plan.json list reveals with the full 'evidence.E007' id,
  // others use just 'E007'. Normalize so both forms resolve to the same
  // entry in DATA.evidence.
  (ev.reveals || []).forEach(raw => {
    const rawStr = String(raw);
    const eid = rawStr.startsWith("evidence.") ? rawStr : "evidence." + rawStr;
    if (!state.evidenceFlags[eid]) state.evidenceFlags[eid] = {};
    state.evidenceFlags[eid].discovered = true;
    const e = DATA.evidence[eid];
    if (e) note("Evidence: " + truncate(e.description, 60));
    else   note("Lead: " + eid.replace(/^evidence\./, ""));
  });
  if (ev.verb === "analyze") {
    (ev.args || []).forEach(a => {
      const m = matchEvidence(String(a));
      if (m) {
        if (!state.evidenceFlags[m.id]) state.evidenceFlags[m.id] = {};
        state.evidenceFlags[m.id].analyzed = true;
      }
    });
  }
  // Any social-verb event on a named character counts as "interviewed"
  // so the hint engine stops nagging us to speak with them again.
  if (socialVerbs.has(ev.verb)) {
    (ev.args || []).forEach(a => {
      if (String(a).startsWith("character.")) {
        if (!state.charactersInterviewed.includes(a)) state.charactersInterviewed.push(a);
      }
    });
  }
  // Strip any leading markdown '# ...' lines the LLM may have left on the
  // plot-point prose (spurious chapter-style headings inside event text).
  const cleanNarrative = (ev.narrative || "").replace(/^(?:#+[^\n]*\n\s*)+/, "").trim();
  addLog(highlightText(cleanNarrative), "narration", "— " + ev.description + " —", /*asHtml=*/ true);
  addLog("Your case grows clearer. (" + state.executedEvents.length + " / " + DATA.goal_events_needed + " plot events explored.)", "system");
  // If that was the last event at this location, nudge the player to move.
  const remainingAtHere = Object.values(DATA.events).filter(
    ev2 => ev2.location === state.location && !state.executedEvents.includes(ev2.id)
  ).length;
  if (remainingAtHere === 0) {
    addLog(
      "Nothing more to investigate at " + (currentLoc() ? currentLoc().name : "this scene") +
      ". Try an exit from the left panel, or open 🗺️ Where next? to see what's still open elsewhere.",
      "system"
    );
  }
  // Final-event nudge, once.
  const pendingCountAfter = Object.values(DATA.events).filter(
    ev2 => !state.executedEvents.includes(ev2.id)
  ).length;
  if (pendingCountAfter === 0 && !state.allDoneAnnounced) {
    state.allDoneAnnounced = true;
    addLog(highlightText("Every trail you could follow has been followed. Every interview logged, every record pulled, every room canvassed. Your notebook is full. It is time to name the poisoner — type 'accuse <suspect>'."), "narration", "— the case, fully explored —", /*asHtml=*/ true);
  }
}

function handleMove(target) {
  const loc = matchLocation(target);
  if (!loc) {
    dmLog("move", "rejected move: '" + target + "' not reachable", "");
    addLog("You cannot find a way to " + target + " from here. (Click an exit in the left panel, or try one of the names listed there.)", "outcome");
    return;
  }
  const cur = currentLoc();
  if (!cur || !cur.adjacent.includes(loc.id)) {
    dmLog("move", "rejected move: " + loc.name + " not adjacent", "");
    // Find a one-hop intermediary to suggest
    let via = null;
    if (cur) {
      for (const adjId of cur.adjacent) {
        const adjLoc = DATA.locations[adjId];
        if (adjLoc && adjLoc.adjacent && adjLoc.adjacent.includes(loc.id)) { via = adjLoc; break; }
      }
    }
    const viaStr = via ? ` — you'd need to pass through ${via.name} first` : "";
    addLog(
      `You cannot reach ${loc.name} directly from ${cur ? cur.name : "here"}${viaStr}. Use the exits on the left panel to navigate.`,
      "outcome"
    );
    return;
  }
  state.lastLocation = state.location;
  state.location = loc.id;
  encounterAll(loc.characters);
  dmLog("move", "detective moved to " + loc.name, "no classification");
  addLog("You make your way to " + loc.name + ".", "system");
  addLog(loc.description, "outcome");
  const sketch = sceneSketch(loc);
  if (sketch) addLog(highlightText(sketch), "outcome", null, /*asHtml=*/ true);
}

function handleLook() {
  const loc = currentLoc();
  if (!loc) return;
  addLog(loc.description, "outcome");
  const sketch = sceneSketch(loc);
  if (sketch) addLog(highlightText(sketch), "outcome", null, /*asHtml=*/ true);
}

function handleExamine(target) {
  if (!target) { handleLook(); return; }
  // First try a plan event at this location
  const ev = eventAtHereMatching("examine", target);
  if (ev) { executeEvent(ev); return; }
  // Maybe user is pointing at evidence here, just not the right kind of examine
  const loc = currentLoc();
  if (loc) {
    for (const rawEid of loc.evidence) {
      const eid = _normEvidenceId(rawEid);
      const e = DATA.evidence[eid];
      if (!e) continue;
      const tgtLow = target.toLowerCase();
      const descWords = (e.description || "").toLowerCase().split(/\W+/).filter(w => w.length > 3);
      const aliasMatch = (e.aliases || []).some(a => tgtLow.includes(a));
      const descMatch = descWords.some(w => tgtLow.includes(w));
      if (aliasMatch || descMatch) {
        dmLog("consistent", "examined " + eid + " in-place (no new plan event)", "no plan effect");
        addLog("You examine the " + truncate(e.description, 60) + ", but nothing new reveals itself here.", "outcome");
        return;
      }
    }
  }
  dmLog("consistent", "examine '" + target + "' — nothing matched", "no plan effect");
  addLog("You look, but " + target + " doesn't seem to be here — or Inspector Rothwell can't make sense of it from this angle.", "outcome");
}

function handleInterview(target) {
  // Try to match a plan event at this location first. Consult-type events
  // may reference off-screen experts (e.g. 'Dr. Helena Frost' called in to
  // authenticate a painting) who aren't in the room's character list.
  const evPre = eventAtHereMatching("interview", target);
  if (evPre) { executeEvent(evPre); return; }
  const c = matchCharacter(target);
  if (!c) {
    addLog("You call out for " + target + ", but no one answers. Try questioning someone listed in the left panel, or move to a location where that person is waiting.", "outcome");
    return;
  }
  const loc = currentLoc();
  if (!loc || !loc.characters.includes(c.id)) {
    addLog(c.name + " isn't here. You'll need to find them first.", "outcome");
    return;
  }
  // Nothing plan-relevant left to extract from this character at this
  // location. Mark them "interviewed" so the hint engine stops looping.
  if (!state.charactersInterviewed.includes(c.id)) state.charactersInterviewed.push(c.id);
  dmLog("consistent", "questioned " + c.name + " (no plan event)", "no plan effect");
  addLog(c.name + " answers politely but says nothing that moves the case forward here.", "outcome");
}

function handleAnalyze(target) {
  const ev = eventAtHereMatching("analyze", target);
  if (ev) { executeEvent(ev); return; }
  if (!currentLoc().id.includes("lab") && !currentLoc().id.includes("forensic")) {
    dmLog("consistent", "analyze '" + target + "' in non-lab setting", "no plan effect");
    addLog("You'd need proper equipment. Try a forensic laboratory.", "outcome");
    return;
  }
  dmLog("consistent", "analyze '" + target + "' (no matching plan event)", "no plan effect");
  addLog("The analysis yields nothing useful.", "outcome");
}

function handleSearch(target) {
  const ev = eventAtHereMatching("search", target);
  if (ev) { executeEvent(ev); return; }
  dmLog("consistent", "search '" + target + "' (no matching plan event)", "no plan effect");
  addLog("You search " + (target || "the area") + " thoroughly but find nothing new.", "outcome");
}

function handleAccuse(target) {
  const c = matchCharacter(target);
  if (!c) {
    dmLog("consistent", "accuse '" + target + "' — unknown name", "no plan effect");
    addLog("You announce an accusation, but the name doesn't register with anyone present. Be specific: try 'accuse <surname>'.", "outcome");
    return;
  }
  if (state.executedEvents.length < DATA.goal_events_needed) {
    dmLog("exceptional", "premature accusation of " + c.name, "intervention: evidence threshold not met");
    addLog("You haven't gathered enough evidence yet. An accusation without proof will be dismissed. Keep investigating.", "exception");
    return;
  }
  if (c.id === DATA.real_criminal_id) {
    state.gameOver = true;
    dmLog("constituent", "correct accusation: " + c.name, "case closed");
    addLog(highlightText("You lay the hollow ring on the evidence table with deliberate care. " +
      "Dr. Pemberton — broken in the interrogation hours earlier — has already " +
      "given you the skeleton of the account. Charlotte Devereaux's fingerprints " +
      "on the planted jeweler's receipt do the rest. You turn to " + c.name +
      " and read the warrant aloud: the gallery owner, the half-sister, the " +
      "poisoner. Her face composes itself into that flawless marble calm that " +
      "has served her so well for so long. It does not survive the click of the " +
      "handcuffs."), "victory", "— THE CASE IS CLOSED —", /*asHtml=*/ true);
    addLog(highlightText("Outside, fog rises off the Thames in slow, indifferent ribbons. In the " +
      "butler's pantry of the Ashford townhouse, a housemaid named Margaret " +
      "grips a silver tray until her knuckles whiten — she had watched the " +
      "mechanism click through a sliver of doorway on the night of the toast, " +
      "and told no one. Now that the arrest is done, she will not need to. You " +
      "step out into the grey dawn with Detective Martinez at your shoulder, " +
      "the case file heavy in your coat, and listen to the constable's whistle " +
      "fading up the square."), "narration", "— the loose end —", /*asHtml=*/ true);
    addLog(highlightText("Justice comes fitfully in this city — most often late, and never entire. " +
      "But tonight, at least, it has come for " + DATA.victim_name + ". " +
      "The Hartley Affair is closed."), "narration", "— coda —", /*asHtml=*/ true);
    addLog("Case record closed. Press Reset to begin a new investigation.", "system");
  } else {
    state.gameOver = true;
    dmLog("exceptional", "wrong accusation: " + c.name, "real killer escapes; case closed with wrongful arrest");
    addLog(highlightText("You name " + c.name + " as the poisoner of " + DATA.victim_name + ". " +
      "The room falls silent. Constable Morris steps forward, handcuffs in hand. " +
      c.name + " protests — weakly at first, then with escalating desperation — " +
      "but the chain of evidence you have assembled looks tidy enough on paper, " +
      "and the magistrate will not ask too many questions tonight. The arrest is " +
      "done. A reporter from the Daily Chronicle is already scribbling his " +
      "headline."), "narration", "— A WRONGFUL ARREST —", /*asHtml=*/ true);
    addLog(highlightText("Somewhere at the edge of the room — by a window, lit gold by the gaslight — " +
      "another figure watches the proceedings with an expression carefully " +
      "arranged between sympathy and concern. That figure's fingers brush a ring " +
      "on their hand for the briefest instant, then release. You do not notice. " +
      "You are already drafting your report."), "narration", "— the one who walked free —", /*asHtml=*/ true);
    addLog(highlightText("The Hartley Affair is closed in the ledger of Scotland Yard, but not in " +
      "the city that reads about it tomorrow. Somewhere, the truth drifts up " +
      "the Thames with the morning fog — thin, persistent, and, for now, unheard."), "narration", "— coda —", /*asHtml=*/ true);
    addLog("A wrongful arrest has ended the case. Press Reset to try again.", "system");
  }
}

function handleDestroy(target) {
  // Drama-manager "exceptional" pantomime
  dmLog("exceptional",
        "destructive action on '" + target + "' refused",
        "intervention: preserve story path");
  addLog("You reach to tamper with " + target + ", but catch yourself. An inspector who destroys evidence is no inspector at all. (The drama manager has intervened — your story still has a path to the truth.)", "exception");
}

// -------------------- API mode dispatch --------------------
async function _runCommandViaAPI(raw, forceEventId) {
  const cmdEl   = document.getElementById("cmd");
  const submitEl = document.querySelector("#input-form button[type=submit]");
  cmdEl.disabled = true;
  if (submitEl) submitEl.disabled = true;
  try {
    const base = API_URL || "";
    const body = {command: raw};
    if (forceEventId) body.force_event_id = forceEventId;
    const resp = await fetch(base + "/api/step", {
      method: "POST",
      headers: {"Content-Type": "application/json"},
      body: JSON.stringify(body),
    });
    if (!resp.ok) throw new Error("Server " + resp.status);
    const data = await resp.json();

    for (const e of (data.log_entries || [])) {
      const txt = e.as_html ? highlightText(e.text) : e.text;
      addLog(txt, e.cls || "outcome", e.title || null, e.as_html || false);
    }
    if (data.triggered_event_id &&
        !state.executedEvents.includes(data.triggered_event_id)) {
      state.executedEvents.push(data.triggered_event_id);
    }
    if (data.moved_to) {
      state.lastLocation = state.location;
      state.location = data.moved_to;
    }
    for (const k of (data.new_knowledge || [])) {
      if (!state.knowledge.includes(k)) state.knowledge.push(k);
    }
    for (const [eid, flags] of Object.entries(data.new_evidence_flags || {})) {
      if (!state.evidenceFlags[eid]) state.evidenceFlags[eid] = {};
      Object.assign(state.evidenceFlags[eid], flags);
    }
    for (const cid of (data.characters_encountered || [])) encounter(cid);
    for (const cid of (data.characters_interviewed || [])) {
      if (!state.charactersInterviewed.includes(cid))
        state.charactersInterviewed.push(cid);
    }
    if (data.dm_entry) {
      state.dmLog.push({
        turn: state.turns,
        kind:        data.dm_entry.kind || "consistent",
        summary:     data.dm_entry.summary || "",
        detail:      data.dm_entry.detail || "",
        // extra fields for minimal view
        event_desc:  data.dm_entry.event_desc  || "",
        plan_change: data.dm_entry.plan_change || "",
        goal_verdict:data.dm_entry.goal_verdict|| "",
        reveals:     data.dm_entry.reveals     || [],
        remaining_after: data.dm_entry.remaining_after ?? null,
        t: Date.now(),
      });
      if (state.dmLog.length > 200) state.dmLog.shift();
    }
    if (data.game_over) state.gameOver = true;
  } catch (err) {
    addLog("(LLM server error: " + err.message + ". Check the Colab notebook.)", "system");
  } finally {
    cmdEl.disabled = false;
    if (submitEl) submitEl.disabled = false;
    cmdEl.focus();
  }
}

// -------------------- dispatch --------------------
async function runCommand(raw, forceEventId) {
  forceEventId = forceEventId || null;
  if (state.gameOver) {
    addLog("The case is already closed. Press Reset to begin a new investigation.", "system");
    return;
  }
  addLog(raw, "user");
  state.turns += 1;

  const action = interpret(raw);
  const isAccuse = action.verb === "accuse";

  if (API_URL !== null && !isAccuse) {
    await _runCommandViaAPI(raw, forceEventId);
    saveState();
    renderSidebar();
    renderHints();
    const g = document.getElementById("guide-content");
    if (g && !g.classList.contains("hidden")) renderGuide();
    const d = document.getElementById("dmlog-content");
    if (d && !d.classList.contains("hidden")) renderDmLog();
    return;
  }

  // JS-only path: if a plan event ID was forced (hint chip), execute directly.
  if (forceEventId && DATA.events[forceEventId]) {
    const forcedEv = DATA.events[forceEventId];
    if (!state.executedEvents.includes(forceEventId) && forcedEv.location === state.location) {
      executeEvent(forcedEv);
      saveState();
      renderSidebar();
      renderHints();
      const g = document.getElementById("guide-content");
      if (g && !g.classList.contains("hidden")) renderGuide();
      const d = document.getElementById("dmlog-content");
      if (d && !d.classList.contains("hidden")) renderDmLog();
      return;
    }
  }

  switch (action.verb) {
    case "noop":    break;
    case "quit":    addLog("You step back from the case. Press Reset to start over.", "system"); break;
    case "look":    handleLook(); break;
    case "notebook":
    case "notes":   addLog("See the right panel for your notebook entries.", "system"); break;
    case "help":
      addLog("Try any of: look · examine <thing> · go to <location> · question <person> · analyze <object> · search <place> · accuse <person>. You can also click an exit on the left panel to move.", "system");
      break;
    case "move":    handleMove(action.target); break;
    case "examine": handleExamine(action.target); break;
    case "interview": handleInterview(action.target); break;
    case "analyze": handleAnalyze(action.target); break;
    case "search":  handleSearch(action.target); break;
    case "visit":   handleMove(action.target); break;
    case "take":    addLog("Better not disturb evidence. Examine it in place instead.", "outcome"); break;
    case "accuse":  handleAccuse(action.target); break;
    case "destroy": handleDestroy(action.target); break;
    case "custom":
    default:
      addLog("Inspector Rothwell tilts his head. \"" + raw + "\" isn't a move he can make on this case — try a simpler verb like look, examine, go, question, analyze, or accuse.", "system");
  }
  saveState();
  renderSidebar();
  renderHints();
  const g = document.getElementById("guide-content");
  if (g && !g.classList.contains("hidden")) renderGuide();
  const d = document.getElementById("dmlog-content");
  if (d && !d.classList.contains("hidden")) renderDmLog();
}

// -------------------- wiring --------------------
document.getElementById("input-form").addEventListener("submit", (e) => {
  e.preventDefault();
  const el = document.getElementById("cmd");
  const raw = el.value;
  el.value = "";
  if (!raw.trim()) return;
  // Enforce 8-word ceiling like the backend engine
  const words = raw.trim().split(/\s+/).slice(0, 8);
  const finalRaw = words.join(" ");
  // Use stored hint force-event only if the player submitted without editing
  let forceId = null;
  if (_hintEventId && _hintCmd && finalRaw.toLowerCase() === _hintCmd.toLowerCase()) {
    forceId = _hintEventId;
  }
  _hintEventId = null;
  _hintCmd = null;
  runCommand(finalRaw, forceId);
});

// Clear pending hint state whenever the player types in the box
document.getElementById("cmd").addEventListener("input", () => {
  _hintEventId = null;
  _hintCmd = null;
});

document.getElementById("reset-btn").addEventListener("click", async () => {
  if (!confirm("Start a new investigation? Your current progress will be lost.")) return;
  if (API_URL !== null) {
    try { await fetch((API_URL || "") + "/api/new_game", {method: "POST"}); } catch (_) {}
  }
  localStorage.removeItem(STORAGE_KEY);
  state = freshState();
  logEl.innerHTML = "";
  greet();
  renderSidebar();
  renderHints();
  document.getElementById("cmd").value = "";
  document.getElementById("cmd").focus();
});

// Mutually-exclusive toggle helpers — opening one panel closes the others.
const _PANELS = [
  { toggleId: "hints-toggle",  contentId: "hints-content",  labelClose: "💡 Hide hints",       labelOpen: "💡 Stuck? Show hints",     onOpen: () => renderHints() },
  { toggleId: "guide-toggle",  contentId: "guide-content",  labelClose: "🗺️ Hide guide",        labelOpen: "🗺️ Where next?",           onOpen: () => renderGuide() },
  { toggleId: "dmlog-toggle",  contentId: "dmlog-content",  labelClose: "🎬 Hide DM log",       labelOpen: "🎬 Drama Manager log",      onOpen: () => renderDmLog() },
];
function _closeAllPanels(exceptId) {
  _PANELS.forEach(p => {
    if (p.contentId === exceptId) return;
    const box = document.getElementById(p.contentId);
    const tog = document.getElementById(p.toggleId);
    if (box && !box.classList.contains("hidden")) {
      box.classList.add("hidden");
      if (tog) { tog.classList.remove("open"); tog.textContent = p.labelOpen; }
    }
    // Hide verbose label when DM log closes
    if (p.contentId === "dmlog-content") {
      const lbl = document.getElementById("dm-verbose-label");
      if (lbl) lbl.style.display = "none";
    }
  });
}
_PANELS.forEach(p => {
  const tog = document.getElementById(p.toggleId);
  if (!tog) return;
  tog.addEventListener("click", () => {
    const box = document.getElementById(p.contentId);
    const isHidden = box.classList.contains("hidden");
    _closeAllPanels(null); // close everything first
    if (isHidden) {
      box.classList.remove("hidden");
      tog.classList.add("open");
      tog.textContent = p.labelClose;
      if (p.contentId === "dmlog-content") {
        const lbl = document.getElementById("dm-verbose-label");
        if (lbl) lbl.style.display = "";
      }
      p.onOpen();
    }
  });
});

// Verbose / minimal toggle checkbox for DM log.
document.getElementById("dm-verbose-chk").addEventListener("change", (e) => {
  dmVerbose = e.target.checked;
  renderDmLog();
});

// -------------------- novel overlay --------------------
function _mdToHtml(md) {
  // Minimal markdown renderer: headings, bold, italic, paragraphs, line breaks.
  const s = md
    .replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;")
    .replace(/^#{3}\s+(.*)$/gm,"<h3>$1</h3>")
    .replace(/^#{2}\s+(.*)$/gm,"<h2>$1</h2>")
    .replace(/^#\s+(.*)$/gm,"<h1>$1</h1>")
    .replace(/\*\*(.+?)\*\*/g,"<strong>$1</strong>")
    .replace(/\*(.+?)\*/g,"<em>$1</em>")
    .replace(/\n{2,}/g,"</p><p>")
    .replace(/\n/g,"<br>");
  return "<p>" + s + "</p>";
}
document.getElementById("novel-btn").addEventListener("click", () => {
  const story = DATA.story_text || "";
  if (!story.trim()) {
    alert("No story text available.");
    return;
  }
  document.getElementById("novel-content").innerHTML = _mdToHtml(story);
  document.getElementById("novel-overlay").style.display = "block";
  document.body.style.overflow = "hidden";
});
document.getElementById("novel-close").addEventListener("click", () => {
  document.getElementById("novel-overlay").style.display = "none";
  document.body.style.overflow = "";
});

function greet() {
  addLog("CASE: The Hartley Affair", "system");
  addLog(highlightText("You are " + DATA.detective_name + " of Scotland Yard. It is past midnight. " +
    "Tonight, the host of a private gallery reception — " + DATA.victim_name + " — " +
    "collapsed in his own home during the opening toast. The attending physician called it heart failure; " +
    "you are unconvinced. A constable has just opened the door for you, and the body lies where it fell."), "narration", "— the call that brought you here —", /*asHtml=*/ true);
  const loc = currentLoc();
  addLog(
    "You step into " + (loc ? loc.name : "the scene") + ". " + (loc ? loc.description : ""),
    "outcome"
  );
  addLog(
    "Open-ended investigation. Type any command (≤ 8 words). A good first move is 'examine body'. " +
    "You can move with 'go to <place>', 'question <name>', 'analyze <object>'. When you know the poisoner, 'accuse <name>'.",
    "system"
  );
}

renderSidebar();
renderHints();
if (state.executedEvents.length === 0 && state.turns === 0) greet();
else addLog("(Session resumed. " + state.executedEvents.length + " plot events already explored.)", "system");
document.getElementById("cmd").focus();
</script>
</body>
</html>
"""


def build() -> None:
    data = build_game_data()
    html_out = HTML_TEMPLATE.replace("__GAME_DATA__", json.dumps(data, ensure_ascii=False))
    OUT.write_text(html_out, encoding="utf-8")
    print(f"Wrote {OUT}  ({OUT.stat().st_size/1024:.1f} KB)")
    print(f"Events: {len(data['events'])}  Locations: {len(data['locations'])}  Characters: {len(data['characters'])}")
    print(f"Starting location: {data['starting_location']}  ({data['locations'][data['starting_location']]['name']})")
    print(f"Win condition: accuse {data['real_criminal_name']} (id={data['real_criminal_id']}) after {data['goal_events_needed']} events")


if __name__ == "__main__":
    build()


In [ ]:
%%writefile web/api_server.py
"""FastAPI server for the LLM-powered detective game.

Serves a dynamically built game page (DATA injected from data/) with the
API_URL wired in, so the browser routes every command through the real
action-interpreter + drama-manager + LLM narration stack.

Usage (from repo root):
    uvicorn web.api_server:app --host 0.0.0.0 --port 8080

Or from /content in Colab after all modules are written:
    import uvicorn
    uvicorn.run("api_server:app", host="0.0.0.0", port=8080, reload=False)
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse
from pydantic import BaseModel

# Make sure the project root (or /content in Colab) is importable.
ROOT = Path(__file__).resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from game_engine import EngineConfig, GameEngine  # noqa: E402
from story_to_plan import load_plan               # noqa: E402
from world_builder import load_world              # noqa: E402

app = FastAPI(title="Detective Game API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

_engine: GameEngine | None = None
_html_cache: str | None = None
DATA_DIR = ROOT / "data"


def _make_engine() -> GameEngine:
    plan = load_plan(str(DATA_DIR / "plan.json"))
    world = load_world(str(DATA_DIR / "world.json"))
    detective_name = "Inspector Rothwell"
    case_path = DATA_DIR / "case_file.json"
    if case_path.exists():
        try:
            case = json.loads(case_path.read_text(encoding="utf-8"))
            detective_name = case.get("detective", {}).get("name", detective_name)
        except Exception:
            pass
    config = EngineConfig(
        narrate_with_llm=True,
        log_dir=ROOT / "logs" / "web",
        detective_name=detective_name,
    )
    return GameEngine(plan, world, config)


def _get_engine() -> GameEngine:
    global _engine
    if _engine is None:
        _engine = _make_engine()
    return _engine


def _build_html() -> str:
    """Return the game HTML with DATA from data/ and API_URL injected."""
    global _html_cache
    if _html_cache is not None:
        return _html_cache

    # Prefer build_game's template so DATA always matches the generated mystery.
    web_dir = ROOT / "web"
    if str(web_dir) not in sys.path:
        sys.path.insert(0, str(web_dir))
    try:
        from build_game import HTML_TEMPLATE, build_game_data  # type: ignore
        data = build_game_data()
        html = HTML_TEMPLATE.replace("__GAME_DATA__", json.dumps(data, ensure_ascii=False))
    except Exception:
        # Fallback: root index.html (may have a different mystery baked in)
        html = (ROOT / "index.html").read_text(encoding="utf-8")

    # Inject empty-string API_URL → JS detects this as API mode (relative URL).
    html = html.replace("let API_URL = null;", 'let API_URL = "";')
    _html_cache = html
    return html


@app.get("/", response_class=HTMLResponse)
async def serve_index():
    return HTMLResponse(_build_html())


@app.post("/api/new_game")
async def new_game():
    global _engine, _html_cache
    _engine = _make_engine()
    _html_cache = None  # bust so DATA is re-read on next page load
    return {"status": "ok"}


class StepRequest(BaseModel):
    command: str
    force_event_id: str | None = None


@app.post("/api/step")
async def api_step(req: StepRequest):
    eng = _get_engine()
    try:
        if req.force_event_id:
            return eng.step_force_event(req.force_event_id)
        return eng.step(req.command)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc)) from exc


## 4. Verify input files

`plan.json` and `world.json` are independent outputs checked separately:

| Have | Step 7 behaviour |
|------|------------------|
| `plan.json` + `world.json` | loads both — no LLM needed |
| `plan.json` only | loads plan, runs `world_builder` (needs vLLM) |
| neither | needs `plot_points.json` + `case_file.json`; runs full pipeline |

Upload pre-generated files to `/content/data/` to skip the corresponding step.

In [ ]:
import pathlib, json, sys

have_world = pathlib.Path('data/world.json').exists()
have_plan  = pathlib.Path('data/plan.json').exists()

print(f'plan.json  : {"found" if have_plan else "will be generated from Phase I files"}')
print(f'world.json : {"found" if have_world else "will be generated (needs LLM)"}')

if not have_plan:
    required = ['data/plot_points.json', 'data/case_file.json']
    missing  = [f for f in required if not pathlib.Path(f).exists()]
    if missing:
        print('MISSING — needed to generate plan.json:')
        for f in missing: print(' •', f)
        sys.exit(1)
    plot_points = json.loads(pathlib.Path('data/plot_points.json').read_text())
    case_file   = json.loads(pathlib.Path('data/case_file.json').read_text())
    print(f'plot_points.json : {len(plot_points)} beats')
    print(f'case_file.json   : victim = {case_file.get("victim", {}).get("name", "?")}')

print('Ready.')


## 5. Launch vLLM server

First run downloads model weights (3–8 min). Subsequent runs load from Colab cache.

In [ ]:
import subprocess, pathlib, sys, os
log_path = pathlib.Path('logs/vllm_server.log')
log_path.write_text('')

cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', os.environ['LLM_MODEL'],
    '--host', '0.0.0.0', '--port', str(PORT),
    '--dtype', os.environ['LLM_DTYPE'],
    '--max-model-len', '4096',
    '--gpu-memory-utilization', '0.85',
    '--enforce-eager',
    '--api-key', 'EMPTY',
]
vllm_proc = subprocess.Popen(cmd, stdout=open(log_path, 'ab'), stderr=subprocess.STDOUT)
print('vLLM pid =', vllm_proc.pid)


## 6. Wait for vLLM to be ready

In [ ]:
import time, urllib.request, pathlib

def probe():
    try:
        req = urllib.request.Request(
            f"{os.environ['LLM_ENDPOINT']}/models",
            headers={'Authorization': 'Bearer EMPTY'})
        with urllib.request.urlopen(req, timeout=5) as r:
            return r.status == 200
    except Exception:
        return False

def _show_log(tail=60):
    log = pathlib.Path('logs/vllm_server.log')
    if log.exists():
        lines = log.read_text(errors='replace').splitlines()
        print('\n── vllm_server.log (last', min(tail, len(lines)), 'lines) ──')
        print('\n'.join(lines[-tail:]))
        print('── end of log ──')
    else:
        print('(log file not found)')

for i in range(240):
    if probe():
        print('vLLM READY after', i * 5, 's')
        break
    # If the process died, no point waiting further — show log immediately
    rc = vllm_proc.poll()
    if rc is not None:
        _show_log()
        raise RuntimeError(f'vLLM process exited early with code {rc}')
    if i % 12 == 0 and i > 0:
        print(f'... still waiting ({i * 5}s)', flush=True)
    time.sleep(5)
else:
    _show_log()
    raise RuntimeError('vLLM did not become ready after 20 min — see log above')


## 7. Phase II pipeline

Each step is skipped if its output file already exists in `/content/data/`.

In [ ]:
import sys, pathlib
sys.path.insert(0, '/content')

from world_builder import build_world, load_world, save_world

# ── Step 1: plan.json ────────────────────────────────────────────────────
if pathlib.Path('data/plan.json').exists():
    print('plan.json found — skipping story_to_plan.')
    from story_to_plan import load_plan
    plan = load_plan('data/plan.json')
    print(f'  {len(plan.events)} events, {len(plan.causal_links)} causal links')
else:
    print('--- story_to_plan ---')
    from phase1_story_generator import load_checkpoint
    from story_to_plan import build_plan
    case_file   = load_checkpoint('data/case_file.json')
    plot_points = load_checkpoint('data/plot_points.json')
    plan = build_plan(case_file, plot_points, out_path='data/plan.json')
    print(f'  plan.json written: {len(plan.events)} events, {len(plan.causal_links)} causal links')

# ── Step 2: world.json ───────────────────────────────────────────────────
if pathlib.Path('data/world.json').exists():
    print('world.json found — skipping world_builder (no LLM needed).')
    world = load_world('data/world.json')
else:
    print('--- world_builder (LLM calls) ---')
    world = build_world(plan)
    save_world(world, 'data/world.json')
    print('  world.json written')

print(f'Ready: {len(world.locations)} locations')
for lid, loc in world.locations.items():
    print(f'  {loc.name}: exits={sorted(loc.adjacent)}')


## 7b. Interactive game test (optional)

Run this cell to play a test session directly in the notebook before starting the web server.
Every command shows a concise drama-manager log, the narration, and a location snapshot.
Type `quit` to exit the loop and continue to step 8.

In [ ]:
import sys, pathlib
sys.path.insert(0, '/content')

from story_to_plan import load_plan
from world_builder import load_world
from game_engine import EngineConfig, GameEngine

_t_plan   = load_plan('data/plan.json')
_t_world  = load_world('data/world.json')
_t_engine = GameEngine(
    _t_plan, _t_world,
    EngineConfig(narrate_with_llm=True, log_dir=pathlib.Path('logs/test')),
)

def _path_summary(eng):
    if eng.drama.goal_satisfied(eng.state):
        return 'GOAL SATISFIED — case ready to close'
    rem = len(eng.drama.remaining)
    return f'{rem} plan event(s) remaining — investigation ongoing'

def _hint_commands(eng):
    loc_id = eng.state['detective']['location']
    hints = []
    for eid in eng.drama.remaining:
        ev = eng.drama.plan.events[eid]
        if ev.location != loc_id:
            continue
        verb = ev.verb
        if verb in ('examine', 'search', 'investigate', 'observe'):
            verb = 'examine'
        elif verb in ('interview', 'consult', 'confront', 'visit'):
            verb = 'question'
        arg_str = ''
        for a in ev.args:
            s = str(a)
            if s.startswith('character.'):
                name = eng.state.get(s, {}).get('name', s.split('.')[-1].replace('_', ' '))
                arg_str = name.split()[-1]
                break
            elif s.startswith('evidence.'):
                desc = eng.state.get(s, {}).get('description', '')
                arg_str = ' '.join(desc.split()[:4]) if desc else s
                break
            elif not s.startswith('location.'):
                arg_str = s[:35]
                break
        hint = f'{verb} {arg_str}'.strip() if arg_str else verb
        if hint not in hints:
            hints.append(hint)
        if len(hints) >= 4:
            break
    return hints

def _print_location(eng):
    print(eng.render_location())
    hints = _hint_commands(eng)
    if hints:
        print('  Try: ' + '  |  '.join(hints))

print('=' * 56)
print('  Interactive game test  (type quit to exit)')
print('=' * 56)
print()
print(_t_engine.render_map())
print()
_print_location(_t_engine)

while True:
    try:
        cmd = input('\n> ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n[session ended]')
        break
    if cmd.lower() in ('quit', 'exit', 'q'):
        print('[session ended]')
        break
    if not cmd:
        continue

    result = _t_engine.step(cmd)
    tag    = result['classification']

    # ── DM log ───────────────────────────────────────────────────
    dm_line = f'[DM] {tag}'
    if result.get('triggered_event_id'):
        dm_line += f"  event={result['triggered_event_id']}"
    if result.get('moved_to'):
        dm_line += f"  moved→{result['moved_to']}"
    print(dm_line)

    if tag == 'exceptional':
        for entry in reversed(_t_engine.drama.log):
            if entry.kind == 'accommodation':
                p = entry.payload
                print(f"     removed  : {p.get('removed_events', [])}")
                print(f"     added    : {p.get('replacement_event_ids', [])}")
                rat = (p.get('rationale') or '')[:200]
                if rat:
                    print(f'     rationale: {rat}')
                break
        for entry in reversed(_t_engine.drama.log):
            if entry.kind == 'classification':
                threats = [t.get('event_id') for t in entry.payload.get('soft_threats', [])]
                if threats:
                    print(f'     threats  : {threats}')
                break

    print(f'[DM] {_path_summary(_t_engine)}')

    # ── Narration ────────────────────────────────────────────────
    print()
    for entry in result.get('log_entries', []):
        if entry.get('title'):
            print(f"  {entry['title']}")
        print(f"  {entry['text']}")

    # ── Location snapshot + hints ────────────────────────────────
    print()
    _print_location(_t_engine)

    if result.get('game_over'):
        print('\n=== CASE SOLVED ===')
        break


## 8. Start the FastAPI web server

In [ ]:
import subprocess, pathlib, socket
sys.path.insert(0, '/content/web')

def _wait_port_free(port, timeout=10):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with socket.socket() as s:
                s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
                s.bind(('0.0.0.0', port))
            return True
        except OSError:
            time.sleep(0.3)
    return False

# Kill previous process (SIGKILL — cannot be ignored) then wait for port.
if '_web_proc' in globals() and _web_proc is not None:
    _web_proc.kill()
    _web_proc.wait()
    _web_proc = None
_wait_port_free(WEB_PORT)

log_path = pathlib.Path('logs/web/uvicorn.log')
log_path.parent.mkdir(parents=True, exist_ok=True)

_web_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'api_server:app',
     '--host', '0.0.0.0', '--port', str(WEB_PORT), '--log-level', 'warning'],
    cwd='/content/web',
    stdout=open(log_path, 'ab'), stderr=subprocess.STDOUT,
)
time.sleep(2)
if _web_proc.poll() is None:
    print(f'Web server running on port {WEB_PORT}')
else:
    print(f'ERROR: web server exited immediately — check {log_path}')


## 9. Open the game

Click the URL below. It stays live as long as this Colab session is running.

In [ ]:
from google.colab.output import eval_js
url = eval_js(f'google.colab.kernel.proxyPort({WEB_PORT})')
print('='*60)
print(url)
print('='*60)


## 10. Cleanup

Stops vLLM and frees the GPU.

In [ ]:
try:
    vllm_proc.terminate()
    vllm_proc.wait(timeout=10)
    print('vLLM stopped')
except Exception as e:
    print('cleanup:', e)
